<a href="https://colab.research.google.com/github/emisoft-designs/emeraldeo/blob/knowledgebase/Knowledge_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings

# Suppress DeprecationWarning from jupyter_client related to utcnow()
warnings.filterwarnings( "ignore")

print("Deprecation warning from jupyter_client suppressed.")

Deprecation warning from jupyter_client suppressed.


In [2]:
import getpass
import os


def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Please provide your {var}")

_set_if_undefined("GOOGLE_API_KEY")
_set_if_undefined("TAVILY_API_KEY")
_set_if_undefined("SERPER_API_KEY")

Please provide your GOOGLE_API_KEY··········
Please provide your TAVILY_API_KEY··········
Please provide your SERPER_API_KEY··········


In [3]:
from pydantic import BaseModel, Field, model_validator
from typing import ClassVar, Dict, Any, List, Optional, Annotated, Sequence, Literal
from langchain_core.documents import Document
from typing_extensions import TypedDict
from datetime import datetime, timezone
from enum import Enum
import uuid
import math
import logging
import operator

# -------------------- Setup Logging --------------------
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)  # Use INFO in production

# -------------------- Enums and Constants --------------------
class CrawlStatus(str, Enum):
    """Crawling status enumeration."""
    NEUTRAL = "neutral"
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    SUCCESS = "success"
    FAILED = "failed"
    PARTIAL = "partial"
    LAST_ERROR = "last_error"

class DocumentSource(str, Enum):
    """Document source types."""
    KNOWLEDGE_BASE = "knowledge_base"
    EXTERNAL_API = "external_api"
    WEB_CRAWL = "web_crawl"
    SEARCH_SNIPPET = "search_snippet"

class ContentSource(str, Enum):
    """Semantic categories assigned by LLM (e.g., academic, blog, forum)."""
    ACADEMIC = "academic"
    GOVERNMENT = "government"
    WIKI = "wikipedia"
    GIT_REPO = "git_repo"
    NEWS = "news"
    BLOG = "blog"
    REPORT = "report"
    FORUM = "forum"
    OTHER = "other"

class QueryComplexity(str, Enum):
    """Query complexity levels."""
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class QualityScoreConfig(float, Enum):
    RELEVANCE_BIAS = 0.1
    SOURCE_SCORE_FLOOR = 0.3
    RICHNESS_LOG_DIVISOR = 20
    GAP_PENALTY_FACTOR = 0.05
    COMPLEXITY_WEIGHT = 0.2
    DIVERSITY_WEIGHT = 0.1

# -------------------- Sub-Models --------------------
class SubQuery(BaseModel):
    """Represents a generated sub-query for targeted search."""
    query: str = Field(..., description="The sub-query text.")
    priority: float = Field(0.5, ge=0.0, le=1.0, description="Priority of this sub-query.")
    deep: bool = Field(False, description="Whether to perform a deep search for this query.")
    query_type: Literal["general", "technical", "trends"] = Field("general", description="Type of the sub-query.")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class DocumentResult(BaseModel):
    """Document result with standardized metadata."""
    page_content: str = Field(..., description="The document content")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Document metadata")
    content_length: Optional[int] = Field(default=0, ge=0, description="Content length in characters")
    source_type: DocumentSource = Field(default=DocumentSource.SEARCH_SNIPPET, description="Source type")
    content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
    relevance_score: Optional[float] = Field(default=None, ge=0.0, le=1.0, description="Relevance score")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

    @model_validator(mode="after")
    def compute_content_length(self) -> "DocumentResult":
        """Ensure content_length is always computed from page_content if missing/zero."""
        if self.page_content and (not self.content_length or self.content_length == 0):
            object.__setattr__(self, "content_length", len(self.page_content))
        return self

class ExtractedURL(BaseModel):
    """URL with extraction metadata."""
    query: str
    url: str = Field(..., description="The URL")
    domain: Optional[str] = Field(default=None, description="Domain extracted from URL")
    priority: float = Field(0.5, ge=0.0, le=1.0, description="Crawling priority")
    should_crawl: bool = Field(True, description="Whether this URL should be crawled")
    content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
    estimated_crawl_time: Optional[float] = Field(default=None, ge=1, description="Estimated crawl time in seconds")
    title: Optional[str] = Field(default=None, description="Page title if available")
    snippet: Optional[str] = Field(default=None, description="Content snippet")
    source_engine: Optional[str] = Field(default=None, description="Search engine that found this URL")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    retry_count: int = Field(default=0, description="Number of times crawling this URL has been retried.")

    @model_validator(mode="after")
    def compute_domain(self) -> "ExtractedURL":
        """Ensure domain is auto-computed from URL if missing."""
        if not self.domain and self.url:
            try:
                from urllib.parse import urlparse
                parsed_url = urlparse(self.url)
                if parsed_url.netloc:
                    object.__setattr__(self, "domain", parsed_url.netloc)
            except Exception:
                pass
        return self

class CrawlStats(BaseModel):
    """Crawling statistics."""
    attempted: int = Field(default=0, ge=0, description="Number of URLs attempted")
    successful: int = Field(default=0, ge=0, description="Number of successful crawls")
    failed: int = Field(default=0, ge=0, description="Number of failed crawls")
    total_response_time: float = Field(default=0.0, ge=0.0, description="Total response time in seconds")

class KnowledgeMetadata(BaseModel):
    """Session-level metadata."""
    session_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    last_updated: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    total_iterations: int = Field(default=0, ge=0)
    final: bool = Field(False)
    crawl_stats: CrawlStats = Field(default_factory=CrawlStats)

# -------------------- LangGraph TypedDict --------------------
class LangGraphKnowledgeState(TypedDict):
    """LangGraph state schema, using reducers for list updates."""
    main_query: str
    messages: Annotated[List[str], operator.add]
    sub_queries_to_generate: Annotated[List[SubQuery], operator.add]
    sub_queries_to_process: Annotated[List[SubQuery], operator.add]
    processed_queries: Annotated[List[str], operator.add]
    knowledge_base_results: Annotated[List[DocumentResult], operator.add]
    external_api_results: Annotated[List[DocumentResult], operator.add]
    crawler_results: Annotated[List[DocumentResult], operator.add]
    extracted_urls: Annotated[List[ExtractedURL], operator.add]
    crawl_queue: Annotated[List[ExtractedURL], operator.add]
    cleaned_chunks: Annotated[List[DocumentResult], operator.add]
    gaps: Annotated[List[str], operator.add]
    state_metrics: Dict[str, Any]
    metadata: KnowledgeMetadata
    errors: Annotated[List[str], operator.add]

# -------------------- Main KnowledgeState --------------------
class KnowledgeState(BaseModel):
    """Represents the state of the knowledge acquisition process."""
    main_query: str = Field(..., description="The main query driving the knowledge acquisition.")
    messages: List[Any] = Field(default_factory=list)
    sub_queries_to_generate: List[SubQuery] = Field(default_factory=list)
    sub_queries_to_process: List[SubQuery] = Field(default_factory=list)
    processed_queries: List[str] = Field(default_factory=list)

    knowledge_base_results: List[DocumentResult] = Field(default_factory=list)
    external_api_results: List[DocumentResult] = Field(default_factory=list)
    crawler_results: List[DocumentResult] = Field(default_factory=list)

    extracted_urls: List[ExtractedURL] = Field(default_factory=list)
    crawl_queue: List[ExtractedURL] = Field(default_factory=list)
    crawl_status: CrawlStatus = Field(default=CrawlStatus.NEUTRAL)

    cleaned_chunks: List[DocumentResult] = Field(default_factory=list)
    gaps: List[str] = Field(default_factory=list)
    state_metrics: Dict[str, Any] = Field(default_factory=dict)
    metadata: KnowledgeMetadata = Field(default_factory=KnowledgeMetadata)
    errors: List[str] = Field(default_factory=list)

    _LANGGRAPH_STATE_SCHEMA: ClassVar = LangGraphKnowledgeState

    # -------------------- Validators --------------------
    @model_validator(mode="after")
    def validate_counts(self) -> "KnowledgeState":
        """Ensure crawl stats are consistent."""
        # Ensure attempted count is at least the number of extracted URLs marked for crawl
        attempted_crawl_urls = [url for url in self.extracted_urls if url.should_crawl]
        if self.metadata.crawl_stats.attempted < len(attempted_crawl_urls):
             logger.warning(f"Crawl stats 'attempted' count ({self.metadata.crawl_stats.attempted}) is less than extracted URLs marked for crawl ({len(attempted_crawl_urls)}).")
        # Ensure successful + failed <= attempted
        if self.metadata.crawl_stats.successful + self.metadata.crawl_stats.failed > self.metadata.crawl_stats.attempted:
             logger.warning("Crawl stats successful + failed exceeds attempted.")
        return self


    # -------------------- Properties --------------------
    @property
    def total_documents(self) -> int:
        return len(self.knowledge_base_results) + len(self.external_api_results) + len(self.crawler_results)

    @property
    def success_rate(self) -> float:
        stats = self.metadata.crawl_stats
        return stats.successful / stats.attempted if stats.attempted else 0.0

    @property
    def has_sufficient_content(self) -> bool:
        # Consider cleaned chunks and KB results for sufficiency
        total_chunks = len(self.knowledge_base_results) + len(self.cleaned_chunks)
        min_required_chunks = self.state_metrics.get("min_required_chunks", 5)
        return total_chunks >= min_required_chunks

    @property
    def should_continue_crawling(self) -> bool:
        # Continue crawling if within iteration limit, not sufficient content, and crawl queue is not empty
        max_iterations = self.state_metrics.get("max_iterations", 5)
        return self.metadata.total_iterations < max_iterations and not self.has_sufficient_content and bool(self.crawl_queue)

    # -------------------- Methods to Update State --------------------
    def add_queries(self, new_sub_queries: List[SubQuery]) -> None:
        self.sub_queries_to_generate.extend(new_sub_queries)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_results(self, new_results: List[DocumentResult], source: DocumentSource) -> None:
        if source == DocumentSource.KNOWLEDGE_BASE:
            self.knowledge_base_results.extend(new_results)
            print(f"Added {len(new_results)} results from {source}")
        elif source in (DocumentSource.EXTERNAL_API, DocumentSource.SEARCH_SNIPPET):
            self.external_api_results.extend(new_results)
            print(f"Added {len(new_results)} results from {source}")
        elif source == DocumentSource.WEB_CRAWL:
            self.crawler_results.extend(new_results)
            print(f"Added {len(new_results)} results from {source}")
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_urls_to_crawl(self, urls: List[ExtractedURL]) -> None:
        self.crawl_queue.extend(urls)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_urls(self, new_urls: List[ExtractedURL], add_to_queue: bool = True) -> None:
        self.extracted_urls.extend(new_urls)
        if add_to_queue:
            self.crawl_queue.extend(new_urls)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def mark_sub_queries_processed(self, query_texts: List[str]) -> None:
        self.processed_queries.extend(query_texts)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_error(self, error_msg: str) -> None:
        self.errors.append(f"[{datetime.now(timezone.utc).isoformat()}] {error_msg}")
        self.metadata.last_updated = datetime.now(timezone.utc)
        logger.error(f"Error occurred: {error_msg}")

    def update_crawl_stats(self, attempted: int = 0, successful: int = 0, failed: int = 0, time: float = 0.0) -> None:
        self.metadata.crawl_stats.attempted += attempted
        self.metadata.crawl_stats.successful += successful
        self.metadata.crawl_stats.failed += failed
        self.metadata.crawl_stats.total_response_time += time
        self.metadata.last_updated = datetime.now(timezone.utc)

    # -------------------- Quality Score --------------------
    def calculate_weighted_quality_score(self) -> float:
        # Combine relevant document lists for scoring
        docs_for_scoring = self.knowledge_base_results + self.cleaned_chunks

        if not docs_for_scoring:
            return 0.0

        total_content_length = sum(res.content_length or 0 for res in docs_for_scoring)
        # Avoid division by zero if RICHNESS_LOG_DIVISOR is used in a way that can be zero
        richness_score = math.log1p(total_content_length / QualityScoreConfig.RICHNESS_LOG_DIVISOR.value)

        total_relevance = sum(
            res.relevance_score for res in docs_for_scoring if res.relevance_score is not None
        )
        relevance_score = (total_relevance / len(docs_for_scoring)) * QualityScoreConfig.RELEVANCE_BIAS.value

        # Consider unique content sources from cleaned chunks and KB
        unique_sources = set()
        for res in docs_for_scoring:
             if res.content_source:
                  unique_sources.add(res.content_source)

        diversity_score = (len(unique_sources) / len(ContentSource)) * QualityScoreConfig.DIVERSITY_WEIGHT.value

        complexity_level = self.state_metrics.get("query_complexity", QueryComplexity.MEDIUM)
        complexity_weight_map = {
            QueryComplexity.HIGH: 1.0,
            QueryComplexity.MEDIUM: 0.5,
            QueryComplexity.LOW: 0.1
        }
        complexity_score = complexity_weight_map.get(complexity_level, 0.5) * QualityScoreConfig.COMPLEXITY_WEIGHT.value

        gap_penalty = len(self.gaps) * QualityScoreConfig.GAP_PENALTY_FACTOR.value

        score = (richness_score + relevance_score + diversity_score + complexity_score) - gap_penalty
        return max(0.0, score)

    # -------------------- Export Methods --------------------
    def to_dict(self) -> Dict[str, Any]:
        return self.model_dump()

    def to_summary(self) -> Dict[str, Any]:
        # Ensure quality score is calculated before summarizing
        quality_score = self.calculate_weighted_quality_score()
        return {
            "main_query": self.main_query,
            "total_documents": self.total_documents,
            "indexed_chunks": len(self.knowledge_base_results) + len(self.cleaned_chunks), # Count documents ready for RAG
            "crawl_queue_size": len(self.crawl_queue),
            "attempted_crawls": self.metadata.crawl_stats.attempted,
            "successful_crawls": self.metadata.crawl_stats.successful,
            "failed_crawls": self.metadata.crawl_stats.failed,
            "success_rate": self.success_rate,
            "quality_score": quality_score,
            "final": self.metadata.final,
            "errors": self.errors
        }

    @classmethod
    def create_knowledge_state(cls, main_query: str) -> "KnowledgeState":
      return cls(main_query=main_query, messages=[]) # Added messages field

    # -------------------- Helper Conversion Functions --------------------
    @staticmethod
    def knowledge_state_to_typed_dict(state: "KnowledgeState") -> Dict[str, Any]:
        return state.model_dump()

    @staticmethod
    def typed_dict_to_knowledge_state(data: Dict[str, Any]) -> "KnowledgeState":
        return KnowledgeState(**data)

    @staticmethod
    def convert_langchain_doc_to_document_result(
        doc: Document,
        source_type: DocumentSource = DocumentSource.SEARCH_SNIPPET,
        relevance_score: Optional[float] = None
    ) -> DocumentResult:
        """Converts a LangChain Document to a Pydantic DocumentResult."""
        content_source = doc.metadata.get("content_source")
        if isinstance(content_source, str):
            try:
                content_source_enum = ContentSource(content_source)
            except ValueError:
                content_source_enum = ContentSource.OTHER # Default if conversion fails
        elif isinstance(content_source, ContentSource): # Handle if it's already an enum
             content_source_enum = content_source
        else:
            content_source_enum = ContentSource.OTHER # Default if not a string or enum

        # Ensure relevance score is clamped to [0, 1] if provided
        clamped_relevance_score = max(0.0, min(1.0, relevance_score)) if relevance_score is not None else None


        return DocumentResult(
            page_content=doc.page_content,
            metadata=doc.metadata,
            source_type=source_type,
            content_source=content_source_enum,
            relevance_score=clamped_relevance_score, # Use the clamped score
            content_length=len(doc.page_content) if doc.page_content else 0
        )

    @staticmethod
    def convert_search_result_to_extracted_url(result: Dict[str, Any], query: str, priority: float = 0.5) -> ExtractedURL:
        return ExtractedURL(
            query=query,
            url=result.get('url', ''),
            title=result.get('title'),
            snippet=result.get('snippet'),
            source_engine=result.get('engine'),
            priority=priority,
            should_crawl=True
        )

In [4]:
!pip install langchain-tavily langchain-community duckduckgo-search --quiet
!pip install -U langchain-google-genai --quiet # Ensure Google GenAI is installed

from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, Dict, Any, List
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from duckduckgo_search import DDGS
from langchain_community.embeddings import SentenceTransformerEmbeddings # Import SentenceTransformerEmbeddings
import os # Import os to access environment variables
from google.colab import userdata # Import userdata for Colab secrets


load_dotenv()

# Assuming these config classes are defined elsewhere or will be defined:
# from tools import URLAnalysisConfig, CrawlerConfig, VectorStoreConfig
# from KnowledgeState import LLMConfig # LLMConfig is often separate or simple

# --- Define Sub-Config Models used within GlobalConfig ---
# If these are already defined in your tools/other files, you can remove these definitions
# and ensure they are imported correctly. Assuming they are not fully defined yet for demo.

class SearchConfig(BaseModel):
    """Configuration for the SearchEngine."""
    external_max_results: int = Field(default=8, ge=1, description="Maximum search results per query for external searches.")
    subquery_max_search: int = Field(default=3, ge=1, description="Maximum number of subqueries to run in a batch.")
    subquery_results_per_query: int = Field(default=5, ge=1, description="Maximum search results per subquery.")
    # Add other search engine specific configs here if needed (e.g., API keys)
    # Tavily and DDGS clients are complex types, use Any or specific protocol if defined elsewhere
    tavily_client: Optional[Any] = Field(default=None, description="Initialized TavilySearch client.")
    ddgs_client: Optional[Any] = Field(default=None, description="Initialized DuckDuckGoSearch client.") # Use Any for now

    model_config = ConfigDict(extra='forbid', arbitrary_types_allowed=True)

class URLAnalysisConfig(BaseModel):
    """Configuration for the URLAnalyzer."""
    max_urls: int = Field(default=10, ge=1, le=50, description="Maximum number of URLs to return.")
    priority_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Minimum priority for a URL to be considered crawlable.")
    enable_ml_scoring: bool = Field(default=True, description="Enable ML-based relevance scoring via LLM.")
    high_value_domains: Dict[str, float] = Field(
        default={
            'wikipedia.org': 0.8, 'arxiv.org': 0.7, 'github.com': 0.6,
            'medium.com': 0.5, 'towardsdatascience.com': 0.5,
            'ieee.org': 0.8, 'acm.org': 0.8, '.edu': 0.8, '.gov': 0.8
        },
        description="Domains with assigned priority scores."
    )
    low_value_patterns: List[str] = Field(
        default=[
            r'facebook\.com', r'twitter\.com', r'linkedin\.com',
            r'instagram\.com', r'pinterest\.com', r'reddit\.com/r/\w+/comments',
            r'youtube\.com/watch', r'tiktok\.com', r'^\s*javascript:'
        ],
        description="Regex patterns for low-value URLs."
    )
    domain_to_content_source: Dict[str, str] = Field( # Using str here assuming ContentSource Enum will be handled
         default={
            'wikipedia.org': "wiki",
            'arxiv.org': "academic",
            'github.com': "git_repo",
            'nature.com': "academic",
            'ieee.org': "academic",
            'acm.org': "academic",
            '.gov': "government",
            '.edu': "academic",
        },
        description="Mapping of domains to ContentSource strings."
    )
    model_config = ConfigDict(extra='forbid') # Ensure no extra fields are allowed

class CrawlerConfig(BaseModel):
    """Configuration for the CrawlerEngine."""
    chunk_size: int = Field(default=1000, ge=100, description="Size of text chunks.")
    chunk_overlap: int = Field(default=200, ge=0, description="Overlap between text chunks.")
    max_concurrent: int = Field(default=5, ge=1, description="Maximum concurrent crawl tasks.")
    timeout: int = Field(default=15, ge=5, description="Timeout for each crawl request in seconds.")
    max_content_length: int = Field(default=150000, ge=10000, description="Max length of crawled content to prevent memory issues.")
    enable_embedding_content_scoring: bool = Field(default=True, description="Enable embedding-based scoring of crawled content.") # Changed to embedding scoring
    embedding_model_name: str = Field("sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use for content scoring.") # Added embedding model config
    embedding_relevance_threshold: float = Field(default=0.4, ge=0.0, le=1.0, description="Relevance score threshold for embedding-scored chunks.") # Changed threshold name
    max_retries: int = Field(default=2, ge=0, description="Maximum number of times to retry a failed URL crawl.") # Added max_retries

class CleanerConfig(BaseModel):
    """Configuration for the CleanerEngine."""
    chunk_size: int = Field(default=1200, ge=200, description="Size of text chunks.")
    chunk_overlap: int = Field(default=160, ge=0, description="Overlap between text chunks.")
    use_semantic_chunking: bool = Field(default=False, description="Use embedding-based semantic chunking.")
    embedding_model_name: str = Field("sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use.")
    model_config = ConfigDict(extra='forbid') # Ensure no extra fields are allowed


class VectorStoreConfig(BaseModel):
    """Configuration for the VectorStoreNode."""
    vector_store_type: str = Field(
        default="chroma",
        description="Type of vector store to use: 'chroma', 'postgres', or 'sqlite'."
    )
    collection_name: str = Field(default="default-collection", description="The name of the vector collection/table.")
    chroma_persist_directory: str = Field(default="./chroma_db", description="Directory for Chroma persistence.")
    # Add embedding model config here, or keep it separate if preferred
    embedding_model_name: str = Field(default="sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use.")
    embedding_model_provider: str = Field(default="huggingface", description="Provider for the embedding model: 'huggingface' or 'sentence-transformer'.")
    # Add embedding model itself as a field, allowing Any or specific protocol
    embedding_model: Optional[Any] = Field(default=None, description="Initialized Embedding Model instance.") # Added field for embedding model

    model_config = ConfigDict(extra='forbid', arbitrary_types_allowed=True)

class LLMConfig(BaseModel):
    """Configuration for Language Models."""
    # Updated default models to Gemini
    chat_model_name: str = Field(default="gemini-2.5-flash", description="Name of the main chat model.")
    tool_model_name: str = Field(default="gemini-2.5-flash", description="Name of the tool-calling model.")
    chat_model_provider: str = Field(default="google_genai", description="Provider for the main chat model: 'google' or 'openai'.")
    tool_model_provider: str = Field(default="google_genai", description="Provider for the tool-calling model: 'google' or 'openai'.")
    # Keep embedding model separate or align with SentenceTransformer
    embedding_model_name: str = Field(default="text-embedding-ada-002", description="Name of the embedding model.") # This one might be redundant if SentenceTransformer is used directly
    temperature: float = Field(default=0.0, ge=0.0, le=2.0, description="Temperature for creative tasks.")
    # Add other LLM-specific configs (e.g., API keys, max tokens)

class DecisionsConfig(BaseModel):
    """Configuration for Graph Decision Logic."""
    confidence_threshold: float = Field(default=0.7, ge=0.0, le=1.0, description="KB confidence threshold to skip external search.")
    low_success_rate_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Crawl success rate below which crawling is stopped.")
    quality_threshold: float = Field(default=0.6, ge=0.0, le=1.0, description="Content quality score threshold to stop searching.")
    max_iterations: int = Field(default=5, ge=1, description="Maximum number of graph iterations.")
    min_results: int = Field(default=2, ge=0, description="Minimum number of results needed for quality assessment.")
    max_final_documents: int = Field(default=20, ge=1, description="Maximum number of documents in the final result set.")


class GlobalConfig(BaseModel):
    """
    Comprehensive configuration for the Knowledge Acquisition workflow.
    """
    search: "SearchConfig" = Field(default_factory=lambda: SearchConfig())
    url_analyzer: "URLAnalysisConfig" = Field(default_factory=lambda: URLAnalysisConfig())
    crawler: "CrawlerConfig" = Field(default_factory=lambda: CrawlerConfig())
    cleaner: "CleanerConfig" = Field(default_factory=lambda: CleanerConfig())
    vector_store: "VectorStoreConfig" = Field(default_factory=lambda: VectorStoreConfig())
    llm: LLMConfig = Field(default_factory=LLMConfig)
    decisions: "DecisionsConfig" = Field(default_factory=lambda: DecisionsConfig())

    model_config = ConfigDict(extra="forbid")


# --- Global LLM Registry ---
class LLMRegistry:
    _models: Dict[str, Any] = {}

    @classmethod
    def init_from_config(cls, config: LLMConfig):
        # Ensure GOOGLE_API_KEY is set for Google GenAI models
        google_api_key = userdata.get('GOOGLE_API_KEY') or os.environ.get("GOOGLE_API_KEY")
        if not google_api_key:
             raise ValueError("GOOGLE_API_KEY not found. Please set it in Colab secrets or environment variables.")

        # Initialize chat model (used for main agent/decisions)
        # Use init_chat_model which handles different providers
        try:
            cls._models["chat"] = init_chat_model(
                config.chat_model_name,
                model_provider=config.chat_model_provider,
                temperature=config.temperature,
                google_api_key=google_api_key # Pass API key for Google models
            )
            logger.info(f"Initialized chat model: {config.chat_model_name}")
        except Exception as e:
             logger.error(f"Failed to initialize chat model {config.chat_model_name}: {e}")
             raise e # Re-raise to indicate failure

        # Initialize tool model (used for tool calling/structured output)
        try:
            cls._models["tool"] = init_chat_model(
                config.tool_model_name,
                temperature=config.temperature, # Tools often prefer lower temperature
                model_provider=config.tool_model_provider,
                google_api_key=google_api_key # Pass API key for Google models
            )
            logger.info(f"Initialized tool model: {config.tool_model_name}")
        except Exception as e:
             logger.error(f"Failed to initialize tool model {config.tool_model_name}: {e}")
             # Decide if tool model initialization failure is critical or can be warning
             # For this workflow, a tool model is likely essential, so re-raise
             raise e


        # Embedding model from LLMConfig might be different from the one in Crawler/VectorStoreConfig
        # Depending on your architecture, you might initialize it here or keep it separate
        # For now, we'll rely on the embedding model passed directly to the tools
        # via the main process_query function.

        return cls._models

    @classmethod
    def get(cls, role: str = "chat"):
        if role not in cls._models:
            raise ValueError(f"LLM '{role}' not initialized. Call init_from_config first.")
        return cls._models[role]
# Example usage:
# global_config = GlobalConfig()
# print(global_config.model_dump_json(indent=2))

Here's a test script to demonstrate the functionality of the `KnowledgeState` model.

In [5]:
# Create an initial KnowledgeState instance
initial_state = KnowledgeState.create_knowledge_state(main_query="What are the latest advancements in AI?")
print("Initial State:")
print(initial_state.to_summary())

# Add some sub-queries
sub_queries = [
    SubQuery(query="Recent breakthroughs in natural language processing"),
    SubQuery(query="New developments in computer vision", priority=0.8),
    SubQuery(query="Ethical considerations in AI", priority=0.6), # Add another query
]
initial_state.add_queries(sub_queries)
print("\nState after adding sub-queries:")
print(initial_state.sub_queries_to_generate)

# Add some document results including edge cases for relevance and length
kb_results = [
    DocumentResult(page_content="Content about NLP breakthrough 1", source_type=DocumentSource.KNOWLEDGE_BASE, relevance_score=0.9),
    DocumentResult(page_content="Content about CV development 1", source_type=DocumentSource.KNOWLEDGE_BASE, relevance_score=0.7),
    DocumentResult(page_content="Short but highly relevant fact.", source_type=DocumentSource.KNOWLEDGE_BASE, relevance_score=1.0), # Short, high relevance
]
web_results = [
    DocumentResult(page_content="Blog post on AI trends" * 50, source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.BLOG, relevance_score=0.1), # Long, low relevance
    DocumentResult(page_content="Academic paper on new algorithm" * 10, source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.ACADEMIC, relevance_score=0.95),
    DocumentResult(page_content="Irrelevant content.", source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.FORUM, relevance_score=0.0), # Short, zero relevance
    DocumentResult(page_content="Very long and irrelevant document content. " * 200, source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.OTHER, relevance_score=0.05), # Very long, very low relevance
]
initial_state.add_results(kb_results, source=DocumentSource.KNOWLEDGE_BASE)
initial_state.add_results(web_results, source=DocumentSource.WEB_CRAWL)
print("\nState after adding results:")
print(f"Total documents: {initial_state.total_documents}")
print(f"Knowledge base results: {len(initial_state.knowledge_base_results)}")
print(f"Crawler results: {len(initial_state.crawler_results)}")


# Add some extracted URLs
extracted_urls = [
    ExtractedURL(query=initial_state.main_query, url="https://example.com/blog/ai-trends", content_source=ContentSource.BLOG),
    ExtractedURL(query=initial_state.main_query, url="https://anothersite.org/paper.pdf", content_source=ContentSource.ACADEMIC, should_crawl=False),
    ExtractedURL(query=initial_state.main_query, url="https://yetanothersite.com/forum/irrelevant", content_source=ContentSource.FORUM), # Add another URL
]
initial_state.add_urls(extracted_urls)
print("\nState after adding URLs:")
print(f"Extracted URLs: {len(initial_state.extracted_urls)}")
print(f"Crawl Queue: {len(initial_state.crawl_queue)}")

# Update crawl stats - reflecting more attempts
initial_state.update_crawl_stats(attempted=3, successful=2, failed=1, time=7.8)
print("\nState after updating crawl stats:")
print(f"Crawl Stats: {initial_state.metadata.crawl_stats}")
print(f"Success Rate: {initial_state.success_rate:.2f}")

# Mark sub-queries as processed
initial_state.mark_sub_queries_processed([sub_queries[0].query, sub_queries[1].query]) # Mark more queries processed
print("\nState after marking queries processed:")
print(f"Processed Queries: {initial_state.processed_queries}")
print(f"Sub-queries to generate: {len(initial_state.sub_queries_to_generate)}")

# Add a gap and an error
initial_state.gaps.append("Missing information on ethical considerations")
initial_state.add_error("Failed to parse a document from irrelevant source")
initial_state.add_error("Timeout during crawl attempt") # Add another error
print("\nState after adding gap and error:")
print(f"Gaps: {initial_state.gaps}")
print(f"Errors: {initial_state.errors}")

# Check computed properties
print("\nComputed Properties:")
# To better test has_sufficient_content, let's add some 'cleaned_chunks'
# In a real scenario, these would come from processing DocumentResults
# Modify to add DocumentResult objects instead of Document objects
initial_state.cleaned_chunks.append(DocumentResult(page_content="Cleaned chunk 1 " * 50, source_type=DocumentSource.WEB_CRAWL, relevance_score=0.7))
initial_state.cleaned_chunks.append(DocumentResult(page_content="Cleaned chunk 2 " * 60, source_type=DocumentSource.WEB_CRAWL, relevance_score=0.8))

print(f"Has sufficient content (placeholder): {initial_state.has_sufficient_content}")
initial_state.metadata.total_iterations = 3 # Simulate more iterations
print(f"Should continue crawling (placeholder): {initial_state.should_continue_crawling}")

# Calculate quality score (demonstration)
# Manually set complexity for demonstration
initial_state.state_metrics["query_complexity"] = QueryComplexity.HIGH
quality_score = initial_state.calculate_weighted_quality_score()
print(f"\nCalculated Quality Score: {quality_score:.2f}")

# Export to dict
state_dict = initial_state.to_dict()
# print("\nState exported to dictionary:")
# print(state_dict)

# Create a new state from the dictionary (demonstration of loading)
loaded_state = KnowledgeState.typed_dict_to_knowledge_state(state_dict)
print("\nState loaded from dictionary:")
print(loaded_state.to_summary())

ERROR:__main__:Error occurred: Failed to parse a document from irrelevant source
ERROR:__main__:Error occurred: Timeout during crawl attempt


Initial State:
{'main_query': 'What are the latest advancements in AI?', 'total_documents': 0, 'indexed_chunks': 0, 'crawl_queue_size': 0, 'attempted_crawls': 0, 'successful_crawls': 0, 'failed_crawls': 0, 'success_rate': 0.0, 'quality_score': 0.0, 'final': False, 'errors': []}

State after adding sub-queries:
[SubQuery(query='Recent breakthroughs in natural language processing', priority=0.5, deep=False, query_type='general', created_at=datetime.datetime(2025, 9, 8, 19, 54, 9, 858213, tzinfo=datetime.timezone.utc)), SubQuery(query='New developments in computer vision', priority=0.8, deep=False, query_type='general', created_at=datetime.datetime(2025, 9, 8, 19, 54, 9, 858223, tzinfo=datetime.timezone.utc)), SubQuery(query='Ethical considerations in AI', priority=0.6, deep=False, query_type='general', created_at=datetime.datetime(2025, 9, 8, 19, 54, 9, 858227, tzinfo=datetime.timezone.utc))]
Added 3 results from DocumentSource.KNOWLEDGE_BASE
Added 4 results from DocumentSource.WEB_CRAWL

In [6]:
from pydantic import BaseModel, Field, field_validator, ConfigDict
from typing import ClassVar, Dict, Any, List, Optional, Annotated, Sequence, Protocol
from typing_extensions import TypedDict
from datetime import datetime, timezone
from langchain_core.documents import Document
from enum import Enum
import uuid
import math
import logging
import operator
import asyncio
import re
from urllib.parse import urlparse
from abc import ABC, abstractmethod

# -------------------- Pydantic models for configuration --------------------

class URLAnalysisConfig(BaseModel):
    """Configuration for the URLAnalyzer."""
    max_urls: int = Field(default=10, ge=1, le=50, description="Maximum number of URLs to return.")
    priority_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Minimum priority for a URL to be considered crawlable.")
    enable_ml_scoring: bool = Field(default=True, description="Enable ML-based relevance scoring via LLM.")
    high_value_domains: Dict[str, float] = Field(
        default={
            'wikipedia.org': 0.8, 'arxiv.org': 0.7, 'github.com': 0.6,
            'medium.com': 0.5, 'towardsdatascience.com': 0.5,
            'ieee.org': 0.8, 'acm.org': 0.8, '.edu': 0.8, '.gov': 0.8
        },
        description="Domains with assigned priority scores."
    )
    low_value_patterns: List[str] = Field(
        default=[
            r'facebook\.com', r'twitter\.com', r'linkedin\.com',
            r'instagram\.com', r'pinterest\.com', r'reddit\.com/r/\w+/comments',
            r'youtube\.com/watch', r'tiktok\.com', r'^\s*javascript:'
        ],
        description="Regex patterns for low-value URLs."
    )
    domain_to_content_source: Dict[str, ContentSource] = Field(
        default={
            'wikipedia.org': ContentSource.WIKI,
            'arxiv.org': ContentSource.ACADEMIC,
            'github.com': ContentSource.GIT_REPO,
            'nature.com': ContentSource.ACADEMIC,
            'ieee.org': ContentSource.ACADEMIC,
            'acm.org': ContentSource.ACADEMIC,
            '.gov': ContentSource.GOVERNMENT,
            '.edu': ContentSource.ACADEMIC,
        },
        description="Mapping of domains to ContentSource enums."
    )
    model_config = ConfigDict(extra='forbid')


# -------------------- Async URL Analyzer Protocol --------------------

class AsyncURLAnalyzerProtocol(Protocol):
    """Protocol for an asynchronous URL analyzer."""
    async def analyze_batch(self, search_results: List[Dict[str, Any]], query: str) -> None:
        """Analyze a batch of search results asynchronously and update state."""
        ...


# -------------------- Enhanced URL Analysis with Pydantic Integration --------------------

class URLAnalyzer(AsyncURLAnalyzerProtocol):
    """Enhanced, state-aware, async URL analyzer that works with Pydantic models."""

    def __init__(self, config: URLAnalysisConfig, llm: Optional[Any] = None):
        self.state = None
        self.config = config
        self.llm = llm

    async def analyze_batch(self, state: KnowledgeState, search_results: List[Dict[str, Any]], query: str) -> KnowledgeState:
        """Analyze a batch of search results asynchronously and update state."""
        self.state = state
        tasks = [self._analyze_single_url(result, query) for result in search_results]
        analyzed_urls = await asyncio.gather(*tasks, return_exceptions=True)

        extracted_urls = []
        for res in analyzed_urls:
            if isinstance(res, Exception):
                self.state.add_error(f"URL analysis error: {str(res)}")
            elif res:
                extracted_urls.append(res)

        # Filter, sort by priority, and add to state
        crawlable_urls = [url for url in extracted_urls if url.should_crawl]
        crawlable_urls.sort(key=lambda x: x.priority, reverse=True)

        final_urls = crawlable_urls[:self.config.max_urls]
        self.state.add_urls(final_urls, add_to_queue=True)
        return self.state

    async def _analyze_single_url(self, result: Dict[str, Any], query: str) -> Optional[ExtractedURL]:
        """Analyze a single URL asynchronously."""
        url = result.get('url', '')
        snippet = result.get('snippet', '')

        if not url:
            return None

        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            path = parsed.path.lower()

            # Use dynamic scoring
            domain_score = await self._calculate_domain_score(domain)
            relevance_score = await self._calculate_relevance_score(url, query, snippet)
            content_type_score = self._calculate_content_type_score(path)

            # Overall priority score (0-1)
            priority = (domain_score * 0.4 + relevance_score * 0.4 + content_type_score * 0.2)
            should_crawl = priority > self.config.priority_threshold and not self._is_low_value_url(url)

            # Determine content source from config
            content_source = self._determine_content_source(domain)

            return ExtractedURL(
                query=query,
                url=url,
                priority=priority,
                should_crawl=should_crawl,
                snippet=snippet,
                title=result.get('title'),
                source_engine=result.get('engine'),
                content_source=content_source,
                # Using a dummy value for now, could be dynamic
                estimated_crawl_time=5.0
            )

        except Exception as e:
            raise Exception(f"Failed to analyze URL {url}: {e}")

    async def _calculate_domain_score(self, domain: str) -> float:
        """Calculate domain authority score using dynamic config."""
        score = 0.3 # Base score
        for hv_domain, hv_score in self.config.high_value_domains.items():
            if hv_domain in domain:
                score += hv_score
                break
        return min(1.0, score)

    async def _calculate_relevance_score(self, url: str, query: str, snippet: str) -> float:
        """Calculate relevance score using either LLM or traditional method."""
        if self.config.enable_ml_scoring and self.llm:
            return await self._calculate_ml_relevance_score_llm(url, query, snippet)
        else:
            return self._calculate_traditional_relevance(url, query, snippet)

    def _calculate_traditional_relevance(self, url: str, query: str, snippet: str) -> float:
        """Fallback for traditional relevance scoring."""
        query_terms = query.lower().split()
        url_lower = url.lower()
        snippet_lower = snippet.lower()

        url_matches = sum(1 for term in query_terms if term in url_lower)
        snippet_matches = sum(1 for term in query_terms if term in snippet_lower)

        url_score = min(1.0, url_matches / len(query_terms)) if query_terms else 0
        snippet_score = min(1.0, snippet_matches / len(query_terms)) if snippet and query_terms else 0

        return (url_score * 0.3 + snippet_score * 0.7)

    async def _calculate_ml_relevance_score_llm(self, url: str, query: str, snippet: str) -> float:
        """Use LLM for semantic relevance scoring."""
        prompt = f"""
        Rate the relevance of this URL to the query on a scale of 0.0 to 1.0:
        Query: {query}
        URL: {url}
        Snippet: {snippet[:200]}

        Consider: semantic relevance, content authority, freshness
        Respond with just a number between 0.0 and 1.0
        """
        try:
            response = await self.llm.ainvoke(prompt)
            # Add robustness for non-numeric LLM output
            score = float(re.search(r"(\d+(\.\d+)?)", response.content).group(1))
            return max(0.0, min(1.0, score))
        except Exception as e:
            self.state.add_error(f"ML scoring failed for {url}: {str(e)}")
            return self._calculate_traditional_relevance(url, query, snippet)

    def _calculate_content_type_score(self, path: str) -> float:
        """Score based on URL path content type indicators."""
        high_value_patterns = [
            r'/article/', r'/blog/', r'/post/', r'/news/', r'/research/',
            r'/paper/', r'/guide/', r'/tutorial/', r'/doc/', r'/wiki/'
        ]

        low_value_patterns = [
            r'/login', r'/register', r'/cart', r'/checkout', r'/contact',
            r'/about', r'/privacy', r'/terms'
        ]

        for pattern in high_value_patterns:
            if re.search(pattern, path):
                return 0.8

        for pattern in low_value_patterns:
            if re.search(pattern, path):
                return 0.1

        return 0.5

    def _determine_content_source(self, domain: str) -> ContentSource:
        """Determine content source based on domain using dynamic config."""
        for domain_pattern, content_source in self.config.domain_to_content_source.items():
            if domain_pattern in domain:
                return content_source
        return ContentSource.OTHER

    def _is_low_value_url(self, url: str) -> bool:
        """Check if URL matches low-value patterns."""
        return any(re.search(pattern, url) for pattern in self.config.low_value_patterns)


Here is a test script to demonstrate the functionality of the `URLAnalyzer` and `URLAnalysisConfig`.

In [7]:
import asyncio
import nest_asyncio
import re
import time
from dataclasses import dataclass, field
from typing import List, Dict, Optional

nest_asyncio.apply()


# Init KnowledgeState
mock_state = KnowledgeState.create_knowledge_state("Latest advancements in AI?")

# Config
config = URLAnalysisConfig(
    max_urls=5,
    priority_threshold=0.4,
    enable_ml_scoring=False,  # flip to True if you wire up an async LLM
    high_value_domains={"openai.com": 0.9, "deepmind.google": 0.8},
    low_value_patterns=[r"forum", r"login", r"signup"]
)

# Analyzer
url_analyzer = URLAnalyzer(config=config)

# --- Real AI-related URLs for testing ---
search_results = [
    {
        "url": "https://openai.com/research/",
        "title": "OpenAI Research",
        "snippet": "Explore OpenAI’s latest advancements in artificial intelligence.",
        "engine": "google",
    },
    {
        "url": "https://deepmind.google/discover/blog",
        "title": "DeepMind Blog",
        "snippet": "Discover the latest AI research and breakthroughs from DeepMind.",
        "engine": "google",
    },
    {
        "url": "https://www.nature.com/collections/ai-ml",
        "title": "Nature AI & Machine Learning",
        "snippet": "Scientific publications on AI and ML progress.",
        "engine": "bing",
    },
    {
        "url": "https://arxiv.org/list/cs.AI/recent",
        "title": "arXiv AI Recent",
        "snippet": "Preprints covering artificial intelligence and machine learning research.",
        "engine": "duckduckgo",
    },
    {
        "url": "https://www.microsoft.com/en-us/research/research-area/artificial-intelligence/",
        "title": "Microsoft Research AI",
        "snippet": "Advancing AI to solve real-world problems.",
        "engine": "google",
    },
    {
        "url": "https://www.reddit.com/r/MachineLearning/",
        "title": "Reddit Machine Learning",
        "snippet": "Community forum discussions about AI/ML.",
        "engine": "bing",
    },
]

print("\nAnalyzing batch of search results...")
mock_state = asyncio.run(url_analyzer.analyze_batch(mock_state, search_results, mock_state.main_query))

print("\n=== Analysis complete ===")
for i, url_obj in enumerate(mock_state.extracted_urls, 1):
    print(f"{i}. {url_obj.url} | Priority: {url_obj.priority:.2f} | Crawl: {url_obj.should_crawl}")



Analyzing batch of search results...

=== Analysis complete ===
1. https://openai.com/research/ | Priority: 0.77 | Crawl: True
2. https://deepmind.google/discover/blog | Priority: 0.67 | Crawl: True


In [8]:
!pip install langchain_tavily langchain_core ddg duckduckgo_search htmldate google-search-results --quiet
!pip install -U ddgs --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
htmldate 1.9.3 requires lxml<6,>=5.3.0; platform_system != "Darwin" or python_version > "3.8", but you have lxml 6.0.1 which is incompatible.


In [9]:
warnings.filterwarnings( "ignore")



from htmldate import find_date
from datetime import datetime, timedelta
from pydantic import BaseModel, Field
from typing import ClassVar, Dict, Any, List, Optional, AsyncGenerator
from langchain_core.documents import Document
from langchain_tavily import TavilySearch
# from duckduckgo_search import DDGS  # Assuming DDGS is the modern async-compatible way
from langchain_community.tools import DuckDuckGoSearchResults # Import the LangChain DDG tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper # Import the DDG wrapper
from langchain_community.utilities import GoogleSerperAPIWrapper # Import Google Serper
# from KnowledgeState import KnowledgeState, DocumentSource # Assuming these are available
# from config import SearchConfig # Assuming SearchConfig is available
import asyncio
import logging
import re
import json # Import json

logger = logging.getLogger(__name__)

# -------------------- Pydantic Model for Search Results --------------------

class SearchResultSchema(BaseModel):
    """Schema for a single search result."""
    url: str = Field(..., description="The URL of the search result.")
    snippet: str = Field(..., description="A snippet of the content.")
    title: str = Field(..., description="The title of the document.")
    engine: str = Field(..., description="The search engine used.")
    query: str = Field(..., description="The query used to find this result.")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Additional metadata.")


# -------------------- Enhanced Search Engine with Async Integration --------------------
class SearchEngine:
    """Enhanced search class for async, state-aware operations."""

    # Remove state from __init__ as it will be passed to run_searches
    def __init__(self, config: SearchConfig):
        self.state = None
        self.config = config
        # Access clients from config
        self._tavily = config.tavily_client
        # self._ddgs = config.ddgs_client
        # Use the LangChain DuckDuckGo tool
        self._ddg_tool = DuckDuckGoSearchResults(
            api_wrapper=DuckDuckGoSearchAPIWrapper(region="wt-wt", time="y", max_results=config.external_max_results),
            source="text" # Use text source for general search
        )
        # Initialize Google Serper
        try:
            self._serper = GoogleSerperAPIWrapper()
        except Exception as e:
            logger.warning(f"Failed to initialize Google Serper: {e}")
            self._serper = None


        self.engines = {}
        if self._tavily:
            self.engines["tavily"] = self._tavily_search
        # if self._ddgs:
        #     self.engines["ddg"] = self._ddg_search
        if self._ddg_tool:
             self.engines["ddg_langchain"] = self._ddg_langchain_search
        if self._serper:
            self.engines["serper"] = self._serper_search


    # Modify run_searches to accept state as an argument AND return the state
    async def run_searches(self, state: KnowledgeState, queries: List[str], max_results_per_query: int = 10) -> KnowledgeState:
        """
        Runs multiple searches concurrently across all available engines, prioritizing recent data.
        Updates the passed-in state with search results and returns the updated state.
        """
        self.state = state
        if not queries:
            return self.state # Return state even if no queries

        all_tasks = []
        for query in queries:
            if self._tavily:
                # Pass query and max_results to individual search methods
                all_tasks.append(self._tavily_search(query=query, max_results=self.config.external_max_results, time_range="year"))
            # if self._ddgs:
            #      # Pass only expected arguments to DDG search
            #     all_tasks.append(self._ddg_search(query=query, max_results=self.config.external_max_results))
            if self._ddg_tool:
                 all_tasks.append(self._ddg_langchain_search(query=query, max_results=self.config.external_max_results))
            if self._serper:
                # Google Serper doesn't have a direct time_range param in the tool,
                # but the API wrapper might support 'tbs' parameter indirectly.
                # For now, we'll use the default search.
                all_tasks.append(self._serper_search(query=query, max_results=self.config.external_max_results))


        # Run all search tasks concurrently
        raw_results_list = await asyncio.gather(*all_tasks, return_exceptions=True)

        all_typed_results = []
        for raw_results in raw_results_list:
            if isinstance(raw_results, Exception):
                self.state.add_error(f"Search task failed: {str(raw_results)}") # Add error to the passed-in state
                continue
            all_typed_results.extend(raw_results)

        # Add logging here to check the raw results
        logger.info(f"SearchEngine.run_searches: Received {len(all_typed_results)} raw typed results from search engines.")
        # Log previews of the first few raw results
        for i, res in enumerate(all_typed_results[:3]):
            logger.debug(f"SearchEngine.run_searches - Raw Result {i+1}: Engine={res.engine}, Title={res.title[:50]}..., Snippet={res.snippet[:100]}...")


        # Convert SearchResultSchema to DocumentResult, ensuring URL and publication_date are included in metadata
        document_results = [
            DocumentResult(
                page_content=res.snippet,
                metadata={
                    **res.metadata, # Include existing metadata
                    "title": res.title,
                    "engine": res.engine,
                    "query": res.query,
                    "url": res.url, # Explicitly include URL
                    "publication_date": res.metadata.get("publication_date") # Explicitly include publication_date
                },
                source_type=DocumentSource.SEARCH_SNIPPET,
            )
            for res in all_typed_results
        ]

        # Add logging here to check the converted documents before adding to state
        logger.info(f"SearchEngine.run_searches: Converted {len(document_results)} raw results to DocumentResult.")
        # Log previews of the first few converted documents
        for i, doc in enumerate(document_results[:3]):
             logger.debug(f"SearchEngine.run_searches - Converted Doc {i+1}: URL={doc.metadata.get('url', 'N/A')}, Title={doc.metadata.get('title', 'N/A')[:50]}..., Snippet={doc.page_content[:100]}...")


        # Add results to the passed-in state object
        self.state.add_results(document_results, source=DocumentSource.SEARCH_SNIPPET)

        # Add logging after adding results to state
        logger.info(f"SearchEngine.run_searches: Added {len(document_results)} documents to state with source SEARCH_SNIPPET.")
        logger.info(f"SearchEngine.run_searches: State external_api_results count BEFORE returning: {len(self.state.external_api_results)}")

        # Return the updated state object
        return self.state


    async def _tavily_search(self, query: str, max_results: int, time_range: str = "year") -> List[SearchResultSchema]:
        """Tavily search implementation using async methods with a time filter."""
        try:
            # Tavily's `time_range` options are 'day', 'week', 'month', or 'year'.
            # We use 'year' to cover the last 3 years as requested.
            results = await self._tavily.ainvoke({"query": query, "max_results": max_results, "time_range": time_range})

            search_results = []
            if results and isinstance(results, list):
                for r in results:
                    url = r.get("url") or r.get("link")
                    snippet = r.get("snippet") or r.get("body")
                    title = r.get("title", "")

                    if url and snippet:
                        metadata = {
                            "raw_score": r.get("score", 0.0),
                            "publication_date": r.get("publication_time") # Note: You'll need to parse this if you want a datetime object
                        }
                        search_results.append(SearchResultSchema(
                            url=url,
                            snippet=snippet,
                            title=title,
                            engine="tavily",
                            query=query,
                            metadata=metadata
                        ))
            return search_results
        except Exception as e:
            logger.error(f"Tavily search error for query '{query}': {e}")
            raise e

    async def _ddg_langchain_search(self, query: str, max_results: int) -> List[SearchResultSchema]:
        """DuckDuckGo search using LangChain's DuckDuckGoSearchResults tool."""
        try:
            # The tool's result is a string, need to parse it.
            # The time filter is set in the wrapper during initialization.
            result_string = await self._ddg_tool.ainvoke(query)

            # Define a regex pattern to capture key-value pairs for each result entry
            # This pattern looks for "key: value" structure, handling spaces and different content types
            # It assumes results are separated by a pattern like ", key:" or start with "key:"
            # This is still heuristic and might need adjustment based on more examples.
            # Let's try a pattern that captures blocks like "title: ..., link: ..., snippet: ..."
            result_pattern = re.compile(r'title:\s*(.*?),\s*link:\s*(.*?),\s*snippet:\s*(.*?)(?:,|$)', re.DOTALL)

            results = []
            # Iterate over matches in the string
            for match in result_pattern.finditer(result_string):
                title = match.group(1).strip()
                link = match.group(2).strip()
                snippet = match.group(3).strip()

                if link and snippet:
                    # LangChain's DDG tool might not provide publication date directly in metadata
                    metadata = {
                        # No raw score from this tool by default
                        "publication_date": None # Placeholder - date extraction happens later
                    }
                    results.append(SearchResultSchema(
                        url=link, # Use link from regex
                        snippet=snippet,
                        title=title,
                        engine="ddg_langchain",
                        query=query,
                        metadata=metadata
                    ))

            if not results:
                 # If regex parsing didn't find any results, log the raw string
                 logger.warning(f"DuckDuckGo regex parsing found no results. Raw string: {result_string[:500]}...")
                 # Fallback to the previous heuristic parsing if regex fails completely
                 fallback_results = self._parse_ddg_fallback(result_string)
                 if fallback_results:
                      logger.info("DuckDuckGo regex parsing failed, but fallback parsing found results.")
                      # Convert fallback results to SearchResultSchema
                      results = [
                           SearchResultSchema(
                                url=r.get('url', ''),
                                snippet=r.get('snippet', ''),
                                title=r.get('title', ''),
                                engine="ddg_langchain_fallback", # Indicate fallback was used
                                query=query,
                                metadata={"publication_date": None}
                           ) for r in fallback_results if r.get('url') and r.get('snippet')
                      ]
                 else:
                      logger.error(f"Could not parse DuckDuckGo search result string using regex or fallback. Raw string: {result_string}") # Log full raw string on final failure
                      return [] # Return empty if both fail


            search_results = results # Use the results from regex or fallback


            return search_results
        except Exception as e:
            logger.error(f"LangChain DuckDuckGo search error for query '{query}': {e}. Raw string: {result_string}", exc_info=True) # Log exception with raw string
            raise e # Re-raise the exception

    def _parse_ddg_fallback(self, result_string: str) -> List[Dict[str, Any]]:
        """
        Fallback parsing for DuckDuckGo search results if regex or JSON load fails.
        Attempts to extract relevant fields using regex or simple splitting.
        This is a heuristic and may not be perfect for all formats.
        """
        results = []
        # Split the string by common delimiters or patterns indicating separate results
        # This is a rough heuristic. A more sophisticated parser might be needed.
        # Let's refine this to specifically look for "title: ..., link: ..., snippet: ..." blocks
        # This pattern is similar to the one in _ddg_langchain_search but might be used if the main one fails
        fallback_pattern = re.compile(r'title:\s*(.*?),\s*link:\s*(.*?),\s*snippet:\s*(.*?)(?:,|$)', re.DOTALL)

        for match in fallback_pattern.finditer(result_string):
             title = match.group(1).strip()
             link = match.group(2).strip()
             snippet = match.group(3).strip()
             if link and snippet:
                  results.append({'url': link, 'title': title, 'snippet': snippet})

        if not results:
             # If the more targeted fallback pattern fails, try the original split heuristic
             raw_entries = re.split(r'\s*,\s*(?=\w+:)', result_string) # Split by comma followed by key:
             current_entry = {}
             for item in raw_entries:
                  key_value_match = re.match(r'(\w+):\s*(.*)', item, re.DOTALL)
                  if key_value_match:
                       key = key_value_match.group(1)
                       value = key_value_match.group(2).strip()
                       current_entry[key] = value
                  if 'link' in current_entry and 'snippet' in current_entry and 'title' in current_entry:
                       # Found a complete set of fields, add to results
                       results.append(current_entry)
                       current_entry = {} # Reset for next entry

             # Clean up potential leading/trailing brackets or quotes from the original heuristic
             for entry in results:
                  for key in ['link', 'title', 'snippet']:
                       if key in entry:
                            entry[key] = entry[key].strip().strip('"') # Remove leading/trailing quotes
                            entry[key] = entry[key].replace('\\"', '"') # Simple fix for escaped quotes within values


        # Filter out entries that don't have minimum required fields
        cleaned_results = [
            {'url': r.get('link', r.get('url', '')), 'title': r.get('title', ''), 'snippet': r.get('snippet', '')}
            for r in results if r.get('link') or r.get('url') and r.get('snippet') and r.get('title')
        ]
        return cleaned_results


    async def _serper_search(self, query: str, max_results: int) -> List[SearchResultSchema]:
        """Google Serper search implementation."""
        try:
            # Serper API wrapper doesn't directly support max_results in invoke,
            # it's configured during initialization or via specific parameters.
            # We'll use the default for now or explore adding a custom run method.
            # For this example, we'll call the sync method in a thread.
            results = await asyncio.to_thread(self._serper.results, query)

            search_results = []
            if results and 'organic' in results:
                for r in results['organic']:
                    url = r.get("link") or r.get("url")
                    snippet = r.get("snippet") or r.get("description") # Serper uses 'description'
                    title = r.get("title", "")

                    if url and snippet:
                        # Serper often provides a date
                        publication_date_str = r.get("date")
                        metadata = {
                            "publication_date": publication_date_str
                        }
                        search_results.append(SearchResultSchema(
                            url=url,
                            snippet=snippet,
                            title=title,
                            engine="serper",
                            query=query,
                            metadata=metadata
                        ))
            return search_results
        except Exception as e:
            logger.error(f"Google Serper search error for query '{query}': {e}")
            raise e

Here is a test script to demonstrate the functionality of the `SearchEngine` class.

In [10]:
import warnings
warnings.filterwarnings("ignore")

import asyncio
import nest_asyncio
import os
from langchain_core.documents import Document
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper, GoogleSerperAPIWrapper

# If Tavily is installed:
# from langchain_tavily import TavilySearch

nest_asyncio.apply()


# ---------------- Main ----------------

# Init KnowledgeState
mock_state = KnowledgeState.create_knowledge_state("What is Artificial Intelligence?")

# Init Tavily client if API key exists
tavily_api_key = os.environ.get("TAVILY_API_KEY")
if tavily_api_key:
    from langchain_tavily import TavilySearch
    tavily_client = TavilySearch(api_key=tavily_api_key)
else:
    tavily_client = None

config = SearchConfig(
    external_max_results=5,
    subquery_max_search=2,
    subquery_results_per_query=3,
    tavily_client=tavily_client
)

search_engine = SearchEngine(config=config)

queries_to_run = ["latest AI models", "future of machine learning"]

mock_state = asyncio.run(search_engine.run_searches(mock_state, queries_to_run, max_results_per_query=3))

print("\n=== Results ===")
for i, doc in enumerate(mock_state.external_api_results[:10]):
    print(f"{i+1}. [{doc.metadata['engine']}] {doc.metadata['title']} -> {doc.metadata['url']}")


INFO:__main__:SearchEngine.run_searches: Received 26 raw typed results from search engines.
INFO:__main__:SearchEngine.run_searches: Converted 26 raw results to DocumentResult.
INFO:__main__:SearchEngine.run_searches: Added 26 documents to state with source SEARCH_SNIPPET.
INFO:__main__:SearchEngine.run_searches: State external_api_results count BEFORE returning: 26


Added 26 results from DocumentSource.SEARCH_SNIPPET

=== Results ===
1. [ddg_langchain] The hottest AI models, what they do, and how to use … -> https://techcrunch.com/2025/03/30/the-hottest-ai-models-what-they-do-and-how-to-use-them/
2. [ddg_langchain] Top AI Models List - Geekflare -> https://geekflare.com/blog/top-ai-models-list/
3. [ddg_langchain] Top Generative AI Models to Explore in 2025 -> https://www.geeksforgeeks.org/blogs/generative-ai-models/
4. [serper] Models -> https://deepmind.google/models/
5. [serper] Comparison of Models: Intelligence, Performance & Price ... -> https://artificialanalysis.ai/models
6. [serper] Two in-house models in support of our mission | Microsoft AI -> https://microsoft.ai/news/two-new-in-house-models/
7. [serper] What AI Models are expected to be released in the next 2 ... -> https://www.reddit.com/r/singularity/comments/1lnayv6/what_ai_models_are_expected_to_be_released_in_the/
8. [serper] The Top 5 AI Models of 2025: What's New and How to Use 

In [11]:
!pip install playwright --quiet
!playwright install --with-deps chromium firefox # Install necessary browser binaries

!pip install --upgrade htmldate --quiet

Installing dependencies...
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,623 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,581

In [12]:
!pip install --upgrade htmldate --quiet # Ensure latest version

import time
import asyncio
import logging
from typing import Dict, Any, List, Optional, Protocol, Union
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, AsyncChromiumLoader
import bs4
import re
import httpx
from pydantic import BaseModel, Field
from datetime import datetime, timezone
from htmldate import find_date # New import for date extraction
import dateutil.parser # Needed for flexible date parsing

# -------------------- Async Crawler Protocol --------------------

class AsyncCrawlerProtocol(Protocol):
    """Protocol for an asynchronous web crawler."""
    async def crawl_urls_batch(self, urls_to_crawl: List[ExtractedURL]) -> None:
        """Asynchronously crawl a batch of URLs and update state."""
        ...

class CrawlerEngine(AsyncCrawlerProtocol):
    """
    Enhanced, state-aware, async crawler for intelligent knowledge acquisition.
    Uses AsyncChromiumLoader for dynamic sites and falls back to WebBaseLoader,
    now including logic to prioritize recently published content and using embeddings
    for content relevance scoring.
    """
    def __init__(self, config: CrawlerConfig, embedding_model: Optional[Any] = None): # Added embedding_model
        self.state = None
        self.config = config
        self.embedding_model = embedding_model # Store embedding model
        self.semaphore = asyncio.Semaphore(self.config.max_concurrent)
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap
        )
        self.time_threshold = datetime.now(timezone.utc) - timedelta(days=3 * 365) # For 3 years

        if self.config.enable_embedding_content_scoring and not self.embedding_model:
             logger.warning("Embedding content scoring enabled but no embedding model provided. Disabling.")
             self.config.enable_embedding_content_scoring = False


    async def crawl_urls_batch(self, state: KnowledgeState, urls_to_crawl: List[ExtractedURL]) -> KnowledgeState:
        """
        Crawl a list of ExtractedURLs, process the content, and update the state.
        This method is the main entry point for crawling within the LangGraph node.
        """
        self.state = state

        if not urls_to_crawl:
            return self.state

        # Sort URLs to prioritize those with fewer retries and higher priority
        # Simple approach: sort by retry_count (ascending) then by priority (descending)
        urls_to_process_in_batch = sorted(urls_to_crawl, key=lambda x: (x.retry_count, -x.priority))
        urls_to_process_in_batch = urls_to_process_in_batch[:self.config.max_concurrent] # Take only max_concurrent

        # Create a set of URLs being processed in this batch for easy removal later
        urls_being_processed_set = {url.url for url in urls_to_process_in_batch}


        crawl_tasks = [self._crawl_single_url(url_info) for url_info in urls_to_process_in_batch]
        results = await asyncio.gather(*crawl_tasks, return_exceptions=True)

        successful_crawls = 0
        failed_crawls = 0
        total_time = 0.0

        # Process results and handle retries
        processed_urls_count = len(urls_to_process_in_batch)

        # Remove the URLs processed in this batch from the state's crawl_queue
        # This needs to be done carefully to avoid issues with modifying a list while iterating
        # A robust way is to rebuild the queue or filter it.
        initial_queue = list(self.state.crawl_queue) # Get a snapshot
        self.state.crawl_queue.clear() # Clear the queue

        # Rebuild the queue with URLs NOT in the processed batch and any that were re-queued by _crawl_single_url
        # URLs re-queued by _crawl_single_url are already added back to state.crawl_queue in that method.
        # So, we just need to add back the URLs from the initial queue that were *not* processed in this batch.
        for url_info in initial_queue:
            if url_info.url not in urls_being_processed_set:
                self.state.crawl_queue.append(url_info)

        # Now process the results from the batch
        for result in results:
            if isinstance(result, Exception):
                # Error was handled within _crawl_single_url (either retried or discarded)
                failed_crawls += 1 # Still count as a failed attempt for stats
            elif result:
                # Successful crawl result (Document, crawl_time)
                document, crawl_time = result

                # Check publication date before further processing
                publication_date_str = document.metadata.get("publication_date")
                if publication_date_str:
                    try:
                        # Ensure date parsing is robust to different formats
                        pub_date = dateutil.parser.parse(publication_date_str)
                        # Ensure comparison is between timezone-aware datetimes
                        if pub_date.tzinfo is None:
                            # Assume UTC if no timezone info
                            pub_date = pub_date.replace(tzinfo=timezone.utc)

                        if pub_date < self.time_threshold:
                            logger.info(f"Skipping older document from {document.metadata.get('url')}")
                            continue # Skip processing this document
                    except (ValueError, TypeError) as e:
                        logger.warning(f"Could not parse or compare publication date '{publication_date_str}': {e}")

                successful_crawls += 1
                total_time += crawl_time

                # Split and add documents to state
                chunks = await self._split_document(document)

                if self.config.enable_embedding_content_scoring and self.embedding_model:
                    # Use embedding-based scoring
                    scored_chunks = await self._score_content_embedding(chunks, self.state.main_query)
                    relevant_chunks = [
                        c for c in scored_chunks
                        if c.metadata.get("embedding_relevance_score", 0) >= self.config.embedding_relevance_threshold
                    ]
                    if relevant_chunks:
                        self.state.add_results(relevant_chunks, source=DocumentSource.WEB_CRAWL)
                    else:
                         logger.info(f"No relevant chunks found after embedding scoring for {document.metadata.get('url')}")
                else:
                    # Add all chunks if embedding scoring is disabled or not possible
                    self.state.add_results(chunks, source=DocumentSource.WEB_CRAWL)

        self.state.update_crawl_stats(
            attempted=processed_urls_count, # Use the count of URLs processed in this batch
            successful=successful_crawls,
            failed=failed_crawls,
            time=total_time
        )

        # URLs that were successfully crawled or discarded are implicitly removed
        # because they are not re-added to the queue by _crawl_single_url.
        # Only URLs needing retry are added back by _crawl_single_url.
        return self.state # return updated state


    async def _crawl_single_url(self, url_info: ExtractedURL) -> Optional[tuple[Document, float]]:
        """Crawl a single URL safely and return the result or None if retried/discarded."""
        url = url_info.url
        start_time = time.perf_counter()

        try:
            async with self.semaphore:
                # Check if the URL has already been retried too many times
                if url_info.retry_count >= self.config.max_retries: # Use >= for correct check
                    logger.warning(f"Discarding URL {url} after {url_info.retry_count} retries.")
                    # Do not return anything for this URL, it's discarded
                    return None

                document = await self._load_and_extract(url)

                if not document or not document.page_content:
                    self.state.add_error(f"No content extracted from {url}")
                    return None

                end_time = time.perf_counter()
                crawl_time = end_time - start_time

                if len(document.page_content) > self.config.max_content_length:
                    document.page_content = document.page_content[:self.config.max_content_length] + "..."
                    document.metadata["truncated"] = True

                document.metadata.update({
                    "crawl_timestamp": datetime.now(timezone.utc).isoformat(),
                    "content_length": len(document.page_content),
                    "crawl_time": crawl_time,
                    "url": url,
                    "query": self.state.main_query
                })

                # Add publication and modification dates to metadata
                publication_date = await self._extract_publication_date(document.page_content)
                if publication_date:
                    document.metadata["publication_date"] = publication_date

                modification_date = await self._extract_modification_date(document.page_content)
                if modification_date:
                    document.metadata["modification_date"] = modification_date

                # Reset retry count on successful crawl
                url_info.retry_count = 0

                return (document, crawl_time)

        except (httpx.TimeoutException, httpx.ConnectError) as e:
            # Handle specific network errors for retry
            url_info.retry_count += 1
            if url_info.retry_count <= self.config.max_retries:
                logger.warning(f"Network error for {url} (attempt {url_info.retry_count}/{self.config.max_retries}). Re-queueing.")
                # Add the URL back to the crawl queue for retry
                self.state.crawl_queue.append(url_info)
                # Do not return anything for this URL yet, it's being retried
                return None
            else:
                logger.error(f"Network error for {url} after {url_info.retry_count} retries. Discarding URL. Error: {e}")
                self.state.add_error(f"Failed to crawl {url} after retries: {str(e)}")
                # Do not return anything, it's discarded
                return None

        except Exception as e:
            # Handle other exceptions (parsing errors, etc.) - do not retry for these
            logger.error(f"Failed to crawl {url} due to unexpected error: {e}")
            self.state.add_error(f"Failed to crawl {url}: {str(e)}")
            # For other errors, we don't retry and don't return content
            return None


    async def _load_and_extract(self, url: str) -> Optional[Document]:
        """
        Attempts to load a document using AsyncChromiumLoader, falling back to WebBaseLoader.
        Includes basic error handling for URL loading.
        """
        doc = None
        # Try AsyncChromiumLoader first for dynamic content
        try:
            loader = AsyncChromiumLoader([url])
            docs = await loader.aload()
            if docs and docs[0].page_content and docs[0].page_content.strip():
                docs[0].metadata["loader"] = "chromium"
                doc = docs[0]
                logger.info(f"Successfully loaded {url} with ChromiumLoader.")
            else:
                 logger.warning(f"Chromium loader got empty content for {url}.")
        except Exception as e:
            # Log the specific error but don't fail the crawl task yet
            logger.warning(f"Chromium loader failed for {url}: {e}")

        # Fallback to WebBaseLoader if Chromium fails or is not installed, or returned empty content
        if not doc:
            try:
                def blocking_load():
                     return WebBaseLoader(web_paths=(url,)).load()
                docs = await asyncio.to_thread(blocking_load)
                if docs and docs[0].page_content and docs[0].page_content.strip():
                    docs[0].metadata["loader"] = "webbase"
                    doc = docs[0]
                    logger.info(f"Successfully loaded {url} with WebBaseLoader.")
                else:
                    logger.warning(f"WebBaseLoader got empty content for {url}.")
            except Exception as e:
                logger.warning(f"WebBaseLoader failed for {url}: {e}")
                # If both loaders fail, return None
                return None

        return doc


    async def _extract_publication_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the publication date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        """
        try:
            # Using original_date=True to prioritize explicitly marked dates
            # Removed timeout argument as it seems unsupported or causing issues
            return await asyncio.to_thread(find_date, html_content, original_date=True)
        except Exception as e:
            # Log htmldate specific errors
            logger.warning(f"Failed to extract publication date with htmldate: {e}")
            return None

    async def _extract_modification_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the modification date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        Note: htmldate's ability to find modification dates might be limited or require specific arguments not available in all versions.
        Trying without specific arguments first, or exploring alternative libraries might be necessary.
        """
        try:
            # Calling find_date without the problematic 'lastmod=True' or 'timeout'
            # It might still find a date, but less likely to be the modification date specifically
            return await asyncio.to_thread(find_date, html_content)
        except Exception as e:
            logger.warning(f"Failed to extract modification date with htmldate: {e}")
            return None


    async def _split_document(self, doc: Document) -> List[Document]:
        """Split a document into chunks asynchronously."""
        return await asyncio.to_thread(self.splitter.split_documents, [doc])

    async def _score_content_embedding(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use embeddings to score content chunks against the query based on semantic similarity.
        Returns the chunks with an added 'embedding_relevance_score' in their metadata.
        """
        if not self.embedding_model or not chunks:
            return chunks

        try:
            # Embed the query
            query_embedding = await asyncio.to_thread(self.embedding_model.embed_query, query)

            # Embed the documents/chunks
            # Embedding a list of documents can be done in a single call
            chunk_texts = [chunk.page_content for chunk in chunks]
            # Corrected the call to embed_documents - it should be a method call, not subscripting
            chunk_embeddings = await asyncio.to_thread(self.embedding_model.embed_documents, chunk_texts)

            # Calculate cosine similarity between query embedding and each chunk embedding
            # Cosine similarity ranges from -1 (opposite) to 1 (identical).
            # We want scores closer to 1.0.
            scores = []
            for chunk_embedding in chunk_embeddings:
                # Using dot product for cosine similarity if embeddings are normalized
                # If not normalized, need to normalize or use a different similarity metric
                # SentenceTransformer embeddings are usually normalized.
                score = sum(q * c for q, c in zip(query_embedding, chunk_embedding))
                # Ensure score is within [0, 1] range if needed, though cosine similarity is [-1, 1]
                # Map [-1, 1] to [0, 1] for thresholding: (score + 1) / 2
                normalized_score = (score + 1) / 2
                scores.append(normalized_score)


            # Add scores to chunk metadata
            for i, chunk in enumerate(chunks):
                chunk.metadata["embedding_relevance_score"] = scores[i]

            return chunks

        except Exception as e:
            logger.error(f"Embedding scoring failed: {e}")
            # If embedding fails, return original chunks without scores or with a default low score
            for chunk in chunks:
                 chunk.metadata["embedding_relevance_score"] = 0.0 # Assign a low score on failure
            return chunks


    async def _score_content_llm(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use an LLM to score content chunks against the query.
        Returns the chunks with an added 'llm_relevance_score' in their metadata.
        (Kept for reference if needed, but embedding is preferred as per user feedback)
        """
        if not self.llm or not chunks:
            return chunks

        scoring_prompt = """
        You are a quality assurance assistant for a web crawler. Your task is to score document chunks based on their relevance to a given query, and prioritize more recent content.
        Provide a single relevance score from 0.0 to 1.0 for each chunk.
        Take into account both the topical relevance and the recency of the content.
        Query: {query}

        Example Output Format:
        [
          {{ "chunk_index": 0, "score": 0.85 }},
          {{ "chunk_index": 1, "score": 0.20 }},
          ...
        ]

        Now, score the following chunks:
        """

        batch_size = 5
        for i in range(0, len(chunks), batch_size):
            chunk_batch = chunks[i:i+batch_size]
            prompt = scoring_prompt.format(query=query)
            for j, chunk in enumerate(chunk_batch):
                # Include date in the prompt for LLM context
                date_info = f"Publication Date: {chunk.metadata.get('publication_date', 'N/A')}\nModification Date: {chunk.metadata.get('modification_date', 'N/A')}\n"
                prompt += f"\n--- Chunk {i+j} ---\n{date_info}{chunk.page_content[:500]}...\n" # Corrected typo here
            try:
                response = await self.llm.ainvoke(prompt)
                scores = self._parse_llm_response(response.content)

                for score_info in scores:
                    chunk_index = score_info.get("chunk_index")
                    score = score_info.get("score")
                    if chunk_index is not None and score is not None and 0 <= chunk_index < len(chunks):
                         chunks[chunk_index].metadata["llm_relevance_score"] = score
            except Exception as e:
                logger.error(f"LLM scoring failed for batch {i}: {e}")

        return chunks

    def _parse_llm_response(self, text: str) -> List[Dict[str, Union[int, float]]]:
        """Parses the LLM's response to extract scores."""
        try:
            json_match = re.search(r'\[\s*\{.*?\}\s*\]', text, re.DOTALL)
            if json_match:
                # Use json.loads for safer parsing
                import json
                return json.loads(json_match.group(0))
        except Exception as e:
            logger.error(f"Failed to parse LLM response: {e}")
            raise e
        return []

Here is a test script to demonstrate the functionality of the `CrawlerEngine` class. It includes several edge cases.

In [ ]:
import asyncio
import nest_asyncio
import httpx
import re
import time
import logging
from typing import List, Dict, Optional
from dataclasses import dataclass, field
from datetime import datetime, timezone
from bs4 import BeautifulSoup
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_core.documents import Document
from htmldate import find_date
import dateutil.parser

nest_asyncio.apply()
logging.basicConfig(level=logging.INFO)


# # ---------------- KnowledgeState ----------------
# class KnowledgeState:
#     def __init__(self, query: str):
#         self.query = query
#         self.crawl_queue: List["ExtractedURL"] = []
#         self.crawled_results: List[Document] = []
#         self.stats = {"attempted": 0, "successful": 0, "failed": 0, "time": 0.0}

#     classmethod
#     def create_knowledge_state(cls, query: str):
#         return cls(query)

#     def add_results(self, docs: List[Document]):
#         self.crawled_results.extend(docs)

#     def add_error(self, error: str):
#         logging.error(error)

#     def update_crawl_stats(self, attempted: int, successful: int, failed: int, elapsed: float):
#         self.stats["attempted"] += attempted
#         self.stats["successful"] += successful
#         self.stats["failed"] += failed
#         self.stats["time"] += elapsed

#     def __repr__(self):
#         return f"<KnowledgeState query={self.query}, crawled={len(self.crawled_results)}, queue={len(self.crawl_queue)}>"


# # ---------------- Config ----------------
# dataclass
# class CrawlerConfig:
#     chunk_size: int = 500
#     chunk_overlap: int = 100
#     max_concurrent: int = 3
#     timeout: int = 10
#     max_content_length: int = 20000
#     enable_embedding_content_scoring: bool = False
#     embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
#     embedding_relevance_threshold: float = 0.4
#     max_retries: int = 2


# # ---------------- ExtractedURL ----------------
# dataclass
# class ExtractedURL:
#     query: str
#     url: str
#     priority: float
#     should_crawl: bool
#     retry_count: int = 0


# # ---------------- CrawlerEngine ----------------
# class CrawlerEngine:
#     def __init__(self, config: CrawlerConfig, embedding_model=None):
#         self.config = config
#         self.embedding_model = embedding_model

#     async def fetch_url(self, url: str) -> Optional[str]:
#         try:
#             async with httpx.AsyncClient(timeout=self.config.timeout) as client:
#                 resp = await client.get(url)
#                 resp.raise_for_status()
#                 return resp.text[: self.config.max_content_length]
#         except Exception as e:
#             logging.warning(f"Fetch failed for {url}: {e}")
#             return None

#     def extract_text(self, html: str) -> str:
#         soup = BeautifulSoup(html, "html.parser")
#         for script in soup(["script", "style", "noscript"]):
#             script.extract()
#         return " ".join(soup.get_text(separator=" ").split())

#     def extract_date(self, html: str, url: str) -> Optional[str]:
#         try:
#             date_str = find_date(url, html=html)
#             if date_str:
#                 return dateutil.parser.parse(date_str).isoformat()
#         except Exception:
#             return None
#         return None

#     async def crawl_single(self, url_info: ExtractedURL, state: KnowledgeState):
#         start = time.time()
#         html = await self.fetch_url(url_info.url)
#         elapsed = time.time() - start

#         state.stats["attempted"] += 1

#         if not html:
#             url_info.retry_count += 1
#             if url_info.retry_count < self.config.max_retries:
#                 state.crawl_queue.append(url_info)  # retry
#             else:
#                 state.stats["failed"] += 1
#             return

#         text = self.extract_text(html)
#         date = self.extract_date(html, url_info.url)

#         splitter = RecursiveCharacterTextSplitter(
#             chunk_size=self.config.chunk_size,
#             chunk_overlap=self.config.chunk_overlap,
#         )
#         docs = splitter.create_documents([text])

#         # Add metadata
#         for d in docs:
#             d.metadata.update({
#                 "url": url_info.url,
#                 "query": url_info.query,
#                 "publication_date": date
#             })

#         # Optional embedding scoring
#         if self.config.enable_embedding_content_scoring and self.embedding_model:
#             embeddings = self.embedding_model.embed_documents([d.page_content for d in docs])
#             filtered_docs = []
#             for doc, emb in zip(docs, embeddings):
#                 # crude "relevance" by embedding norm threshold
#                 score = sum(x * x for x in emb) ** 0.5
#                 if score >= self.config.embedding_relevance_threshold:
#                     filtered_docs.append(doc)
#             docs = filtered_docs

#         if docs:
#             state.add_results(docs)
#             state.stats["successful"] += 1
#         else:
#             state.stats["failed"] += 1

#         state.stats["time"] += elapsed

#     async def crawl_urls_batch(self, state: KnowledgeState, urls: List[ExtractedURL]):
#         batch = urls[: self.config.max_concurrent]
#         state.crawl_queue = [u for u in state.crawl_queue if u not in batch]

#         await asyncio.gather(*(self.crawl_single(u, state) for u in batch))
#         return state


# # ---------------- Main ----------------
# if __name__ == "__main__":
# Init KnowledgeState
mock_state = KnowledgeState.create_knowledge_state("Latest advancement in AI")

# Config
config = CrawlerConfig(
    chunk_size=500,
    chunk_overlap=100,
    max_concurrent=2,
    timeout=5,
    max_content_length=10000,
    enable_embedding_content_scoring=True,
    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
    embedding_relevance_threshold=0.4,
    max_retries=2
)
print("Crawler Config:", config)

# Embedding model
try:
    real_embedding_model = SentenceTransformerEmbeddings(model_name=config.embedding_model_name)
    print("Initialized embedding model.")
except Exception as e:
    print(f"Embedding init failed: {e}, disabling scoring.")
    real_embedding_model = None
    config.enable_embedding_content_scoring = False

# Engine
crawler_engine = CrawlerEngine(config=config, embedding_model=real_embedding_model)

# Crawl targets
urls_to_crawl_initial = [
    ExtractedURL(query=mock_state.main_query, url="https://microsoft.ai/news/two-new-in-house-models/", priority=0.9, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://deepmind.google/discover/blog", priority=0.85, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://huggingface.co/", priority=0.7, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://github.com/langchain/", priority=0.2, should_crawl=False),  # should not crawl
    ExtractedURL(query=mock_state.main_query, url="https://microsoft.ai/about/", priority=0.9, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://deepmind.google/models/", priority=0.85, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://huggingface.co/", priority=0.7, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/6", priority=0.6, should_crawl=True), # Will timeout, needs retry 1
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/7", priority=0.6, should_crawl=True), # Will timeout, needs retry 1
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/8", priority=0.5, should_crawl=True, retry_count=config.max_retries - 1),

]

# add to queue
mock_state.crawl_queue.extend([u for u in urls_to_crawl_initial if u.should_crawl])

# Run crawler loop
iterations = len(mock_state.crawl_queue)
for i in range(iterations):
    if not mock_state.crawl_queue:
        print("Queue empty. Stopping.")
        break
    print(f"\n--- Iteration {i+1} ---")
    snapshot = list(mock_state.crawl_queue)
    mock_state = asyncio.run(crawler_engine.crawl_urls_batch(mock_state, snapshot))

    print(f"Queue after iteration {i+1}: {len(mock_state.crawl_queue)} remaining")

# Summary
print("\n=== Crawl Summary ===")
print("Stats:", mock_state.crawl_status)
print(f"Total docs stored: {len(mock_state.crawler_results)}")
for d in mock_state.crawler_results[:5]:
    print(f"- Doc from {d.metadata['url']} | Date: {d.metadata.get('publication_date')}")


Crawler Config: chunk_size=500 chunk_overlap=100 max_concurrent=2 timeout=5 max_content_length=10000 enable_embedding_content_scoring=True embedding_model_name='sentence-transformers/all-MiniLM-L6-v2' embedding_relevance_threshold=0.4 max_retries=2


Initialized embedding model.

--- Iteration 1 ---


INFO:__main__:Successfully loaded https://microsoft.ai/news/two-new-in-house-models/ with ChromiumLoader.
INFO:__main__:Successfully loaded https://microsoft.ai/about/ with ChromiumLoader.


Added 28 results from DocumentSource.WEB_CRAWL


Added 27 results from DocumentSource.WEB_CRAWL
Queue after iteration 1: 7 remaining

--- Iteration 2 ---


INFO:__main__:Successfully loaded https://deepmind.google/discover/blog with ChromiumLoader.
INFO:__main__:Successfully loaded https://deepmind.google/models/ with ChromiumLoader.


In [ ]:
import asyncio
import math
from typing import List, Optional, Callable
from pydantic import ValidationError
from sklearn.cluster import AgglomerativeClustering, KMeans  # optional

class SemanticChunker:
    """
    Orchestrates semantic chunking of documents into smaller DocumentResult objects.
    Handles embeddings, clustering/greedy grouping, and fallbacks.
    """

    def __init__(
        self,
        embedding_model,
        config,
        state,
        logger,
        sentence_splitter: Optional[Callable[[str], List[str]]] = None,
        tokenizer: Optional[Callable[[str], int]] = None,
    ):
        """
        Args:
            embedding_model: Model used for sentence embeddings.
            config: Holds chunking settings (size, overlap, mode, etc.).
            state: Object that supports .add_error(msg).
            logger: Logger instance.
            sentence_splitter: Callable to split text into sentences.
            tokenizer: Callable to count tokens in text.
        """
        self.embedding_model = embedding_model
        self.config = config
        self.state = state
        self.logger = logger
        self.sentence_splitter = sentence_splitter
        self.tokenizer = tokenizer

    # ------------------------ Public API ------------------------

    async def chunk(self, doc: "DocumentResult") -> List["DocumentResult"]:
        """
        Entry point: chunk a document using semantic methods with fallbacks.
        """
        doc_id = doc.metadata.get("url") or doc.metadata.get("source") or "unknown"
        self.logger.debug(f"[SemanticChunker] Starting chunking for {doc_id}")

        if not self.embedding_model:
            self.logger.warning(f"[SemanticChunker] No embedding model for {doc_id}, falling back to recursive.")
            return await self._safe_recursive(doc, "no embedding model")

        text = (doc.page_content or "").strip()
        if not text:
            self.logger.debug(f"[SemanticChunker] Empty content for {doc_id}, returning [].")
            return []

        sentences = self._split_sentences(text, doc_id)
        if len(sentences) <= 1:
            self.logger.debug(f"[SemanticChunker] Not enough sentences for {doc_id}, falling back to recursive.")
            return await self._safe_recursive(doc, "not enough sentences")

        # Choose mode
        mode = getattr(self.config, "semantic_chunking_mode", "clustering")
        chunks: List["DocumentResult"] = []

        if mode == "clustering":
            chunks = await self._clustering_mode(doc, sentences, doc_id)
            if not chunks:  # fallback to greedy if clustering fails
                self.logger.warning(f"[SemanticChunker] Clustering failed for {doc_id}, switching to greedy.")
                chunks = await self._greedy_mode(doc, sentences, doc_id)
        else:
            chunks = await self._greedy_mode(doc, sentences, doc_id)

        self.logger.debug(f"[SemanticChunker] Finished for {doc_id}, produced {len(chunks)} chunks.")
        return chunks

    # ------------------------ Sentence Prep ------------------------

    def _split_sentences(self, text: str, doc_id: str) -> List[str]:
        try:
            if self.sentence_splitter:
                return [s.strip() for s in self.sentence_splitter(text) if s.strip()]
            return [s.strip() + ('.' if not s.endswith('.') else '') for s in text.split('. ') if s.strip()]
        except Exception as e:
            self.logger.warning(f"[SemanticChunker] Sentence splitter failed for {doc_id}, fallback to naive: {e}", exc_info=True)
            return [s.strip() for s in text.split('.') if s.strip()]

    def _count_tokens(self, text: str) -> int:
        try:
            if self.tokenizer:
                return self.tokenizer(text)
        except Exception as e:
            self.logger.warning(f"[SemanticChunker] Tokenizer failed: {e}", exc_info=True)
        return max(1, len(text.split()))  # fallback approx

    # ------------------------ Embedding ------------------------

    async def _embed_sentences(self, sentences: List[str], doc_id: str) -> Optional[List[List[float]]]:
        try:
            if asyncio.iscoroutinefunction(self.embedding_model.embed_documents):
                return await self.embedding_model.embed_documents(
                    sentences,
                    batch_size=getattr(self.config, "embedding_batch_size", None),
                )
            return await asyncio.to_thread(self.embedding_model.embed_documents, sentences)
        except Exception as e:
            err = f"[SemanticChunker] Embedding failed for {doc_id}: {e}"
            self.logger.error(err, exc_info=True)
            self.state.add_error(err)
            return None

    # ------------------------ Modes ------------------------

    async def _clustering_mode(self, doc, sentences: List[str], doc_id: str) -> List["DocumentResult"]:
        """
        Cluster sentences based on embeddings and group them into chunks.
        """
        embeddings = await self._embed_sentences(sentences, doc_id)
        if not embeddings:
            return []

        try:
            token_counts = [self._count_tokens(s) for s in sentences]
            target_tokens = getattr(self.config, "chunk_size_tokens", 400)

            total_tokens = sum(token_counts)
            est_clusters = max(1, int(math.ceil(total_tokens / target_tokens)))
            est_clusters = min(est_clusters, len(sentences))

            algo = getattr(self.config, "clustering_algo", "agglo")
            if algo == "kmeans":
                clusterer = KMeans(n_clusters=est_clusters, random_state=42)
            else:
                clusterer = AgglomerativeClustering(n_clusters=est_clusters)

            labels = clusterer.fit_predict(embeddings)

            # group by contiguous labels
            grouped, cur, cur_label = [], [sentences[0]], labels[0]
            for lbl, sent in zip(labels[1:], sentences[1:]):
                if lbl == cur_label:
                    cur.append(sent)
                else:
                    grouped.append(" ".join(cur))
                    cur, cur_label = [sent], lbl
            if cur:
                grouped.append(" ".join(cur))

            return self._split_and_wrap(grouped, doc, doc_id)

        except Exception as e:
            err = f"[SemanticChunker] Clustering mode failed for {doc_id}: {e}"
            self.logger.error(err, exc_info=True)
            self.state.add_error(err)
            return []

    async def _greedy_mode(self, doc, sentences: List[str], doc_id: str) -> List["DocumentResult"]:
        """
        Merge sentences until chunk_size is reached. Simple & reliable.
        """
        try:
            target = getattr(self.config, "chunk_size_tokens", 400)
            overlap = getattr(self.config, "chunk_overlap_tokens", 50)

            chunks, cur_sents, cur_tokens = [], [], 0
            for s in sentences:
                tok = self._count_tokens(s)
                if cur_tokens + tok > target and cur_sents:
                    chunks.extend(self._wrap_chunk(" ".join(cur_sents), doc, doc_id))
                    # overlap
                    if overlap:
                        words = " ".join(cur_sents).split()
                        keep = max(1, int(overlap * len(words) / max(1, self._count_tokens(" ".join(cur_sents)))))
                        cur_sents, cur_tokens = [" ".join(words[-keep:])], self._count_tokens(" ".join(words[-keep:]))
                    else:
                        cur_sents, cur_tokens = [], 0
                cur_sents.append(s)
                cur_tokens += tok

            if cur_sents:
                chunks.extend(self._wrap_chunk(" ".join(cur_sents), doc, doc_id))

            return chunks
        except Exception as e:
            err = f"[SemanticChunker] Greedy mode failed for {doc_id}: {e}"
            self.logger.error(err, exc_info=True)
            self.state.add_error(err)
            return await self._safe_recursive(doc, "greedy failed")

    # ------------------------ Chunk Construction ------------------------

    def _wrap_chunk(self, text: str, doc, doc_id: str) -> List["DocumentResult"]:
        try:
            return [DocumentResult(
                page_content=text.strip(),
                metadata=doc.metadata.copy(),
                content_length=len(text),
                source_type=doc.source_type,
                content_source=doc.content_source,
                relevance_score=doc.relevance_score
            )]
        except ValidationError as ve:
            err = f"[SemanticChunker] Validation error for {doc_id}: {ve}"
            self.logger.error(err, exc_info=True)
            self.state.add_error(err)
            return []

    def _split_and_wrap(self, groups: List[str], doc, doc_id: str) -> List["DocumentResult"]:
        chunks = []
        target = getattr(self.config, "chunk_size_tokens", 400)
        overlap = getattr(self.config, "chunk_overlap_tokens", 50)

        for g in groups:
            cur, cur_tokens, parts = [], 0, []
            for w in g.split():
                cur.append(w)
                cur_tokens = self._count_tokens(" ".join(cur))
                if cur_tokens >= target:
                    parts.append(" ".join(cur))
                    if overlap:
                        keep = max(1, int(overlap * len(cur) / cur_tokens))
                        cur, cur_tokens = cur[-keep:], self._count_tokens(" ".join(cur[-keep:]))
                    else:
                        cur, cur_tokens = [], 0
            if cur:
                parts.append(" ".join(cur))
            for p in parts:
                chunks.extend(self._wrap_chunk(p, doc, doc_id))
        return chunks

    # ------------------------ Fallback ------------------------

    async def _safe_recursive(self, doc, reason: str) -> List["DocumentResult"]:
        doc_id = doc.metadata.get("url") or doc.metadata.get("source") or "unknown"
        try:
            self.logger.info(f"[SemanticChunker] Using recursive fallback for {doc_id}, reason: {reason}")
            return await self._recursive_chunk(doc)
        except Exception as e:
            err = f"[SemanticChunker] Recursive fallback failed for {doc_id}: {e}"
            self.logger.error(err, exc_info=True)
            self.state.add_error(err)
            return []


In [ ]:
import asyncio
import nest_asyncio
from unittest.mock import MagicMock
from langchain_core.documents import Document
import logging
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Any
from langchain_community.embeddings import SentenceTransformerEmbeddings
import sys

# Assume SemanticChunker, CleanerConfig, KnowledgeState, DocumentResult are defined in previous cells

# Configure logging for the test
logging.basicConfig(level=logging.DEBUG, stream=sys.stdout)
test_logger = logging.getLogger(__name__)

nest_asyncio.apply()

# 1. Create a mock KnowledgeState for the SemanticChunker/CleanerEngine to update
mock_state = MagicMock(spec=KnowledgeState)
mock_state.add_error = MagicMock() # Mock the add_error method
# Ensure nested structures if SemanticChunker expects them (though it primarily uses add_error)
mock_state.cleaned_chunks = [] # Simulate the list if needed elsewhere, though SemanticChunker returns chunks

# 2. Create a CleanerConfig instance for semantic chunking
# Note: SemanticChunker uses parameters from CleanerConfig
config = CleanerConfig(
    chunk_size=200, # This might not directly map to semantic chunking depending on implementation
    chunk_overlap=30, # This might not directly map to semantic chunking
    use_semantic_chunking=True, # Ensure this is True, although the SemanticChunker is explicitly called
    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
    # Removed chunk_size_tokens and chunk_overlap_tokens as they are not in CleanerConfig
)
print("Cleaner Config used for SemanticChunker test:")
print(config.model_dump_json(indent=2))

# 3. Initialize a real Embedding Model
try:
    # Initialize the embedding model used by SemanticChunker
    embedding_model = SentenceTransformerEmbeddings(model_name=config.embedding_model_name)
    print("\nInitialized real Embedding Model.")
except Exception as e:
    print(f"\nFailed to initialize real Embedding Model: {e}. Semantic chunking test will likely fail.")
    embedding_model = None

# 4. Create a sample DocumentResult with content for semantic chunking
sample_document = DocumentResult(
    page_content="""Artificial intelligence is transforming industries worldwide. Large language models, like Gemini and GPT, are at the forefront of these advancements. These models excel at understanding and generating human-like text. They have applications in translation, summarization, and creative writing. Recent research focuses on making these models more efficient. Techniques like quantization and distillation are explored. Another key area is ethical AI. Ensuring fairness and reducing bias in LLMs is crucial. The future of AI development involves multimodal models. These can process and generate information across text, images, and audio.""",
    metadata={"url": "https://example.com/ai-overview", "source": "web_crawl", "publication_date": "2025-09-10"},
    source_type=DocumentSource.WEB_CRAWL,
    content_source=ContentSource.BLOG,
    relevance_score=0.9
)
print("\nSample Document for Semantic Chunking:")
print(f"URL: {sample_document.metadata.get('url')}")
print(f"Source Type: {sample_document.source_type}")
print(f"Content Source: {sample_document.content_source}")
print(f"Content Length: {len(sample_document.page_content)}")
print(f"Content Preview: {sample_document.page_content[:200]}...")


# 5. Instantiate the SemanticChunker
# Pass the mock state, the config, the real embedding model, and the test logger
semantic_chunker = SemanticChunker(
    state=mock_state,
    config=config,
    embedding_model=embedding_model,
    logger=test_logger,
    # Assuming sentence_splitter and tokenizer are handled internally or are optional
)
print("\nSemanticChunker instance created.")

# 6. Run the chunk method
print("\nRunning semantic chunking...")
if embedding_model:
    # Wrap in asyncio.run as chunk is an async method
    try:
        chunked_documents = asyncio.run(semantic_chunker.chunk(sample_document))
        print("\nSemantic chunking complete.")

        # 7. Check the results
        print(f"\nGenerated {len(chunked_documents)} chunks:")
        for i, chunk in enumerate(chunked_documents):
            print(f"--- Chunk {i+1} ---")
            print(f"Length: {len(chunk.page_content)}")
            print(f"Source Type: {chunk.source_type}")
            print(f"Content Source: {chunk.content_source}")
            print(f"Original Doc Relevance: {chunk.relevance_score}")
            # Check for semantic chunking specific metadata if added (e.g., scores, boundaries)
            if chunk.metadata:
                print(f"Metadata: {chunk.metadata}")
            print(f"Content: {chunk.page_content[:200]}...") # Print content preview

        print(f"\nadd_error called on mock state: {mock_state.add_error.call_count} times")
        if mock_state.add_error.call_count > 0:
            print("Errors reported:")
            for call in mock_state.add_error.call_args_list:
                print(f"- {call.args[0]}")

    except Exception as e:
        print(f"\nAn error occurred during semantic chunking test: {e}")
        test_logger.error("Semantic chunking test failed.", exc_info=True)

else:
    print("\nSkipping semantic chunking test because embedding model failed to initialize.")

In [ ]:
import asyncio
import logging
from typing import ClassVar, Dict, Any, List, Optional, Protocol, Union
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field, ValidationError # Import ValidationError
import re # Import re for cleaning text
import numpy as np # Import numpy for embedding calculations
# from KnowledgeState import KnowledgeState

# Assume your embedding model is available
# from langchain_community.embeddings import SomeEmbeddingModel

logger = logging.getLogger(__name__)

# -------------------- Async Cleaner Protocol --------------------

class AsyncCleanerProtocol(Protocol):
    """Protocol for an asynchronous document cleaner."""
    async def clean_and_chunk_documents(self, docs: List[DocumentResult]) -> None:
        """Asynchronously clean, chunk, and update state with documents."""
        ...

# -------------------- Enhanced Cleaner Engine --------------------

class CleanerEngine(AsyncCleanerProtocol):
    """
    Enhanced, state-aware, async cleaner for documents.
    Supports intelligent, embedding-based chunking.
    """

    def __init__(self,
                 config: CleanerConfig,
                 embedding_model: Optional[Any] = None):
        self.state = None
        self.config = config
        self.embedding_model = embedding_model
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap
        )
        if self.config.use_semantic_chunking and not self.embedding_model:
            logger.warning("Semantic chunking enabled but no embedding model provided. Falling back to recursive splitting.")
            self.config.use_semantic_chunking = False

    async def clean_and_chunk_documents(self, state: KnowledgeState, docs: List[DocumentResult]) -> KnowledgeState:
        """
        Cleans and chunks documents, then updates the shared KnowledgeState.
        """
        self.state = state # Register current state
        logger.info(f"clean_and_chunk_documents received {len(docs)} documents.")
        if not docs:
            logger.info("No documents to clean and chunk.")
            return self.state

        logger.debug(f"Starting processing for batch of {len(docs)} documents in clean_and_chunk_documents.")
        tasks = []
        for i, doc in enumerate(docs):
             logger.debug(f"clean_and_chunk_documents adding task for Doc {i+1}: URL={doc.metadata.get('url', 'N/A')}, Source={doc.source_type}, Content={doc.page_content[:100]}...")
             tasks.append(self._process_single_doc(doc))


        processed_chunks = await asyncio.gather(*tasks, return_exceptions=True)

        all_cleaned_chunks = []
        for i, result in enumerate(processed_chunks):
            if isinstance(result, Exception):
                # Error was already logged and added to state in _process_single_doc
                logger.error(f"Error processing document {i} in batch (caught in gather): {result}")
            elif result:
                # Result is a list of DocumentResult chunks for a single document
                all_cleaned_chunks.extend(result)

        logger.debug(f"Finished processing batch. Gathered {len(all_cleaned_chunks)} chunks from {len(docs)} documents.")

        if all_cleaned_chunks:
            logger.debug(f"Attempting to extend state.cleaned_chunks with {len(all_cleaned_chunks)} chunks.")
            # Log details of chunks being added to state
            for i, chunk in enumerate(all_cleaned_chunks[:5]): # Log first 5 chunks
                 logger.debug(f"Chunk {i+1}/{len(all_cleaned_chunks)} being added: Type={type(chunk)}, URL={chunk.metadata.get('url', 'N/A')}, Source={chunk.source_type}, Content={chunk.page_content[:100]}...")
                 if not isinstance(chunk, DocumentResult):
                      logger.error(f"Invalid chunk type found: {type(chunk)}. Expected DocumentResult.") # Explicit check
                      self.state.add_error(f"Invalid chunk type found: {type(chunk)}. Expected DocumentResult.") # Add error for invalid type
            try:
                self.state.cleaned_chunks.extend(all_cleaned_chunks)
                logger.info(f"Added {len(all_cleaned_chunks)} total chunks to state. Total cleaned_chunks now: {len(self.state.cleaned_chunks)}")
            except Exception as e:
                 error_msg = f"Error extending state.cleaned_chunks: {e}"
                 logger.error(error_msg, exc_info=True)
                 self.state.add_error(error_msg)

        else:
            logger.info("No chunks generated after processing documents.")

        return self.state # return updated state

    async def _process_single_doc(self, doc: DocumentResult) -> Optional[List[DocumentResult]]:
        """
        Cleans and chunks a single document.
        """
        doc_identifier = doc.metadata.get('url', doc.metadata.get('source', 'unknown'))
        logger.info(f"Processing single document from {doc_identifier}...")
        logger.debug(f"Document content preview: {doc.page_content[:200]}...")
        logger.debug(f"Document metadata: {doc.metadata}")
        logger.debug(f"Document source type: {doc.source_type}")
        logger.debug(f"Document content source: {doc.content_source}")
        logger.debug(f"Document relevance score: {doc.relevance_score}")


        try:
            if not isinstance(doc, DocumentResult):
                 error_msg = f"Received invalid document type: {type(doc)}. Expected DocumentResult. Skipping."
                 logger.error(error_msg)
                 self.state.add_error(error_msg)
                 return None # Skip processing this invalid document

            if not doc or not doc.page_content or not doc.page_content.strip():
                logger.info(f"Skipping empty or whitespace document from {doc_identifier}")
                return None

            cleaned_doc = await self._strip_html(doc)
            if not cleaned_doc or not cleaned_doc.page_content or not cleaned_doc.page_content.strip():
                logger.info(f"Skipping document after HTML stripping from {doc_identifier}")
                return None

            chunks = []
            if self.config.use_semantic_chunking:
                logger.info(f"Semantic chunking document from {doc_identifier}...")
                # Add error handling around semantic chunking
                try:
                    chunks = await self._semantic_chunk(cleaned_doc)
                except Exception as e:
                    error_msg = f"Semantic chunking failed for document from {doc_identifier}: {e}"
                    logger.error(error_msg, exc_info=True)
                    self.state.add_error(error_msg)
                    # Fallback to recursive chunking on semantic chunking failure
                    logger.warning(f"Falling back to recursive chunking for {doc_identifier}")
                    try:
                        chunks = await self._recursive_chunk(cleaned_doc)
                    except Exception as re:
                         error_msg = f"Recursive chunking fallback failed for document from {doc_identifier}: {re}"
                         logger.error(error_msg, exc_info=True)
                         self.state.add_error(error_msg)
                         return None # Return None if recursive fallback also fails


            else:
                logger.info(f"Recursive chunking document from {doc_identifier}...")
                # Add error handling around recursive chunking
                try:
                    chunks = await self._recursive_chunk(cleaned_doc)
                except Exception as e:
                    error_msg = f"Recursive chunking failed for document from {doc_identifier}: {e}"
                    logger.error(error_msg, exc_info=True)
                    self.state.add_error(error_msg)
                    return None # Return None if recursive chunking fails

            # Embedding-based relevance scoring is moved to CrawlerEngine or a separate node
            # LLM-based scoring is also handled elsewhere if needed.
            # Return the generated chunks
            logger.debug(f"Finished processing single doc {doc_identifier}. Generated {len(chunks)} chunks.")
            return chunks


        except Exception as e:
            # Catch any other unexpected errors during single doc processing
            error_msg = f"Unexpected error processing document from {doc_identifier}: {e}"
            logger.error(error_msg, exc_info=True)
            self.state.add_error(error_msg)
            return None # Return None to indicate failure for this document

    async def _strip_html(self, doc: DocumentResult) -> DocumentResult:
        """
        Removes HTML tags from a document's content using a separate thread.
        Includes error handling.
        """
        doc_identifier = doc.metadata.get('url', doc.metadata.get('source', 'unknown'))
        logger.debug(f"Attempting to strip HTML from {doc_identifier}")
        try:
            return await asyncio.to_thread(self._strip_html_sync, doc)
        except Exception as e:
             error_msg = f"Failed to strip HTML from document {doc_identifier}: {e}"
             logger.error(error_msg, exc_info=True)
             self.state.add_error(error_msg)
             # Return an empty document on failure to prevent further processing issues
             # Ensure the returned object is a DocumentResult, even if empty
             return DocumentResult(page_content="", metadata=doc.metadata, source_type=doc.source_type, content_source=doc.content_source, relevance_score=doc.relevance_score)


    def _strip_html_sync(self, doc: DocumentResult) -> DocumentResult:
        """Synchronous HTML stripping with BeautifulSoup."""
        doc_identifier = doc.metadata.get('url', doc.metadata.get('source', 'unknown'))
        logger.debug(f"Starting HTML stripping for {doc_identifier}")
        try:
            if not doc.page_content:
                logger.debug(f"No content to strip HTML from for {doc_identifier}")
                return DocumentResult(page_content="", metadata=doc.metadata, source_type=doc.source_type, content_source=doc.content_source, relevance_score=doc.relevance_score) # Preserve source_type and other fields

            # Add additional cleaning logic here (e.g., removing extra whitespace)
            # Use lxml parser for robustness
            soup = BeautifulSoup(doc.page_content, "lxml")
            text = soup.get_text(" ", strip=True)
            cleaned_text = re.sub(r'\s+', ' ', text).strip()
            logger.debug(f"Successfully stripped HTML from {doc_identifier}. Cleaned content preview: {cleaned_text[:200]}...")
            # Ensure the returned object is a DocumentResult
            return DocumentResult(page_content=cleaned_text, metadata=doc.metadata, source_type=doc.source_type, content_source=doc.content_source, relevance_score=doc.relevance_score) # Preserve source_type and other fields

        except Exception as e:
            # This exception is caught by the async wrapper _strip_html
            logger.error(f"Error during synchronous HTML stripping for {doc_identifier}: {e}", exc_info=True)
            raise e # Re-raise to be caught by the async wrapper


    async def _recursive_chunk(self, doc: DocumentResult) -> List[DocumentResult]:
        """
        Chunks a document using the RecursiveCharacterTextSplitter and converts to DocumentResult.
        Includes error handling.
        """
        doc_identifier = doc.metadata.get('url', doc.metadata.get('source', 'unknown'))
        logger.debug(f"Starting recursive chunking for {doc_identifier}")
        try:
            # The splitting process can be CPU-bound, so run in a thread
            # Ensure the input to split_documents is a list containing a LangChain Document
            if not isinstance(doc, DocumentResult):
                 logger.error(f"Invalid document type for recursive chunking: {type(doc)}. Expected DocumentResult. Skipping.")
                 self.state.add_error(f"Invalid document type for recursive chunking: {type(doc)}. Expected DocumentResult.")
                 return [] # Return empty list for invalid input


            # Convert DocumentResult to LangChain Document for the splitter
            lc_doc = Document(page_content=doc.page_content, metadata=doc.metadata)
            chunks = await asyncio.to_thread(self.splitter.split_documents, [lc_doc])
            logger.debug(f"Finished recursive chunking for {doc_identifier}, created {len(chunks)} chunks")

            # Convert LangChain Document chunks back to DocumentResult, preserving metadata and source type
            chunk_results = []
            for chunk in chunks:
                 try:
                      # Ensure the chunk is a LangChain Document before accessing attributes
                      if not isinstance(chunk, Document):
                           logger.warning(f"Skipping non-Document chunk from splitter: {type(chunk)}. Chunk preview: {str(chunk)[:100]}...")
                           continue # Skip invalid chunk

                      chunk_results.append(DocumentResult(
                         page_content=chunk.page_content,
                         metadata=chunk.metadata,
                         source_type=doc.source_type, # Inherit source type from original doc
                         content_source=doc.content_source, # Inherit content source
                         relevance_score=doc.relevance_score # Inherit relevance score
                     ))
                 except ValidationError as e:
                      error_msg = f"Validation error converting recursive chunk to DocumentResult for {doc_identifier}: {e}. Chunk preview: {chunk.page_content[:100]}..."
                      logger.error(error_msg, exc_info=True)
                      self.state.add_error(error_msg)
                      # Continue to the next chunk if validation fails for one
                 except Exception as e:
                      error_msg = f"Unexpected error converting recursive chunk to DocumentResult for {doc_identifier}: {e}. Chunk preview: {chunk.page_content[:100]}..."
                      logger.error(error_msg, exc_info=True)
                      self.state.add_error(error_msg)
                      # Continue to the next chunk on unexpected error

            logger.debug(f"Successfully converted {len(chunk_results)} recursive chunks to DocumentResult for {doc_identifier}.")
            return chunk_results
        except Exception as e:
            # This exception is caught by the caller (_process_single_doc)
            logger.error(f"Error during recursive chunking for {doc_identifier}: {e}", exc_info=True)
            raise e # Re-raise to be caught by the caller


    async def _semantic_chunk(self, doc: DocumentResult) -> List[DocumentResult]:
        """
        Intelligent, semantic-based chunking using embeddings and converts to DocumentResult.
        Requires an embedding model to be configured.
        Includes error handling and fallback.
        """
        doc_identifier = doc.metadata.get('url', doc.metadata.get('source', 'unknown'))
        logger.debug(f"Starting semantic chunking for {doc_identifier}")

        if not self.embedding_model:
            logger.warning(f"Embedding model not set for semantic chunking for {doc_identifier}. Falling back to recursive.")
            # Fallback to recursive chunking if embedding model is not available
            try:
                return await self._recursive_chunk(doc)
            except Exception as e:
                 error_msg = f"Recursive chunking fallback failed during semantic chunking for {doc_identifier}: {e}"
                 logger.error(error_msg, exc_info=True)
                 self.state.add_error(error_msg)
                 return [] # Return empty list on failure


        try:
            sentences = doc.page_content.split('. ')
            if len(sentences) <= 1:
                logger.debug(f"Not enough sentences for semantic chunking for {doc_identifier}, falling back to recursive.")
                # Fallback if not enough sentences
                try:
                    return await self._recursive_chunk(doc)
                except Exception as e:
                     error_msg = f"Recursive chunking fallback failed during semantic chunking for {doc_identifier}: {e}"
                     logger.error(error_msg, exc_info=True)
                     self.state.add_error(error_msg)
                     return [] # Return empty list on failure


            # This part requires an actual embedding model and clustering logic
            # Example using placeholder logic:
            chunks = []
            current_chunk_sentences = []
            current_length = 0
            # Convert sentences into Document objects for embedding
            sentence_docs = [Document(page_content=s.strip() + '.') for s in sentences if s.strip()]

            if not sentence_docs:
                 logger.debug(f"No valid sentences after splitting for {doc_identifier}.")
                 return [] # Return empty if no valid sentences


            # Embed the sentences
            # Add error handling around embedding calls
            try:
                sentence_embeddings = await asyncio.to_thread(self.embedding_model.embed_documents, [d.page_content for d in sentence_docs])
            except Exception as e:
                error_msg = f"Embedding sentences failed for document from {doc_identifier}: {e}"
                logger.error(error_msg, exc_info=True)
                self.state.add_error(error_msg)
                # Fallback to recursive chunking if embedding fails
                logger.warning(f"Embedding failed, falling back to recursive chunking for {doc_identifier}")
                try:
                     return await self._recursive_chunk(doc)
                except Exception as re:
                      error_msg = f"Recursive chunking fallback after embedding failed for {doc_identifier}: {re}"
                      logger.error(error_msg, exc_info=True)
                      self.state.add_error(error_msg)
                      return [] # Return empty list if fallback fails

            # Simplified semantic chunking logic: combine sentences until chunk_size is reached
            # In a real semantic chunker, you would analyze sentence similarity to find boundaries.
            current_chunk_docs = []
            current_length = 0
            for i, sentence_doc in enumerate(sentence_docs):
                 sentence_length = len(sentence_doc.page_content) + (1 if current_chunk_docs else 0) # Add space if not first
                 if current_length + sentence_length > self.config.chunk_size and current_chunk_docs:
                      chunk_content = ' '.join([d.page_content for d in current_chunk_docs])
                      # Convert to DocumentResult, preserving metadata and source type
                      try:
                           chunks.append(DocumentResult(page_content=chunk_content, metadata=doc.metadata.copy(), source_type=doc.source_type, content_source=doc.content_source, relevance_score=doc.relevance_score))
                      except ValidationError as e:
                           error_msg = f"Validation error converting semantic chunk to DocumentResult for {doc_identifier}: {e}. Chunk preview: {chunk_content[:100]}..."
                           logger.error(error_msg, exc_info=True)
                           self.state.add_error(error_msg)
                           # Skip this chunk if validation fails
                      except Exception as e:
                           error_msg = f"Unexpected error converting semantic chunk to DocumentResult for {doc_identifier}: {e}. Chunk preview: {chunk_content[:100]}..."
                           logger.error(error_msg, exc_info=True)
                           self.state.add_error(error_msg)
                           # Skip this chunk on unexpected error


                      current_chunk_docs = [sentence_doc]
                      current_length = len(sentence_doc.page_content)
                 else:
                      current_chunk_docs.append(sentence_doc)
                      current_length += sentence_length

            if current_chunk_docs:
                 chunk_content = ' '.join([d.page_content for d in current_chunk_docs])
                 # Convert last chunk to DocumentResult
                 try:
                      chunks.append(DocumentResult(page_content=chunk_content, metadata=doc.metadata.copy(), source_type=doc.source_type, content_source=doc.content_source, relevance_score=doc.relevance_score))
                 except ValidationError as e:
                      error_msg = f"Validation error converting last semantic chunk to DocumentResult for {doc_identifier}: {e}. Chunk preview: {chunk_content[:100]}..."
                      logger.error(error_msg, exc_info=True)
                      self.state.add_error(error_msg)
                      # Skip this chunk if validation fails
                 except Exception as e:
                      error_msg = f"Unexpected error converting last semantic chunk to DocumentResult for {doc_identifier}: {e}. Chunk preview: {chunk_content[:100]}..."
                      logger.error(error_msg, exc_info=True)
                      self.state.add_error(error_msg)
                      # Skip this chunk on unexpected error


            logger.debug(f"Finished semantic chunking for {doc_identifier}, created {len(chunks)} chunks")
            return chunks

        except Exception as e:
            # Catch errors during semantic chunking logic (after initial checks)
            error_msg = f"Semantic chunking failed for {doc_identifier}: {e}"
            logger.error(error_msg, exc_info=True)
            self.state.add_error(error_msg)
            # Fallback to recursive chunking on semantic chunking failure
            logger.warning(f"Semantic chunking failed, falling back to recursive chunking for {doc_identifier}")
            try:
                return await self._recursive_chunk(doc)
            except Exception as re:
                 error_msg = f"Recursive chunking fallback failed during semantic chunking for {doc_identifier}: {re}"
                 logger.error(error_msg, exc_info=True)
                 self.state.add_error(error_msg)
                 return [] # Return empty list if recursive fallback also fails


    async def _score_content_embedding(self, chunks: List[DocumentResult], query: str) -> List[DocumentResult]:
        """
        Use embeddings to score content chunks against the query based on semantic similarity.
        Returns the chunks with an added 'embedding_relevance_score' in their metadata.
        Includes error handling.
        """
        # This method should not be called from CleanerEngine based on current design
        # It's kept here for potential future use or if other parts of the code call it directly.
        # The logic below still contains the problematic attribute access.
        # However, the _process_single_doc method no longer calls this based on the config flag.
        # The error must be elsewhere if it's still related to this.

        if not self.embedding_model or not chunks:
            logger.debug("Skipping embedding scoring: no model or no chunks.")
            return chunks

        logger.debug(f"Starting embedding scoring for {len(chunks)} chunks.")

        try:
            # Embed the query
            query_embedding = await asyncio.to_thread(self.embedding_model.embed_query, query)
            logger.debug("Query embedded successfully.")

            # Embed the documents/chunks
            chunk_texts = [chunk.page_content for chunk in chunks]
            # Add error handling around embedding calls
            try:
                chunk_embeddings = await asyncio.to_thread(self.embedding_model.embed_documents, chunk_texts)
                logger.debug(f"Embedded {len(chunk_embeddings)} chunks successfully.")
            except Exception as e:
                 error_msg = f"Embedding chunks failed: {e}"
                 logger.error(error_msg, exc_info=True)
                 self.state.add_error(error_msg)
                 # Return original chunks with default low score if embedding fails
                 for chunk in chunks:
                      chunk.metadata["embedding_relevance_score"] = 0.0 # Assign a low score on failure
                 return chunks


            # Calculate cosine similarity
            scores = []
            if len(query_embedding) != len(chunk_embeddings[0]):
                 logger.warning(f"Query embedding dimension ({len(query_embedding)}) does not match chunk embedding dimension ({len(chunk_embeddings[0])}). Skipping scoring.")
                 # Return original chunks with default low score if dimensions don't match
                 for chunk in chunks:
                      chunk.metadata["embedding_relevance_score"] = 0.0 # Assign a low score on failure
                 return chunks


            for chunk_embedding in chunk_embeddings:
                try:
                    # Ensure embeddings are lists/tuples of numbers before zipping
                    if not isinstance(chunk_embedding, (list, tuple)) or not all(isinstance(x, (int, float)) for x in chunk_embedding):
                         logger.warning(f"Chunk embedding has unexpected format. Skipping scoring for this chunk. Embedding preview: {str(chunk_embedding)[:100]}...")
                         scores.append(0.0) # Assign low score for malformed embedding
                         continue

                    score = sum(q * c for q, c in zip(query_embedding, chunk_embedding))
                    normalized_score = (score + 1) / 2 # Map [-1, 1] to [0, 1]
                    scores.append(normalized_score)
                except Exception as e:
                     logger.warning(f"Error calculating score for a chunk embedding: {e}. Assigning 0.0.", exc_info=True)
                     scores.append(0.0) # Assign low score on calculation error


            # Add scores to chunk metadata
            scored_chunks = []
            for i, chunk in enumerate(chunks):
                try:
                     if i < len(scores): # Ensure index is within bounds
                         chunk.metadata["embedding_relevance_score"] = scores[i]
                     else:
                         logger.warning(f"Score index out of bounds for chunk {i}. Assigning 0.0.")
                         chunk.metadata["embedding_relevance_score"] = 0.0 # Assign low score if index is off
                     scored_chunks.append(chunk) # Add the chunk (scored or not)

                except Exception as e:
                    # Catch errors while adding score to metadata
                    error_msg = f"Error adding embedding score to chunk metadata for chunk {i}: {e}. Chunk preview: {chunk.page_content[:100]}..."
                    logger.error(error_msg, exc_info=True)
                    self.state.add_error(error_msg)
                    # Decide whether to discard the chunk or keep it with default score
                    # Let's keep it with a default low score
                    chunk.metadata["embedding_relevance_score"] = 0.0
                    scored_chunks.append(chunk)


            logger.debug(f"Finished embedding scoring for {len(scored_chunks)} chunks.")
            return scored_chunks

        except Exception as e:
            error_msg = f"Embedding scoring failed overall: {e}"
            logger.error(error_msg, exc_info=True)
            self.state.add_error(error_msg)
            # If embedding fails, return original chunks with a default low score
            for chunk in chunks:
                 chunk.metadata["embedding_relevance_score"] = 0.0 # Assign a low score on failure
            return chunks


    async def _score_content_llm(self, chunks: List[Document], query: str) -> List[DocumentResult]:
        """
        Use an LLM to score content chunks against the query.
        Returns the chunks with an added 'llm_relevance_score' in their metadata.
        (Kept for reference if needed, but embedding is preferred as per user feedback)
        Includes error handling.
        """
        if not self.llm or not chunks:
            return chunks

        logger.debug(f"Starting LLM scoring for {len(chunks)} chunks.")

        scoring_prompt = """
        You are a quality assurance assistant for a web crawler. Your task is to score document chunks based on their relevance to a given query on a scale of 0.0 to 1.0.
        Consider both topical relevance and the content source. Prioritize academic and government sources.
        Provide a single relevance score from 0.0 to 1.0 for each chunk in the exact format specified.

        Query: {query}

        Example Output Format:
        [
          {{ "chunk_index": 0, "score": 0.85 }},
          {{ "chunk_index": 1, "score": 0.20 }},
          ...
        ]

        Now, score the following chunks:
        """

        batch_size = 5
        scored_chunks = []
        for i in range(0, len(chunks), batch_size):
            chunk_batch = chunks[i:i+batch_size]
            prompt = scoring_prompt.format(query=query)
            for j, chunk in enumerate(chunk_batch):
                # Include date and source in the prompt for LLM context
                date_info = f"Publication Date: {chunk.metadata.get('publication_date', 'N/A')}\nModification Date: {chunk.metadata.get('modification_date', 'N/A')}\n"
                source_info = f"Source Type: {chunk.source_type.value if isinstance(chunk.source_type, Enum) else chunk.source_type}\nContent Source: {chunk.content_source.value if isinstance(chunk.content_source, Enum) else chunk.content_source}\n"
                prompt += f"\n--- Chunk {i+j} ---\n{date_info}{source_info}{chunk.page_content[:500]}...\n"

            try:
                response = await _call_llm(self.llm, prompt) # Use helper for retry
                scores = self._parse_llm_response(response)

                for score_info in scores:
                    chunk_index = score_info.get("chunk_index")
                    score = score_info.get("score")
                    # Find the original chunk by index from the full list of chunks
                    if chunk_index is not None and score is not None and 0 <= chunk_index < len(chunks):
                         # Clamp LLM score to [0, 1] and add to metadata
                         chunks[chunk_index].metadata["llm_relevance_score"] = max(0.0, min(1.0, score))
                # Add the processed batch (with potential scores) to the scored_chunks list
                scored_chunks.extend(chunk_batch)

            except Exception as e:
                error_msg = f"LLM scoring failed for batch starting at index {i}: {e}"
                logger.error(error_msg, exc_info=True)
                self.state.add_error(error_msg)
                # Assign a low score or handle failure for the batch
                for chunk in chunk_batch:
                     chunk.metadata["llm_relevance_score"] = 0.0 # Assign a low score on failure
                scored_chunks.extend(chunk_batch) # Still include chunks even if scoring failed


        logger.debug(f"Finished LLM scoring for {len(scored_chunks)} chunks.")
        return scored_chunks # Return the list of chunks with scores added to metadata


    def _parse_llm_response(self, text: str) -> List[Dict[str, Union[int, float]]]:
        """Parses the LLM's response to extract scores. Includes error handling."""
        try:
            json_match = re.search(r'\[\s*\{.*?\}\s*\]', text, re.DOTALL)
            if json_match:
                import json
                # Use json.loads with error handling
                try:
                    return json.loads(json_match.group(0))
                except json.JSONDecodeError as e:
                     logger.warning(f"Failed to decode JSON from LLM response: {e}. Response snippet: {text[:200]}...")
                     return [] # Return empty list on JSON decode error
            else:
                 logger.warning(f"No JSON list found in LLM response. Response snippet: {text[:200]}...")
                 return [] # Return empty list if no JSON list pattern found
        except Exception as e:
            # Catch any other unexpected errors during parsing
            logger.error(f"Unexpected error parsing LLM response: {e}", exc_info=True)
            # Decide whether to add this as a state error or just log
            # self.state.add_error(f"Failed to parse LLM response: {str(e)}")
            return [] # Return empty list on unexpected parsing error

Here is a test script to demonstrate the functionality of the `CleanerEngine` class. It includes several edge cases.

In [ ]:
import asyncio
import nest_asyncio
from langchain_core.documents import Document
import logging
from datetime import datetime
from typing import List
from langchain_community.embeddings import SentenceTransformerEmbeddings

nest_asyncio.apply()

# Assume KnowledgeState, CleanerConfig, CleanerEngine, DocumentResult, DocumentSource, ContentSource are defined above

# 1. Create a KnowledgeState for the CleanerEngine to update
mock_state = KnowledgeState.create_knowledge_state(main_query="Latest advancements in AI")

# 2. Create two CleanerConfig instances (recursive vs semantic chunking)
config_recursive = CleanerConfig(
    chunk_size=200,
    chunk_overlap=20,
    use_semantic_chunking=False
)
config_semantic = CleanerConfig(
    chunk_size=200,
    chunk_overlap=30,
    use_semantic_chunking=True
)

print("Cleaner Config (Recursive):")
print(config_recursive.model_dump_json(indent=2))
print("\nCleaner Config (Semantic):")
print(config_semantic.model_dump_json(indent=2))

# 3. Try to initialize a real embedding model (for semantic chunking)
try:
    real_embedding_model = SentenceTransformerEmbeddings(
        model_name=config_semantic.embedding_model_name
    )
    print("\nInitialized real embedding model for semantic chunking.")
except Exception as e:
    print(f"\nFailed to initialize embedding model: {e}. Falling back to recursive.")
    real_embedding_model = None
    config_semantic.use_semantic_chunking = False

# 4. Create CleanerEngine instances
cleaner_recursive = CleanerEngine(config=config_recursive)
cleaner_semantic = CleanerEngine(
    config=config_semantic, embedding_model=real_embedding_model
)

print("\nCleaner Engines created.")

# 5. Prepare a batch of simulated documents with realistic content
documents_to_clean: List[DocumentResult] = [
    # A news article (web crawl)
    DocumentResult(
        page_content="""
        <html><body>
        <h1>OpenAI Introduces GPT-5</h1>
        <p>OpenAI has announced GPT-5, the successor to GPT-4, with major improvements in reasoning and efficiency.</p>
        <p>The model is expected to power new applications in research and enterprise.</p>
        </body></html>
        """,
        metadata={
            "url": "https://www.theverge.com/2025/09/01/openai-gpt5-release",
            "source": "web_crawl",
            "publication_date": "2025-09-01",
        },
        source_type=DocumentSource.WEB_CRAWL,
        content_source=ContentSource.NEWS,
    ),
    # A research summary (search snippet)
    DocumentResult(
        page_content="Snippet: Google DeepMind releases Gemini 2.0 with multimodal reasoning across text, images, and code.",
        metadata={
            "url": "https://www.deepmind.com/blog/gemini-2-announcement",
            "source": "search_snippet",
            "snippet": "DeepMind releases Gemini 2.0...",
            "title": "DeepMind Gemini 2.0",
        },
        source_type=DocumentSource.SEARCH_SNIPPET,
    ),
    # A long article (forces recursive chunking)
    DocumentResult(
        page_content=("This is a detailed analysis of AI policy and governance. " * 20),
        metadata={
            "url": "https://www.brookings.edu/research/ai-policy-2025/",
            "source": "web_crawl",
            "publication_date": "2025-08-10",
        },
        source_type=DocumentSource.WEB_CRAWL,
        content_source=ContentSource.REPORT,
    ),
    # A semantic chunking test doc
    DocumentResult(
        page_content="NVIDIA unveiled new GPUs optimized for AI workloads. Microsoft announced new Copilot integrations. These updates are crucial for enterprise AI adoption.",
        metadata={
            "url": "https://www.microsoft.com/blog/copilot-expansion",
            "source": "web_crawl",
            "publication_date": "2025-08-25",
        },
        source_type=DocumentSource.WEB_CRAWL,
        content_source=ContentSource.BLOG,
    ),
    # Empty page
    DocumentResult(
        page_content="",
        metadata={"url": "https://www.example.com/empty", "source": "web_crawl"},
        source_type=DocumentSource.WEB_CRAWL,
    ),
    # Whitespace-only page
    DocumentResult(
        page_content="   \n\n   ",
        metadata={"url": "https://www.example.com/whitespace", "source": "web_crawl"},
        source_type=DocumentSource.WEB_CRAWL,
    ),
    # Simple text
    DocumentResult(
        page_content="AI is transforming industries from healthcare to finance.",
        metadata={"url": "https://www.ibm.com/blog/ai-industries", "source": "web_crawl"},
        source_type=DocumentSource.WEB_CRAWL,
    ),
    # Complex HTML
    DocumentResult(
        page_content="""
        <html><body>
        <script>alert('x');</script>
        <div>AI trends in 2025</div>
        <style>.hidden {display: none}</style>
        <p>Key breakthroughs include multimodal reasoning and agent-based systems.</p>
        </body></html>
        """,
        metadata={"url": "https://www.technologyreview.com/ai-trends-2025", "source": "web_crawl"},
        source_type=DocumentSource.WEB_CRAWL,
    ),
    # Knowledge base doc
    DocumentResult(
        page_content="Historical context: The first neural networks were proposed in the 1940s.",
        metadata={"source": "knowledge_base", "kb_id": "kb001"},
        source_type=DocumentSource.KNOWLEDGE_BASE,
    ),
]

print(f"\nPrepared {len(documents_to_clean)} documents for cleaning.")

# 6. Run recursive cleaner
print("\n--- Testing Recursive Chunking ---")
mock_state.cleaned_chunks = []
asyncio.run(cleaner_recursive.clean_and_chunk_documents(mock_state, documents_to_clean))
print("Recursive chunking complete.")
print(f"Chunks created: {len(mock_state.cleaned_chunks)}")
if mock_state.cleaned_chunks:
    print("\nSample chunks (Recursive):")
    for c in mock_state.cleaned_chunks[:3]:
        print(f"- [{len(c.page_content)} chars] {c.metadata.get('url','N/A')}")

# 7. Run semantic cleaner
print("\n--- Testing Semantic Chunking ---")
mock_state.cleaned_chunks = []
asyncio.run(cleaner_semantic.clean_and_chunk_documents(mock_state, documents_to_clean))
print("Semantic chunking complete.")
print(f"Chunks created: {len(mock_state.cleaned_chunks)}")
if mock_state.cleaned_chunks:
    print("\nSample chunks (Semantic):")
    for c in mock_state.cleaned_chunks[:3]:
        score = c.metadata.get("embedding_relevance_score")
        score_str = f", Score={score:.2f}" if score else ""
        print(f"- [{len(c.page_content)} chars] {c.metadata.get('url','N/A')}{score_str}")


In [ ]:
!pip install chromadb --quiet
!pip install langchain-core --quiet
!pip install --upgrade langchain --quiet

import os
import logging
import asyncio
import time
import hashlib # Import hashlib for generating IDs
from typing import Dict, Any, Optional, List

from pydantic import BaseModel, Field, PrivateAttr, ValidationError
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import RedisStore, InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings, SentenceTransformerEmbeddings
from tenacity import retry, stop_after_attempt, wait_exponential, RetryError
from langchain_core.documents import Document # Import Document

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AsyncVectorStoreProtocol(Protocol):
    """Protocol for an asynchronous vector store node."""
    async def index_documents(self, state: KnowledgeState) -> Dict[str, Any]:
        """Indexes cleaned chunks from state and updates vector store."""
        ...

# -------------------------------------------------------------------
# --- VectorStoreNode (Chroma only)
# -------------------------------------------------------------------
class VectorStoreNode(AsyncVectorStoreProtocol):
    _vectorstore: Any = PrivateAttr()
    _embeddings: Any = PrivateAttr()

    def __init__(self, config: VectorStoreConfig, embedding_model: Optional[Any] = None): # Added embedding_model parameter
        self.config = config
        # Store the provided embedding model if available, otherwise initialize based on config
        self._embeddings = embedding_model if embedding_model else self._get_embedding_model()

        try:
            self._initialize_components()
        except (ValueError, ValidationError) as e:
            logger.critical(f"Failed to initialize VectorStoreNode: {e}")
            raise

    # ----------------- Helpers -----------------
    def _get_embedding_model(self):
        if self.config.embedding_model_provider == "huggingface":
            return HuggingFaceEmbeddings(model_name=self.config.embedding_model_name)
        elif self.config.embedding_model_provider == "sentence-transformer":
            return SentenceTransformerEmbeddings(model_name=self.config.embedding_model_name)
        else:
            raise ValueError(f"Unknown embedding provider: {self.config.embedding_model_provider}")

    def _initialize_components(self):
        # Ensure persistence dir exists
        os.makedirs(self.config.chroma_persist_directory, exist_ok=True)

        # If embedding_model was not provided, initialize it now
        if not self._embeddings:
             self._embeddings = self._get_embedding_model()

        # Wrap the base embedder with caching
        # Assuming the base embedder is already set in self._embeddings
        cache_store = InMemoryStore() # Or RedisStore for persistence
        self._embeddings = CacheBackedEmbeddings.from_bytes_store(
            self._embeddings, # Use the initialized embedding model
            document_embedding_cache=cache_store,
            namespace=self.config.collection_name
        )

        # Chroma setup
        self._vectorstore = Chroma(
            collection_name=self.config.collection_name,
            embedding_function=self._embeddings,
            persist_directory=self.config.chroma_persist_directory,
        )
        logger.info(f"Chroma initialized at {self.config.chroma_persist_directory}")

    # ----------------- Indexing -----------------
    @retry(wait=wait_exponential(multiplier=1, min=2, max=10), stop=stop_after_attempt(5), reraise=True)
    async def _index_with_retry(self, documents: List[DocumentResult]):
        start = time.perf_counter()
        # Generate IDs for each document using a hash of the content
        document_ids = [hashlib.sha256(doc.page_content.encode()).hexdigest() for doc in documents]

        # Use asyncio.to_thread for the synchronous add_documents call
        await asyncio.to_thread(self._vectorstore.add_documents, documents, ids=document_ids)

        # Add the generated IDs to the metadata of the DocumentResult objects
        for doc, doc_id in zip(documents, document_ids):
            doc.metadata["vector_id"] = doc_id

        end = time.perf_counter()
        logger.info(f"Indexed {len(documents)} docs in {end - start:.2f}s")

    async def index_documents(self, state: KnowledgeState) -> Dict[str, Any]:
        if not state.cleaned_chunks:
            logger.info("No docs to index.")
            return {"cleaned_chunks": [], "errors": state.errors} # Return current errors

        # Convert to LC Documents if necessary, but DocumentResult is already compatible
        # Ensure they are DocumentResult objects as expected by the caller
        docs_to_index = []
        for d in state.cleaned_chunks:
            if isinstance(d, DocumentResult):
                docs_to_index.append(d)
            # Add handling for standard LangChain Document if they might appear here
            elif isinstance(d, Document):
                 docs_to_index.append(DocumentResult(page_content=d.page_content, metadata=d.metadata))
            elif isinstance(d, dict):
                # Attempt to parse dict into DocumentResult
                try:
                    docs_to_index.append(DocumentResult(**d))
                except ValidationError as e:
                    msg = f"Failed to validate dict as DocumentResult: {e}"
                    state.add_error(msg)
                    logger.error(msg)
            else:
                msg = f"Unsupported type in cleaned_chunks: {type(d)}. Expected DocumentResult or dict."
                state.add_error(msg)
                logger.error(msg)


        if not docs_to_index:
             logger.warning("No valid documents to index after type checking.")
             return {"cleaned_chunks": [], "errors": state.errors}


        try:
            await self._index_with_retry(docs_to_index)
            # Clear cleaned_chunks from state after successful indexing
            state.cleaned_chunks = [] # This clears the list after successful indexing

            # If you need to keep a record of indexed documents with their new vector_id,
            # you would need a separate list in the state for "indexed_documents" or similar,
            # and append the docs_to_index (which now have vector_id in metadata) to that list
            # BEFORE clearing state.cleaned_chunks.
            # For now, we stick to the current state structure and clear cleaned_chunks.

            return {"cleaned_chunks": [], "errors": state.errors} # Indicate chunks are cleared

        except RetryError as e:
            msg = f"Indexing failed after retries: {e}"
            state.add_error(msg)
            logger.error(msg)
            return {"errors": state.errors}
        except Exception as e:
            msg = f"Indexing error: {e}"
            state.add_error(msg, exc_info=True) # Include exc_info for detailed traceback
            logger.error(msg, exc_info=True)
            return {"errors": state.errors}

    # ----------------- Search -----------------
    async def search(self, query: str, k: int = 3):
        # Ensure search also works with the CacheBackedEmbeddings
        # The return type is List[Document] from Chroma's similarity_search
        return await asyncio.to_thread(self._vectorstore.similarity_search, query, k)

In [ ]:
import asyncio
import logging

# Import from your module
# from vector_store_node import VectorStoreNode, VectorStoreConfig, KnowledgeState

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


async def test_chroma_with_cache():
    logger.info("=" * 50)
    logger.info("Starting test scenario: ChromaDB with InMemory Cache")
    logger.info("=" * 50)

    # --- Config ---
    config = VectorStoreConfig(
        collection_name="test_collection",
        chroma_persist_directory="./chroma_db_test",
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
        embedding_model_provider="huggingface"
    )

    # --- Init Node ---
    try:
        node = VectorStoreNode(config)
        logger.info("VectorStoreNode initialized with Chroma.")
    except Exception as e:
        logger.error(f"Failed to initialize Chroma node: {e}")
        return

    # --- Prepare State ---
    # Create raw Document objects
    raw_docs = [
        DocumentResult(page_content="Chroma is a vector database for embeddings.", metadata={"source": "test1"}),
        DocumentResult(page_content="It supports persistence and fast similarity search.", metadata={"source": "test2"}),
    ]

    # Convert Document objects to DocumentResult objects
    cleaned_chunks_results = [
        DocumentResult(
            page_content=doc.page_content,
            metadata=doc.metadata,
            source_type=DocumentSource.KNOWLEDGE_BASE, # Or another appropriate source type
            relevance_score=1.0 # Example score
        )
        for doc in raw_docs
    ]


    state = KnowledgeState(
        main_query="Test query for indexing", # Add main_query
        messages=[], # Add messages
        cleaned_chunks=cleaned_chunks_results # Use DocumentResult objects here
    )

    # --- Index Docs ---
    result = await node.index_documents(state)
    logger.info(f"Indexing result: {result}")

    # --- Search ---
    try:
        query = "What is Chroma?"
        docs = await node.search(query, k=2)
        if docs:
            logger.info(f"✅ Success: Document found for query '{query}'")
            for d in docs:
                logger.info(f"- {d.page_content} (meta={d.metadata})")
        else:
            logger.warning(f"❌ No docs found for query '{query}'")
    except Exception as e:
        logger.error(f"Search failed: {e}")


if __name__ == "__main__":
    asyncio.run(test_chroma_with_cache())

In [ ]:
# # nodes.py

# import logging
# from typing import Dict, Any, List
# import asyncio
# import time
# import hashlib
# from langchain_core.documents import Document
# from pydantic import ValidationError

# # from KnowledgeState import KnowledgeState, CrawlStatus, SubQuery, convert_langchain_doc_to_document_result, DocumentSource, ContentSource
# # from tools import SearchEngine, CrawlerEngine, CleanerEngine, VectorStoreNode
# # from config import GlobalConfig
# from datetime import datetime, timezone

# logger = logging.getLogger(__name__)

# # --- Helper function for LLM calls with retry logic ---

# async def _call_llm(llm: Any, prompt: str) -> str:
#     """Helper to call the LLM and extract the response with retry logic."""
#     max_retries = 3
#     for attempt in range(max_retries):
#         try:
#             # Assuming an async-capable LLM client
#             response = await llm.ainvoke(prompt)
#             if hasattr(response, 'content'):
#                 return response.content
#             return str(response)
#         except Exception as e:
#             logger.warning(f"LLM call attempt {attempt + 1} failed: {e}")
#             if attempt == max_retries - 1:
#                 raise Exception(f"All LLM call attempts failed: {e}")
#             await asyncio.sleep(1)

# # ---- Node functions for LangGraph ----

# async def kb_lookup_node(state: KnowledgeState, vectorstore_node: VectorStoreNode) -> KnowledgeState:
#     """LangGraph node for KB lookup."""
#     try:
#         results_with_scores = await asyncio.to_thread(
#             vectorstore_node._vectorstore.similarity_search_with_score,
#             query=state.main_query,
#             k=6
#         )

#         state.knowledge_base_results.clear()

#         for doc, score in results_with_scores:
#             doc_result = convert_langchain_doc_to_document_result(
#                 doc,
#                 source_type=DocumentSource.KNOWLEDGE_BASE,
#                 relevance_score=1.0 - score
#             )
#             state.knowledge_base_results.append(doc_result)

#         if results_with_scores:
#             avg_score = sum(score for _, score in results_with_scores) / len(results_with_scores)
#             confidence = max(0.0, min(1.0, 1.0 - (avg_score / 2.0)))
#             state.state_metrics["kb_confidence"] = confidence
#         else:
#             state.state_metrics["kb_confidence"] = 0.0

#         logger.info(f"KB lookup complete. Found {len(results_with_scores)} docs, confidence: {state.state_metrics.get('kb_confidence', 0.0):.2f}")

#     except Exception as e:
#         logger.error(f"KB lookup failed: {e}")
#         state.add_error(f"KB lookup failed: {str(e)}")
#         state.state_metrics["kb_confidence"] = 0.0

#     return state


# async def generate_subqueries_node(state: KnowledgeState, llm: Any) -> KnowledgeState:
#     """LangGraph node to generate subqueries."""
#     try:
#         prompt = f"""
#         Given the main query: "{state.main_query}"
#         Generate 3-4 focused sub-queries that would help find comprehensive knowledge.
#         Consider different aspects like:
#         - Technical details and implementation
#         - Recent developments and trends
#         - Practical applications and examples
#         - Comparative analysis and alternatives
#         Format your response as a list of queries, one per line:
#         - [subquery 1]
#         - [subquery 2]
#         - [subquery 3]
#         - [subquery 4]
#         """
#         response_content = await _call_llm(llm, prompt)

#         lines = [line.strip() for line in response_content.split('\n') if line.strip().startswith('-')]
#         subqueries_texts = [line[1:].strip() for line in lines if len(line) > 1]

#         subqueries = []
#         for i, sq in enumerate(subqueries_texts):
#             if sq:
#                 priority = 0.9 if i == 0 else (0.8 if i == 1 else 0.7)
#                 deep = i < 2
#                 query_type = ("technical" if "technical" in sq.lower() or "implementation" in sq.lower() else
#                               "trends" if "recent" in sq.lower() or "developments" in sq.lower() else "general")
#                 subqueries.append(SubQuery(query=sq.strip(), priority=priority, deep=deep, query_type=query_type))

#         if subqueries:
#             state.add_subqueries(subqueries)
#             return state

#         logger.info("Generated no new subqueries.")

#     except Exception as e:
#         logger.error(f"Failed to generate subqueries: {e}")
#         state.add_error(f"Subquery generation failed: {str(e)}")

#     return state


# async def external_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
#     """LangGraph node for external search."""
#     try:
#         await search_engine.run_searches(queries=[state.main_query], max_results_per_query=8)
#         logger.info("External search tool invoked successfully.")
#     except Exception as e:
#         logger.error(f"External search failed: {e}")
#         state.add_error(f"External search failed: {str(e)}")
#     return state


# async def extract_and_rank_urls_node(state: KnowledgeState, url_analyzer: URLAnalyzer) -> KnowledgeState:
#     """LangGraph node to extract and rank URLs."""
#     try:
#         search_results_dicts = [
#             {'url': r.metadata.get('url', ''), 'snippet': r.page_content}
#             for r in state.external_api_results
#         ]

#         await url_analyzer.analyze_batch(search_results=search_results_dicts, query=state.main_query)
#         logger.info("URL analyzer tool invoked successfully.")

#     except Exception as e:
#         logger.error(f"URL extraction failed: {e}")
#         state.add_error(f"URL extraction failed: {str(e)}")

#     return state


# async def web_crawling_node(state: KnowledgeState, crawler_engine: CrawlerEngine) -> KnowledgeState:
#     """LangGraph node for web crawling."""
#     try:
#         if not state.crawl_queue:
#             logger.info("No URLs in crawl queue. Skipping crawling.")
#             return {"crawl_status": CrawlStatus.NEUTRAL}

#         state.crawl_status = CrawlStatus.IN_PROGRESS

#         max_crawls = crawler_engine.config.max_concurrent
#         urls_to_crawl = state.crawl_queue[:max_crawls]

#         await crawler_engine.crawl_urls_batch(urls_to_crawl)

#         state.crawl_queue = state.crawl_queue[max_crawls:]

#         if state.metadata.crawl_stats.successful > 0:
#             state.crawl_status = CrawlStatus.SUCCESS
#         else:
#             state.crawl_status = CrawlStatus.FAILED

#         logger.info(f"Crawling complete. Status: {state.crawl_status}")

#     except Exception as e:
#         logger.error(f"Web crawling failed: {e}")
#         state.add_error(f"Web crawling failed: {str(e)}")
#         state.crawl_status = CrawlStatus.FAILED

#     return state


# async def assess_content_quality_node(state: KnowledgeState) -> KnowledgeState:
#     """LangGraph node for assessing content quality."""
#     try:
#         quality_score = state.calculate_weighted_quality_score()
#         state.state_metrics["quality_score"] = quality_score

#         gaps = []
#         if len(state.knowledge_base_results) + len(state.crawler_results) < 2:
#             gaps.append("Insufficient knowledge base or crawled results")
#         if state.state_metrics.get("kb_confidence", 0.0) < 0.5:
#             gaps.append("Low knowledge base confidence")
#         if quality_score < 0.3:
#             gaps.append("Overall low content quality")

#         state.gaps = gaps

#         logger.info(f"Content quality assessment: score={quality_score:.2f}, gaps={len(gaps)}")

#     except Exception as e:
#         logger.error(f"Content quality assessment failed: {e}")
#         state.add_error(f"Content quality assessment failed: {str(e)}")

#     return state


# async def subquery_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
#     """LangGraph node for targeted search using subqueries."""
#     state = state.ensure
#     try:
#         subqueries_to_search = [sq.query for sq in state.sub_queries_to_generate][:3]

#         if not subqueries_to_search:
#             logger.info("No subqueries available for search. Skipping.")
#             return {}

#         await search_engine.run_searches(queries=subqueries_to_search, max_results_per_query=5)

#         # Move subqueries from `to_generate` to `processed`
#         processed_subqueries = [sq.query for sq in state.sub_queries_to_generate[:3]]
#         state.processed_queries.extend(processed_subqueries)
#         state.sub_queries_to_generate = state.sub_queries_to_generate[3:]

#         logger.info(f"Subquery search invoked for: {processed_subqueries}")

#     except Exception as e:
#         logger.error(f"Subquery search failed: {e}")
#         state.add_error(f"Subquery search failed: {str(e)}")

#     return state


# async def process_documents_node(state: KnowledgeState, cleaner_engine: CleanerEngine) -> KnowledgeState:
#     """LangGraph node for cleaning and chunking documents."""
#     state = KnowledgeState.create_knowledge_state_from_dict(state)
#     try:
#         all_docs = []
#         all_docs.extend(state.knowledge_base_results)
#         all_docs.extend(state.crawler_results)
#         all_docs.extend(state.external_api_results)

#         if not all_docs:
#             logger.info("No documents to process. Skipping.")
#             return state

#         # Call the refactored cleaner tool
#         await cleaner_engine.clean_and_chunk_documents(all_docs)

#         logger.info(f"Document processing complete. Chunks ready.")

#     except Exception as e:
#         logger.error(f"Document processing failed: {e}")
#         state.add_error(f"Document processing failed: {str(e)}")

#     return state


# async def finalize_results_node(state: KnowledgeState) -> KnowledgeState:
#     """LangGraph node for finalizing and sorting results."""
#     try:
#         all_results = state.knowledge_base_results + state.cleaned_chunks

#         if not all_results:
#             logger.info("No documents to finalize.")
#             return state

#         seen_hashes = set()
#         final_docs = []
#         for doc_result in all_results:
#             if doc_result.page_content:
#                 content_hash = hashlib.sha256(doc_result.page_content.encode()).hexdigest()
#                 if content_hash not in seen_hashes:
#                     seen_hashes.add(content_hash)
#                     final_docs.append(doc_result)

#         def sort_key(doc_result):
#             source_priority = {
#                 DocumentSource.KNOWLEDGE_BASE: 4,
#                 DocumentSource.EXTERNAL_API: 3,
#                 DocumentSource.WEB_CRAWL: 2,
#                 DocumentSource.SEARCH_SNIPPET: 1
#             }.get(doc_result.source_type, 1)

#             content_boost = {
#                 ContentSource.ACADEMIC: 1.3,
#                 ContentSource.GOVERNMENT: 1.2,
#                 ContentSource.WIKI: 1.1,
#                 ContentSource.OTHER: 1.0
#             }.get(doc_result.content_source, 1.0) if doc_result.content_source else 1.0

#             relevance = doc_result.relevance_score or 0.5
#             length_factor = min(1.0, (doc_result.content_length or 0) / 1000)

#             return state

#         final_docs.sort(key=sort_key, reverse=True)

#         max_final_docs = state.state_metrics.get('max_final_documents', 20)
#         state.cleaned_chunks = final_docs[:max_final_docs]
#         state.metadata.total_iterations += 1

#         final_quality = state.calculate_weighted_quality_score()
#         state.state_metrics["final_quality_score"] = final_quality

#         state.metadata.final = True

#         logger.info(f"Finalization complete. {len(state.cleaned_chunks)} final documents.")

#     except Exception as e:
#         logger.error(f"Finalization failed: {e}")
#         state.add_error(f"Finalization failed: {str(e)}")

#     return state


# async def error_handler_node(state: KnowledgeState) -> KnowledgeState:
#     """
#     Handles errors recorded in the state.
#     """
#     if state.errors:
#         logger.error(f"Graph encountered errors: {state.errors}")
#     return state


In [ ]:
# nodes.py

import logging
import asyncio
from typing import Dict, Any, List
import time
import hashlib
from langchain_core.documents import Document
from pydantic import ValidationError

# Assume KnowledgeState, CrawlStatus, SubQuery, convert_langchain_doc_to_document_result, DocumentSource, ContentSource are defined
# Assume SearchEngine, CrawlerEngine, CleanerEngine, VectorStoreNode are defined
# Assume GlobalConfig is defined
from datetime import datetime, timezone
import dateutil.parser
from bs4 import BeautifulSoup
import re
import httpx
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, AsyncChromiumLoader
from htmldate import find_date
# Import init_chat_model if needed for direct initialization or type hinting
from langchain.chat_models import init_chat_model


logger = logging.getLogger(__name__)

# --- Helper function for LLM calls with retry logic ---

async def _call_llm(llm: Any, prompt: str) -> str:
    """Helper to call the LLM and extract the response with retry logic."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Assuming an async-capable LLM client
            response = await llm.ainvoke(prompt)
            if hasattr(response, 'content'):
                return response.content
            return str(response)
        except Exception as e:
            logger.warning(f"LLM call attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                raise Exception(f"All LLM call attempts failed: {e}")
            await asyncio.sleep(1)

# ---- Node functions for LangGraph ----

async def kb_lookup_node(state: KnowledgeState, vectorstore_node: VectorStoreNode) -> KnowledgeState:
    """LangGraph node for KB lookup."""
    try:
        results_with_scores = await asyncio.to_thread(
            vectorstore_node._vectorstore.similarity_search_with_score,
            query=state.main_query,
            k=6
        )

        state.knowledge_base_results.clear()

        for doc, score in results_with_scores:
            # Ensure relevance_score is between 0.0 and 1.0
            # Assuming 'score' is a distance metric where lower is better relevance.
            # Clamping the relevance score to be between 0.0 and 1.0
            relevance_score = max(0.0, min(1.0, 1.0 - score))

            doc_result = KnowledgeState.convert_langchain_doc_to_document_result(
                doc,
                source_type=DocumentSource.KNOWLEDGE_BASE,
                relevance_score=relevance_score # Use the clamped score
            )
            state.knowledge_base_results.append(doc_result)

        if results_with_scores:
            # Recalculate average score based on the clamped relevance scores for confidence
            avg_relevance_score = sum(
                max(0.0, min(1.0, 1.0 - score)) for doc, score in results_with_scores
            ) / len(results_with_scores)
            # Use average relevance score for confidence calculation
            confidence = avg_relevance_score # Directly use average relevance as confidence (0-1)

            state.state_metrics["kb_confidence"] = confidence
        else:
            state.state_metrics["kb_confidence"] = 0.0

        logger.info(f"KB lookup complete. Found {len(results_with_scores)} docs, confidence: {state.state_metrics.get('kb_confidence', 0.0):.2f}")

    except Exception as e:
        logger.error(f"KB lookup failed: {e}")
        state.add_error(f"KB lookup failed: {str(e)}")
        state.state_metrics["kb_confidence"] = 0.0

    return state


async def generate_subqueries_node(state: KnowledgeState, llm: Any) -> KnowledgeState:
    """LangGraph node to generate subqueries."""
    try:
        prompt = f"""
        Given the main query: "{state.main_query}"
        Generate 3-4 focused sub-queries that would help find comprehensive knowledge.
        Consider different aspects like:
        - Technical details and implementation
        - Recent developments and trends
        - Practical applications and examples
        - Comparative analysis and alternatives
        Format your response as a list of queries, one per line:
        - [subquery 1]
        - [subquery 2]
        - [subquery 3]
        - [subquery 4]
        """
        response_content = await _call_llm(llm, prompt)

        lines = [line.strip() for line in response_content.split('\n') if line.strip().startswith('-')]
        subqueries_texts = [line[1:].strip() for line in lines if len(line) > 1]

        subqueries = []
        for i, sq in enumerate(subqueries_texts):
            if sq:
                priority = 0.9 if i == 0 else (0.8 if i == 1 else 0.7)
                deep = i < 2
                query_type = ("technical" if "technical" in sq.lower() or "implementation" in sq.lower() else
                              "trends" if "recent" in sq.lower() or "developments" in sq.lower() else "general")
                subqueries.append(SubQuery(query=sq.strip(), priority=priority, deep=deep, query_type=query_type))

        if subqueries:
            state.add_queries(subqueries)
            return state

        logger.info("Generated no new subqueries.")

    except Exception as e:
        logger.error(f"Failed to generate subqueries: {e}")
        state.add_error(f"Subquery generation failed: {str(e)}")

    return state


async def external_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
    """LangGraph node for external search."""
    logger.info("Entering external_search_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to external_search_node: {state.model_dump_json()}") # Add detailed state logging

    try:
        logger.debug("Calling search_engine.run_searches for main query.")
        # Call run_searches and capture the returned state
        updated_state = await search_engine.run_searches(state, queries=[state.main_query], max_results_per_query=8)
        logger.info("External search tool invoked successfully.")

        # Log the state of external_api_results *after* run_searches modifies it
        logger.info(f"After search_engine.run_searches in external_search_node: updated_state.external_api_results has {len(updated_state.external_api_results)} documents.")
        for i, doc in enumerate(updated_state.external_api_results[:3]):
             logger.debug(f"external_search_node - Added Doc {i+1}: URL={doc.metadata.get('url', 'N/A')}, Source={doc.source_type}, Content={doc.page_content[:100]}...")


        # --- Add explicit logging right before returning the state object ---
        logger.info(f"external_search_node: State external_api_results count BEFORE returning: {len(updated_state.external_api_results)}")
        logger.debug(f"State object BEFORE returning: {updated_state.model_dump_json()}")
        logger.debug(f"State type BEFORE returning: {type(updated_state)}")


        # Return the updated state object
        return updated_state

    except Exception as e:
        logger.error(f"External search failed: {e}", exc_info=True)
        state.add_error(f"External search failed: {str(e)}")
        logger.info(f"Exiting external_search_node with error. State summary: errors={len(state.errors)}, external_api_results={len(state.external_api_results)}")
        logger.debug(f"State before returning from external_search_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")

        # Return original state even on error
        return state


async def extract_and_rank_urls_node(state: KnowledgeState, url_analyzer: URLAnalyzer) -> KnowledgeState:
    """LangGraph node to extract and rank URLs."""
    logger.info("Entering extract_and_rank_urls_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to extract_and_rank_urls_node: {state.model_dump_json()}") # Add detailed state logging


    try:
        # Ensure state.external_api_results is a list of DocumentResult objects
        if not isinstance(state.external_api_results, list):
            logger.error(f"state.external_api_results is not a list: {type(state.external_api_results)}")
            # Attempt to recover or add error
            state.add_error(f"Invalid type for external_api_results: {type(state.external_api_results)}")
            return state

        # Filter for valid DocumentResult objects if needed
        valid_search_results = [r for r in state.external_api_results if isinstance(r, DocumentResult)]
        if len(valid_search_results) != len(state.external_api_results):
            logger.warning(f"Found non-DocumentResult objects in state.external_api_results. Expected {len(state.external_api_results)}, found {len(valid_search_results)} valid.")
            state.add_error(f"Invalid objects found in external_api_results.")


        search_results_dicts = [
            {'url': r.metadata.get('url', ''), 'snippet': r.page_content}
            for r in valid_search_results # Use valid_search_results
        ]
        logger.info(f"extract_and_rank_urls_node received {len(search_results_dicts)} search result dicts for analysis.")


        state = await url_analyzer.analyze_batch(state=state, search_results=search_results_dicts, query=state.main_query)
        logger.info("URL analyzer tool invoked successfully.")

        # Add logging after URL analysis to check state
        logger.debug(f"After url_analyzer.analyze_batch: state.extracted_urls has {len(state.extracted_urls)} URLs, state.crawl_queue has {len(state.crawl_queue)} URLs.")
        # Log content preview of the first few added URLs if any
        for i, url_info in enumerate(state.crawl_queue[:3]):
             logger.debug(f"extract_and_rank_urls_node - Added URL {i+1}: URL={url_info.url}, Priority={url_info.priority}, Should Crawl={url_info.should_crawl}")


        logger.debug(f"State before returning from extract_and_rank_urls_node: {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type BEFORE returning: {type(state)}")
        return state # Return the modified state


    except Exception as e:
        logger.error(f"URL extraction failed: {e}", exc_info=True)
        state.add_error(f"URL extraction failed: {str(e)}")
        logger.info(f"Exiting extract_and_rank_urls_node with error. State summary: errors={len(state.errors)}, extracted_urls={len(state.extracted_urls)}, crawl_queue={len(state.crawl_queue)}")
        logger.debug(f"State before returning from extract_and_rank_urls_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")
        return state # Return state even on error


async def web_crawling_node(state: KnowledgeState, crawler_engine: CrawlerEngine) -> KnowledgeState:
    """LangGraph node for web crawling."""
    logger.info("Entering web_crawling_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to web_crawling_node: {state.model_dump_json()}") # Add detailed state logging

    try:
        if not state.crawl_queue:
            logger.info("No URLs in crawl queue. Skipping crawling.")
            state.crawl_status = CrawlStatus.NEUTRAL
            logger.debug(f"State before returning from web_crawling_node (skipping): {state.model_dump_json()}")
            logger.debug(f"State type BEFORE returning (skipping): {type(state)}")
            return state

        state.crawl_status = CrawlStatus.IN_PROGRESS
        logger.info(f"Starting web crawling for {len(state.crawl_queue)} URLs in queue.")

        # The crawler_engine.crawl_urls_batch method now handles batching and queue management internally.
        # Pass the entire current crawl_queue to it.
        # The method will select max_concurrent URLs, crawl them, and update state.crawl_queue
        # by removing processed URLs and re-adding those needing retries.
        state = await crawler_engine.crawl_urls_batch(state, state.crawl_queue)

        # After the batch is processed by the engine, check the state of the queue
        if not state.crawl_queue:
            # If the queue is empty after processing, all relevant URLs were either successful or discarded
            state.crawl_status = CrawlStatus.SUCCESS
        else:
            # If there are still URLs in the queue, it means some need retries
            state.crawl_status = CrawlStatus.PARTIAL # Indicate partial success/ongoing process

        logger.info(f"Crawling complete for batch. Status: {state.crawl_status}. Remaining crawl queue size: {len(state.crawl_queue)}")
        logger.debug(f"State before returning from web_crawling_node: {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type BEFORE returning: {type(state)}")
        return state # Return the modified state

    except Exception as e:
        logger.error(f"Web crawling failed: {e}", exc_info=True)
        state.add_error(f"Web crawling failed: {str(e)}")
        state.crawl_status = CrawlStatus.FAILED
        logger.info(f"Exiting web_crawling_node with error. State summary: errors={len(state.errors)}, crawl_queue={len(state.crawl_queue)}, crawl_stats={state.metadata.crawl_stats.model_dump()}")
        logger.debug(f"State before returning from web_crawling_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")
        return state # Return state even on error

async def process_search_results_node(state: KnowledgeState, cleaner_engine: CleanerEngine) -> KnowledgeState:
    """LangGraph node to process raw search results using the CleanerEngine."""
    logger.info("Entering process_search_results_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to process_search_results_node: {state.model_dump_json()}") # Add detailed state logging

    try:
        raw_search_results = state.external_api_results
        logger.info(f"process_search_results_node received {len(raw_search_results)} raw search results.")

        if not raw_search_results:
            logger.info("No raw search results to process. Skipping.")
            # Explicitly log what is being returned in the state
            logger.debug(f"State before returning from process_search_results_node (skipping): {state.model_dump_json()}")
            logger.debug(f"State type BEFORE returning (skipping): {type(state)}")
            return state

        logger.info(f"Processing {len(raw_search_results)} raw search results in process_search_results_node...")
        # Log content preview of the first few documents being sent to the cleaner
        for i, doc in enumerate(raw_search_results[:3]):
             logger.debug(f"process_search_results_node sending Doc {i+1} to cleaner: URL={doc.metadata.get('url', 'N/A')}, Source={doc.source_type}, Content={doc.page_content[:100]}...")

        # Add logging right before calling the cleaner
        logger.debug(f"Calling cleaner_engine.clean_and_chunk_documents with {len(raw_search_results)} documents.")
        # The cleaner_engine will clean and chunk these documents and add them to state.cleaned_chunks
        # The cleaner_engine already handles adding chunks to state.cleaned_chunks
        state=await cleaner_engine.clean_and_chunk_documents(state, raw_search_results)

        # Clear the raw search results after processing them
        state.external_api_results = []

        logger.info(f"Finished processing raw search results in process_search_results_node. Total cleaned_chunks count: {len(state.cleaned_chunks)}")
        logger.debug(f"State before returning from process_search_results_node: {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type BEFORE returning: {type(state)}")
        return state

    except Exception as e:
        logger.error(f"An unexpected error occurred within process_search_results_node: {e}", exc_info=True)
        state.add_error(f"Error in process_search_results_node: {str(e)}")
        logger.info(f"Exiting process_search_results_node with error. State summary: errors={len(state.errors)}, cleaned_chunks={len(state.cleaned_chunks)}, external_api_results={len(state.external_api_results)}")
        logger.debug(f"State before returning from process_search_results_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")
        return state # Return state even on error


async def assess_content_quality_node(state: KnowledgeState) -> KnowledgeState:
    """LangGraph node for assessing content quality."""
    logger.info("Entering assess_content_quality_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to assess_content_quality_node: {state.model_dump_json()}") # Add detailed state logging


    try:
        quality_score = state.calculate_weighted_quality_score()
        state.state_metrics["quality_score"] = quality_score

        gaps = []
        # Check counts of relevant documents (KB results + cleaned chunks)
        relevant_doc_count = len(state.knowledge_base_results) + len(state.cleaned_chunks)
        if relevant_doc_count < state.state_metrics.get("min_results", 2): # Use config value for min results
             gaps.append(f"Insufficient relevant documents ({relevant_doc_count} found, min {state.state_metrics.get('min_results', 2)} required)")

        if state.state_metrics.get("kb_confidence", 0.0) < state.state_metrics.get("confidence_threshold", 0.5): # Use config value for confidence threshold
            gaps.append("Low knowledge base confidence")
        if quality_score < state.state_metrics.get("quality_threshold", 0.3): # Use config value for quality threshold
            gaps.append("Overall low content quality")

        state.gaps = gaps

        logger.info(f"Content quality assessment: score={quality_score:.2f}, gaps={len(gaps)}. Relevant docs: {relevant_doc_count}")
        logger.debug(f"State before returning from assess_content_quality_node: {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type BEFORE returning: {type(state)}")
        return state # Return the modified state

    except Exception as e:
        logger.error(f"Content quality assessment failed: {e}", exc_info=True)
        state.add_error(f"Content quality assessment failed: {str(e)}")
        logger.info(f"Exiting assess_content_quality_node with error. State summary: errors={len(state.errors)}, quality_score={state.state_metrics.get('quality_score', 'N/A')}")
        logger.debug(f"State before returning from assess_content_quality_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")
        return state # Return state even on error


async def subquery_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
    """LangGraph node for targeted search using subqueries."""
    logger.info("Entering subquery_search_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to subquery_search_node: {state.model_dump_json()}") # Add detailed state logging


    try:
        # Select a batch of subqueries to process
        subqueries_to_search = [sq.query for sq in state.sub_queries_to_generate][:state.state_metrics.get("subquery_max_search", 3)] # Use config value for max subqueries

        if not subqueries_to_search:
            logger.info("No subqueries available for search. Skipping.")
            logger.debug(f"State before returning from subquery_search_node (skipping): {state.model_dump_json()}")
            logger.debug(f"State type BEFORE returning (skipping): {type(state)}")
            return state

        logger.info(f"Running subquery search for: {subqueries_to_search}")
        # Call run_searches and capture the returned state
        updated_state = await search_engine.run_searches(state, queries=subqueries_to_search, max_results_per_query=state.state_metrics.get("subquery_results_per_query", 5)) # Use config value for results per subquery

        # Move processed subqueries from `to_generate` to `processed`
        processed_subqueries = [sq.query for sq in state.sub_queries_to_generate[:len(subqueries_to_search)]] # Use the actual number processed
        state.processed_queries.extend(processed_subqueries)
        state.sub_queries_to_generate = state.sub_queries_to_generate[len(subqueries_to_search):]

        # Log the state of external_api_results *after* run_searches modifies it
        logger.info(f"After search_engine.run_searches in subquery_search_node: updated_state.external_api_results has {len(updated_state.external_api_results)} documents.")
        for i, doc in enumerate(updated_state.external_api_results[:3]):
             logger.debug(f"subquery_search_node - Added Doc {i+1}: URL={doc.metadata.get('url', 'N/A')}, Source={doc.source_type}, Content={doc.page_content[:100]}...")


        # --- Add explicit logging right before returning the state object ---
        logger.info(f"subquery_search_node: State external_api_results count BEFORE returning: {len(updated_state.external_api_results)}")
        logger.debug(f"State object BEFORE returning: {updated_state.model_dump_json()}")
        logger.debug(f"State type BEFORE returning: {type(updated_state)}")


        # Return the updated state object
        return updated_state


    except Exception as e:
        logger.error(f"Subquery search failed: {e}", exc_info=True)
        state.add_error(f"Subquery search failed: {str(e)}")
        logger.info(f"Exiting subquery_search_node with error. State summary: errors={len(state.errors)}, external_api_results={len(state.external_api_results)}")
        logger.debug(f"State before returning from subquery_search_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")
        return state # Return state even on error


async def process_crawled_documents_node(state: KnowledgeState, cleaner_engine: CleanerEngine) -> KnowledgeState:
    """LangGraph node for cleaning and chunking crawled documents."""
    logger.info("Entering process_crawled_documents_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to process_crawled_documents_node: {state.model_dump_json()}") # Add detailed state logging

    # Add a top-level try-except to catch any exceptions
    try:
        crawled_docs = state.crawler_results
        logger.info(f"process_crawled_documents_node received {len(crawled_docs)} documents.")
        if crawled_docs:
            logger.debug(f"First document in crawled_docs: {crawled_docs[0]}")

        if not crawled_docs:
            logger.info("No crawled documents to process in process_crawled_documents_node. Skipping.")
            # Clear the list defensively if it somehow wasn't empty but contained no valid docs
            state.crawler_results = []
            logger.info("Exiting process_crawled_documents_node early as no documents to process.")
            logger.debug(f"State before returning from process_crawled_documents_node (early exit): {state.model_dump_json()}") # Add detailed state logging
            logger.debug(f"State type BEFORE returning (early exit): {type(state)}")
            return state

        logger.info(f"Processing {len(crawled_docs)} crawled documents in process_crawled_documents_node...")
        # Log content preview of the first few documents being sent to the cleaner
        for i, doc in enumerate(crawled_docs[:3]):
             logger.debug(f"process_crawled_documents_node sending Doc {i+1} to cleaner: URL={doc.metadata.get('url', 'N/A')}, Source={doc.source_type}, Content={doc.page_content[:100]}...")

        # Add logging right before calling the cleaner
        logger.debug(f"Calling cleaner_engine.clean_and_chunk_documents with {len(crawled_docs)} documents.")
        # The cleaner_engine will clean and chunk these documents and add them to state.cleaned_chunks
        # The cleaner_engine already handles adding chunks to state.cleaned_chunks
        state = await cleaner_engine.clean_and_chunk_documents(state, crawled_docs)

        # Clear the raw crawler results after processing them
        state.crawler_results = []

        logger.info(f"Finished processing crawled documents in process_crawled_documents_node. Total cleaned_chunks count: {len(state.cleaned_chunks)}")
        logger.debug(f"State before returning from process_crawled_documents_node: {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type BEFORE returning: {type(state)}")
        return state

    except Exception as e:
        logger.error(f"An unexpected error occurred within process_crawled_documents_node: {e}", exc_info=True)
        state.add_error(f"Error in process_crawled_documents_node: {str(e)}")
        logger.info(f"Exiting process_crawled_documents_node with error. State summary: errors={len(state.errors)}, cleaned_chunks={len(state.cleaned_chunks)}, crawler_results={len(state.crawler_results)}")
        logger.debug(f"State before returning from process_crawled_documents_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")
        return state # Return state even on error


async def finalize_results_node(state: KnowledgeState) -> KnowledgeState:
    """LangGraph node for finalizing and sorting results."""
    logger.info("Entering finalize_results_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to finalize_results_node: {state.model_dump_json()}") # Add detailed state logging


    try:
        # Finalization now combines KB results and the cleaned chunks (from both search and crawl)
        all_results = state.knowledge_base_results + state.cleaned_chunks

        if not all_results:
            logger.info("No documents to finalize.")
            logger.debug(f"State before returning from finalize_results_node (skipping): {state.model_dump_json()}")
            logger.debug(f"State type BEFORE returning (skipping): {type(state)}")
            return state

        seen_hashes = set()
        final_docs = []
        for doc_result in all_results:
            if doc_result.page_content:
                # Use a combination of URL and content hash for uniqueness if available
                identifier = doc_result.metadata.get('url', '') + hashlib.sha256(doc_result.page_content.encode()).hexdigest()
                if identifier not in seen_hashes:
                    seen_hashes.add(identifier)
                    final_docs.append(doc_result)

        def sort_key(doc_result):
            source_priority = {
                DocumentSource.KNOWLEDGE_BASE: 4,
                DocumentSource.EXTERNAL_API: 3, # Search results (raw snippets - should be processed)
                DocumentSource.WEB_CRAWL: 2,    # Crawled results (raw - should be processed)
                DocumentSource.SEARCH_SNIPPET: 1 # Raw snippets (should be processed before finalization, but keep for robustness)
            }.get(doc_result.source_type, 1)

            content_boost = {
                ContentSource.ACADEMIC: 1.3,
                ContentSource.GOVERNMENT: 1.2,
                ContentSource.WIKI: 1.1,
                ContentSource.OTHER: 1.0
            }.get(doc_result.content_source, 1.0) if doc_result.content_source else 1.0

            # Use the highest available relevance score (embedding, LLM, or original search snippet score)
            # Prioritize embedding score if available, then LLM score, then original relevance, then default
            relevance = doc_result.metadata.get("embedding_relevance_score",
                                                 doc_result.metadata.get("llm_relevance_score",
                                                                          doc_result.relevance_score or 0.5))
            # Ensure relevance is clamped to [0, 1]
            relevance = max(0.0, min(1.0, relevance))


            length_factor = min(1.0, (doc_result.content_length or 0) / 1000)

            # Combine factors - adjust weights as needed
            # Example: Prioritize KB, then recent/relevant crawled content, then search snippets
            # Incorporate source priority, content boost, and relevance
            # A simple weighted sum, or a more complex ranking function could be used
            # For now, let's use a combination of source priority and relevance,
            # and potentially content boost.
            # Higher score is better.
            combined_score = (source_priority * 0.4) + (relevance * 0.4) + (content_boost * 0.2)
            return combined_score

        # Sort by the combined score in descending order
        final_docs.sort(key=sort_key, reverse=True)

        max_final_docs = state.state_metrics.get('max_final_documents', 20)
        # Ensure the state's cleaned_chunks list is updated with the *final_docs* list
        # This is where the final, unique, sorted documents are stored
        state.cleaned_chunks = final_docs[:max_final_docs]
        # Clear other raw/intermediate lists that are no longer needed
        state.knowledge_base_results = []
        state.crawler_results = []
        state.external_api_results = []


        state.metadata.total_iterations += 1

        final_quality = state.calculate_weighted_quality_score()
        state.state_metrics["final_quality_score"] = final_quality

        state.metadata.final = True

        logger.info(f"Finalization complete. {len(state.cleaned_chunks)} final documents.")
        logger.debug(f"State before returning from finalize_results_node: {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type BEFORE returning: {type(state)}")
        return state

    except Exception as e:
        logger.error(f"Finalization failed: {e}", exc_info=True)
        state.add_error(f"Finalization failed: {str(e)}")
        logger.info(f"Exiting finalize_results_node with error. State summary: errors={len(state.errors)}, cleaned_chunks={len(state.cleaned_chunks)}")
        logger.debug(f"State before returning from finalize_results_node (error exit): {state.model_dump_json()}") # Add detailed state logging
        logger.debug(f"State type before returning (error exit): {type(state)}")
        return state # Return state even on error


async def error_handler_node(state: KnowledgeState) -> KnowledgeState:
    """
    Handles errors recorded in the state.
    """
    logger.info("Entering error_handler_node.")
    logger.debug(f"State type on entry: {type(state)}")
    logger.debug(f"State on entry to error_handler_node: {state.model_dump_json()}") # Add detailed state logging


    if state.errors:
        logger.error(f"Graph encountered errors: {state.errors}")
    # The error handler doesn't add new errors, it just logs existing ones.
    # It should typically lead to the END state.
    logger.info("Exiting error_handler_node.")
    logger.debug(f"State type BEFORE returning: {type(state)}")
    return state # Return the state with errors

In [ ]:
# decisions.py

import logging
from typing import Dict, Any
# from KnowledgeState import KnowledgeState, CrawlStatus
# from config import GlobalConfig

logger = logging.getLogger(__name__)

# --- Decision functions for LangGraph conditional edges ---

def should_search_external_node(state: KnowledgeState, config: GlobalConfig) -> str:
    """Decision logic based on KB confidence, using GlobalConfig."""
    try:
        confidence_threshold = config.decisions.confidence_threshold
        kb_confidence = state.state_metrics.get("kb_confidence", 0.0)

        if kb_confidence >= confidence_threshold:
            return "finalize"
        else:
            return "external_search"
    except Exception as e:
        logger.error(f"External search decision failed: {e}")
        # Transition to error handler on failure
        state.add_error(f"External search decision failed: {e}")
        return "error_handler"

def should_crawl_urls_node(state: KnowledgeState, config: GlobalConfig) -> str:
    """Decision logic for web crawling, using GlobalConfig."""
    try:
        if not state.should_continue_crawling:
            return "assess_quality"

        crawl_stats = state.metadata.crawl_stats
        if crawl_stats.attempted > 0 and state.success_rate < config.decisions.low_success_rate_threshold:
            logger.info("Skipping crawling due to low success rate.")
            return "assess_quality"

        if state.crawl_queue:
            return "web_crawling"
        else:
            return "assess_quality"

    except Exception as e:
        logger.error(f"Crawl decision failed: {e}")
        # Transition to error handler on failure
        state.add_error(f"Crawl decision failed: {e}")
        return "error_handler"

def should_do_subquery_search_node(state: KnowledgeState, config: GlobalConfig) -> str:
    """Decision logic for subquery search, using GlobalConfig."""
    try:
        quality_threshold = config.decisions.quality_threshold
        max_iterations = config.decisions.max_iterations

        if state.has_sufficient_content and state.calculate_weighted_quality_score() >= quality_threshold:
            return "process_documents"

        if state.metadata.total_iterations >= max_iterations:
            return "process_documents"

        if state.sub_queries_to_generate:
            return "subquery_search"
        else:
            return "process_documents"

    except Exception as e:
        logger.error(f"Subquery search decision failed: {e}")
        # Transition to error handler on failure
        state.add_error(f"Subquery search decision failed: {e}")
        return "error_handler"


In [ ]:
# workflow.py

!pip install langgraph --quiet

import asyncio
import logging
from typing import Dict, Any
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# from nodes import (
#     kb_lookup_node,
#     generate_subqueries_node,
#     external_search_node,
#     extract_and_rank_urls_node,
#     web_crawling_node,
#     assess_content_quality_node,
#     subquery_search_node,
#     process_documents_node, # This was a placeholder comment, not a node name
#     finalize_results_node,
#     error_handler_node
# )
# from KnowledgeState import LangGraphKnowledgeState
# from decisions import (
#     should_search_external_node,
#     should_crawl_urls_node,
#     should_do_subquery_search_node
# )
# from config import GlobalConfig

logger = logging.getLogger(__name__)

def create_knowledge_graph(tools_and_configs: Dict[str, Any]) -> StateGraph:
    """
    Creates and compiles the LangGraph for knowledge acquisition.
    Uses the modern TypedDict-based state for compatibility.
    """
    try:
        # Assuming KnowledgeState, CrawlStatus, etc., are defined in other cells
        workflow = StateGraph(KnowledgeState) # Use the Pydantic model directly

        config: GlobalConfig = tools_and_configs["config"]
        llm = tools_and_configs['llm'] # Assuming LLM is passed in tools_and_configs
        vectorstore_node = tools_and_configs['vectorstore_node']
        url_analyzer = tools_and_configs['url_analyzer']
        search_engine = tools_and_configs['search_engine']
        crawler_engine = tools_and_configs['crawler_engine']
        cleaner_engine = tools_and_configs['cleaner_engine']

        # Define async wrapper functions for nodes that need extra arguments
        async def kb_lookup_wrapper(state):
            return await kb_lookup_node(state, vectorstore_node)

        async def generate_subqueries_wrapper(state):
            return await generate_subqueries_node(state, llm)

        async def external_search_wrapper(state):
            # Pass search_engine here, node will pass state to search_engine.run_searches
            return await external_search_node(state, search_engine)

        async def extract_urls_wrapper(state):
            return await extract_and_rank_urls_node(state, url_analyzer)

        async def web_crawling_wrapper(state):
            return await web_crawling_node(state, crawler_engine)

        async def assess_quality_wrapper(state):
            # This node doesn't take extra tools, but wrap for consistency or if it might later
            return await assess_content_quality_node(state)

        async def subquery_search_wrapper(state):
             # Pass search_engine here, node will pass state to search_engine.run_searches
            return await subquery_search_node(state, search_engine)

        async def process_search_results_wrapper(state):
             return await process_search_results_node(state, cleaner_engine)

        async def process_crawled_documents_wrapper(state):
             return await process_crawled_documents_node(state, cleaner_engine)

        async def finalize_wrapper(state):
            # This node doesn't take extra tools, but wrap for consistency
            return await finalize_results_node(state)

        async def error_handler_wrapper(state):
            # This node doesn't take extra tools, but wrap for consistency
            return await error_handler_node(state)


        # Add all nodes, passing the async wrapper functions
        workflow.add_node("kb_lookup", kb_lookup_wrapper)
        workflow.add_node("generate_subqueries", generate_subqueries_wrapper)
        workflow.add_node("external_search", external_search_wrapper)
        workflow.add_node("extract_urls", extract_urls_wrapper)
        workflow.add_node("web_crawling", web_crawling_wrapper)
        workflow.add_node("assess_quality", assess_quality_wrapper)
        workflow.add_node("subquery_search", subquery_search_wrapper)
        workflow.add_node("process_search_results", process_search_results_wrapper)
        workflow.add_node("process_crawled_documents", process_crawled_documents_wrapper)
        workflow.add_node("finalize", finalize_wrapper)
        workflow.add_node("error_handler", error_handler_wrapper)


        # Define the graph flow
        workflow.set_entry_point("kb_lookup")

        # Decision point after KB lookup
        # Note: should_search_external_node takes state and config and is NOT async
        workflow.add_conditional_edges(
            "kb_lookup",
            lambda state: should_search_external_node(state, config),
            {
                "external_search": "generate_subqueries", # Go to generate subqueries after external search is decided
                "finalize": "finalize",
                "error_handler": "error_handler" # Add error transition
            }
        )

        # Subquery generation and external search
        workflow.add_edge("generate_subqueries", "external_search")
        # After external search, process the raw search results
        workflow.add_edge("external_search", "process_search_results")
        # After processing search results, extract URLs from them
        workflow.add_edge("process_search_results", "extract_urls")


        # Crawling decision after URL extraction
        # This decision needs to check if there are URLs in the crawl queue
        # Note: should_crawl_urls_node takes state and config and is NOT async
        workflow.add_conditional_edges(
            "extract_urls",
            lambda state: should_crawl_urls_node(state, config), # Use the decision function that checks queue and config
            {
                "web_crawling": "web_crawling",
                "assess_quality": "assess_quality", # If no urls to crawl, move to quality assessment
                "error_handler": "error_handler" # Add error transition
            }
        )

        # After web crawling batch, decide based on remaining queue
        # This decision function is NOT async
        def continue_crawling_decision(state: KnowledgeState) -> str:
            """Decision after web crawling batch."""
            # If there are still URLs in the queue (including retries), loop back to process crawled documents
            # Changed logic: After crawling, ALWAYS process the crawled documents first.
            # The decision to continue crawling happens *after* processing the crawled batch.
            logger.info("Decision after web_crawling: Proceeding to process_crawled_documents.")
            return "process_crawled_documents"


        workflow.add_conditional_edges(
             "web_crawling",
             continue_crawling_decision,
             {
                 "process_crawled_documents": "process_crawled_documents", # Always process docs after crawling
                 "error_handler": "error_handler" # Add error transition
             }
        )

        # After processing a batch of crawled documents, decide whether to continue crawling or assess quality
        # This decision function is NOT async
        def decide_after_processing_crawled(state: KnowledgeState, config: GlobalConfig) -> str:
             """Decision after processing crawled documents."""
             # If the crawl queue is still not empty, continue crawling
             if state.crawl_queue:
                  logger.info(f"Crawl queue has {len(state.crawl_queue)} URLs after processing. Looping back to web_crawling.")
                  return "web_crawling"
             else:
                  # If the crawl queue is empty, assess quality to decide if more subqueries/search are needed
                  logger.info("Crawl queue is empty after processing. Moving to assess_quality.")
                  return "assess_quality"

        # Add conditional edge after processing crawled documents
        workflow.add_conditional_edges(
             "process_crawled_documents",
             lambda state: decide_after_processing_crawled(state, config),
             {
                  "web_crawling": "web_crawling", # Loop back to crawl next batch
                  "assess_quality": "assess_quality", # Move to quality assessment if queue is empty
                  "error_handler": "error_handler"
             }
        )


        # Quality-based decisions
        # Note: should_do_subquery_search_node takes state and config and is NOT async
        workflow.add_conditional_edges(
            "assess_quality",
            lambda state: should_do_subquery_search_node(state, config),
            {
                "subquery_search": "subquery_search", # If more search/subqueries needed
                "finalize": "finalize", # If sufficient quality, finalize
                "error_handler": "error_handler" # Add error transition
            }
        )

        # Subquery search loops back to process search results (to process new search results from subquery search)
        workflow.add_edge("subquery_search", "process_search_results") # Go to process_search_results after subquery search


        # Finalize node can transition to END or error_handler
        # The lambda for conditional edge is NOT async
        workflow.add_conditional_edges(
            "finalize",
            lambda state: END if not state.errors else "error_handler",
            {
                END: END,
                "error_handler": "error_handler"
            }
        )


        # Add error transitions (fallback to the error handler)
        # These are fallback edges if a node raises an unhandled exception
        # Redundant edges removed as conditional edges handle transitions including errors
        workflow.add_edge("error_handler", END) # Error handler always ends

        return workflow

    except Exception as e:
        logger.error(f"Graph creation failed: {e}")
        raise

In [ ]:
# main cell

!pip install -U langchain-google-genai --quiet
!pip install sentence-transformers --quiet # Ensure embedding model is available

import logging
import asyncio
from typing import Optional, Dict, Any
import time
import os

from langgraph.checkpoint.memory import MemorySaver
# from langchain_openai import ChatOpenAI # If using OpenAI
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.tools import DuckDuckGoSearchResults # Import the LangChain DDG tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper # Import the DDG wrapper
from langchain_community.utilities import GoogleSerperAPIWrapper # Import Google Serper
from langchain_tavily import TavilySearch # Import TavilySearch


logger = logging.getLogger(__name__)

async def process_query(query: str, global_config: GlobalConfig) -> KnowledgeState:
    """Enhanced query processing returning Pydantic state."""
    if not query or not query.strip():
        raise ValueError("Query cannot be empty")

    try:
        # Initialize LLMs using the global registry (assuming it's properly set up elsewhere)
        # If LLMRegistry is not used, you would initialize LLMs directly here:
        # llm = init_chat_model(...)
        # tool_llm = init_chat_model(...)
        # For now, we'll assume LLMRegistry is the way to go and pass the chat LLM to tools that need one.
        # Ensure LLMRegistry is initialized before this function is called if needed by tools.
        try:
             llm = LLMRegistry.get("chat")
             # tool_llm = LLMRegistry.get("tool") # Get tool LLM if needed by tools
        except ValueError:
             logger.warning("LLMRegistry not initialized. Initializing default chat model.")
             llm = init_chat_model(global_config.llm.chat_model_name, temperature=global_config.llm.temperature)
             # tool_llm = init_chat_model(global_config.llm.tool_model_name, temperature=global_config.llm.temperature) # Init tool LLM if needed
             # Note: If using LLMRegistry, it should be initialized once before calling process_query

        # Initialize Embedding Model (needed by CrawlerEngine and VectorStoreNode)
        # Use the model name from CrawlerConfig as it's explicitly for content scoring/embedding
        try:
            embedding_model = SentenceTransformerEmbeddings(model_name=global_config.crawler.embedding_model_name)
            logger.info(f"Initialized Embedding Model: {global_config.crawler.embedding_model_name}")
        except Exception as e:
             logger.error(f"Failed to initialize Embedding Model: {e}")
             embedding_model = None # Handle case where embedding model fails to init

        # Create initial state
        initial_state = KnowledgeState.create_knowledge_state(query.strip())

        # Initialize tools, passing the state, relevant configs, and dependencies (LLM, embedding model)
        # Assuming tool constructors are updated to accept these
        # The actual search clients are initialized in the main() function based on API keys.
        # We pass the *initialized* SearchConfig (which contains the clients) to the SearchEngine.
        # Remove dummy client definitions and assignment
        # class DummyTavily:
        #      async def ainvoke(self, *args, **kwargs):
        #           logger.info("DummyTavily ainvoke called")
        #           return [] # Return empty list
        # class DummyDDGTool:
        #      async def ainvoke(self, *args, **kwargs):
        #           logger.info("DummyDDGTool ainvoke called")
        #           return "[]" # Return empty string representing empty list
        # class DummySerper:
        #      def results(self, *args, **kwargs):
        #           logger.info("DummySerper results called")
        #           return {'organic': []} # Return empty organic results

        # global_config.search.tavily_client = DummyTavily() # Remove this line


        # Now initialize the tools, passing config and necessary dependencies
        # The SearchEngine constructor will use the actual clients passed in global_config.search
        url_analyzer = URLAnalyzer(config=global_config.url_analyzer, llm=llm) # URL Analyzer needs LLM
        search_engine = SearchEngine(config=global_config.search) # Search Engine uses its config for clients
        crawler_engine = CrawlerEngine(config=global_config.crawler, embedding_model=embedding_model) # Crawler needs embedding model
        cleaner_engine = CleanerEngine(config=global_config.cleaner, embedding_model=embedding_model) # Cleaner needs embedding model for semantic chunking
        # VectorStoreNode needs config and embedding model
        vectorstore_node = VectorStoreNode(config=global_config.vector_store, embedding_model=embedding_model)


        # Bundle tools and config for graph initialization
        tools_and_configs = {
            "llm": llm, # Pass the main chat LLM
            "config": global_config,
            "vectorstore_node": vectorstore_node,
            "url_analyzer": url_analyzer,
            "search_engine": search_engine,
            "crawler_engine": crawler_engine,
            "cleaner_engine": cleaner_engine
            # Add other tools here as they are created
        }

        # Create the graph
        workflow = create_knowledge_graph(tools_and_configs)
        logger.info("Knowledge graph created.")
        # Compile with memory for state persistence
        memory = MemorySaver()
        compiled_workflow = workflow.compile(checkpointer=memory)
        logger.info("Workflow compiled.")
        # Run the workflow
        workflow_config = {"configurable": {"thread_id": f"knowledge_subgraph_{hash(query)}_{int(time.time())}"}}
        # The workflow operates on and returns the Pydantic KnowledgeState instance directly now
        final_state = await compiled_workflow.ainvoke(initial_state, workflow_config)

        logger.info(f"Knowledge acquisition completed for query: '{query}'")
        # The returned object is already the Pydantic state
        return final_state

    except Exception as e:
        logger.error(f"Processing query '{query}' failed: {e}")
        # Create an error state using the class method for consistency
        error_state = KnowledgeState.create_knowledge_state(main_query=query)
        error_state.add_error(f"Workflow execution failed: {str(e)}")
        return error_state


async def main():
    # 2. Create actual search clients (Tavily, DDG Tool, and Serper)
    # Ensure API keys are set in the environment (e.g., using _set_if_undefined or .env)
    tavily_api_key = os.environ.get("TAVILY_API_KEY")
    if not tavily_api_key:
        print("TAVILY_API_KEY not found. Skipping Tavily test.")
        tavily_client = None
    else:
        tavily_client = TavilySearch(api_key=tavily_api_key)
        print("Tavily client initialized.")

    serper_api_key = os.environ.get("SERPER_API_KEY")
    if not serper_api_key:
        print("SERPER_API_KEY not found. Skipping Serper test.")
        serper_client = None
    else:
        serper_client = GoogleSerperAPIWrapper() # Serper API key is read from env var by the wrapper
        print("Google Serper client initialized.")


    # Create the LangChain DuckDuckGoSearchResults tool
    # The wrapper configuration (region, time, max_results) is now part of the tool setup
    ddg_tool = DuckDuckGoSearchResults(
        api_wrapper=DuckDuckGoSearchAPIWrapper(region="wt-wt", time="y", max_results=5),
        source="text" # Use text source for general search
    )
    print("LangChain DuckDuckGoSearchResults tool initialized.")

    # Step 1: Load config
    # Create instances of the config classes
    # Pass the actual initialized Tavily client here
    search_config = SearchConfig(
        external_max_results=5, # This will be used by Tavily, and potentially overridden by tool wrapper config
        subquery_max_search=2,
        subquery_results_per_query=3,
        tavily_client=tavily_client, # Pass the actual client instance
    )
    url_analyzer_config = URLAnalysisConfig(
        max_urls=5,
        priority_threshold=0.4,
        enable_ml_scoring=True,
        high_value_domains={'wikipedia.org': 0.9, 'arvix.org': 0.7},
        low_value_patterns=[r'forum\.invalid']
    )
    crawler_config = CrawlerConfig(
        chunk_size=500,
        chunk_overlap=100,
        max_concurrent=3, # Test concurrency
        timeout=5, # Set a lower timeout to trigger simulated timeouts
        max_content_length=10000, # Test truncation
        enable_embedding_content_scoring=True, # Enable embedding scoring for testing
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2", # Specify embedding model
        embedding_relevance_threshold=0.4, # Set a threshold for filtering
        max_retries=2 # Set max retries for testing
    )
    cleaner_config = CleanerConfig(
        chunk_size=200, # Also corrected chunk_size to meet validation
        chunk_overlap=30,
        use_semantic_chunking=True # Test semantic chunking
    )
    vector_store_config = VectorStoreConfig(
        collection_name="test_collection",
        chroma_persist_directory="./chroma_db_test",
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
        embedding_model_provider="huggingface"
    )
    llm_config = LLMConfig()
    decisions_config = DecisionsConfig()


    global_config = GlobalConfig(
        search=search_config.model_dump(), # Pass dictionary
        url_analyzer=url_analyzer_config.model_dump(), # Pass dictionary
        crawler=crawler_config.model_dump(), # Pass dictionary
        vector_store=vector_store_config.model_dump(), # Pass dictionary
        llm=llm_config.model_dump(), # Pass dictionary
        decisions=decisions_config.model_dump() # Pass dictionary
    )
    # Manually assign the actual client instances to the config object
    # This is necessary because model_dump() creates a dictionary, not an instance
    # And Pydantic might not retain the complex object types directly on assignment
    # A better approach might be to pass the clients directly to the tool constructors
    # or ensure the config models handle arbitrary types correctly.
    # For now, let's ensure the SearchEngine gets the actual Tavily client.
    # The SearchEngine's __init__ handles DDGTool and Serper based on env vars.
    global_config.search.tavily_client = tavily_client


    # Step 2: Initialize LLM Registry (if used)
    # Ensure you have GOOGLE_API_KEY or OPENAI_API_KEY set in your environment or Colab secrets
    try:
        LLMRegistry.init_from_config(global_config.llm)
        logger.info("LLMRegistry initialized.")
    except Exception as e:
        logger.error(f"Failed to initialize LLMRegistry: {e}")
        # Handle case where LLM initialization fails

    # Step 3: Run your agent workflow
    query = "What are the latest developments in large language models?"
    # Set up initial state explicitly
    # Use direct instantiation now that create_knowledge_state is added
    initial_state = KnowledgeState.create_knowledge_state(main_query=query)

    # Pass the initial state object to process_query
    final_state = await process_query(query, global_config)

    print("\n--- Final State Summary ---")
    print(f"Main Query: {final_state.main_query}")
    print(f"Total Errors: {len(final_state.errors)}")
    if final_state.errors:
        print("Errors:", final_state.errors)
    print(f"KB Results: {len(final_state.knowledge_base_results)} documents")
    print(f"Crawler Results: {len(final_state.crawler_results)} documents")
    print(f"External API Results (Snippets): {len(final_state.external_api_results)} documents")
    print(f"Final Cleaned Chunks: {len(final_state.cleaned_chunks)} documents")
    print(f"Remaining Crawl Queue: {len(final_state.crawl_queue)} URLs")
    print(
        f"Crawl Stats - Attempted: {final_state.metadata.crawl_stats.attempted}, "
        f"Successful: {final_state.metadata.crawl_stats.successful}, "
        f"Failed: {final_state.metadata.crawl_stats.failed}")
    # Check if final_quality_score is a number before formatting
    final_quality = final_state.state_metrics.get('final_quality_score', 'N/A')
    if isinstance(final_quality, (int, float)):
        print(f"Final Quality Score: {final_quality:.2f}")
    else:
        print(f"Final Quality Score: {final_quality}")

In [ ]:
import asyncio
import logging
import time
import os

# Run the main function to execute the workflow
try:
    asyncio.run(main())
except KeyboardInterrupt:
    print("\nWorkflow interrupted.")
except Exception as e:
    logger.critical(f"An unexpected error occurred during main execution: {e}")

# **AGENT WORKFLOW**



```
# agent_state.py

from typing import TypedDict, Dict, Any, List, Optional, Literal
from langchain_core.messages import BaseMessage
from KnowledgeState import KnowledgeState  # Import your refactored KnowledgeState
import operator
from typing_extensions import Annotated

# --- Define Pydantic models for specific tool outputs (or simple TypedDicts) ---
# Use Pydantic for more complex schemas
from pydantic import BaseModel, Field

class CodeExecutionResult(BaseModel):
    code: str
    output: Optional[str]
    error: Optional[str]

class CodeProjectState(BaseModel):
    project_goal: str
    language: Optional[str]
    framework: Optional[str]
    architecture: Optional[str]
    code_files: Dict[str, str] = Field(default_factory=dict)
    test_results: List[Dict[str, Any]] = Field(default_factory=list)
    reflection_log: List[str] = Field(default_factory=list)
    step_back_reasoning: List[str] = Field(default_factory=list)

class FileOperationResult(BaseModel):
    operation: str
    filename: str
    success: bool
    details: Optional[str]

class BashOperationResult(BaseModel):
    command: str
    output: Optional[str]
    error: Optional[str]

class FinancialTradeResult(BaseModel):
    action: str
    ticker: str
    amount: float
    status: str

class Plan(BaseModel):
    goal: str
    steps: List[str]

class Strategy(BaseModel):
    advice: str
    confidence: float
    source: str

class ReasoningOutput(BaseModel):
    process: str
    conclusion: str

class CorrectiveRAG(BaseModel):
    original_answer: str
    corrected_answer: str
    reasoning: str

class CodePlan(BaseModel):
    """A detailed plan for generating code from a research paper."""
    architecture_summary: str = Field(description="High-level summary of the system architecture.")
    implementation_plan: List[str] = Field(description="Step-by-step plan for implementation.")
    required_packages: List[str] = Field(description="List of required Python packages.")

class CodeResponse(BaseModel):
    preamble: str = Field(description="A preamble describing the code solution.")
    import_block: str = Field(description="The import statements for the code.")
    code: str = Field(description="The main body of the code.")


class AgentState(TypedDict):
    """
    The central state for the agent's graph, using Pydantic models for structured data.
    """
    messages: Annotated[List[BaseMessage], operator.add]
    # Nested KnowledgeState
    knowledge_state: KnowledgeState
    # KnowledgeState as a tool output
    knowledge_results: Optional[KnowledgeState]
    # Tool outputs (using Annotated for list merging)
    code_results: Annotated[List[CodeExecutionResult], operator.add]
    file_results: Annotated[List[FileOperationResult], operator.add]
    bash_results: Annotated[List[BashOperationResult], operator.add]
    financial_trades: Annotated[List[FinancialTradeResult], operator.add]
    coding_project: Optional[CodeProjectState]
    current_task: Optional[str]
    # Current plan, advice, etc. (these will be overwritten, so no operator.add)
    current_plan: Optional[Plan]
    advice: Optional[Strategy]
    reasoning: Optional[ReasoningOutput]
    rag_correction: Optional[CorrectiveRAG]
    # General agent status
    error: Optional[str]
    final_answer: Optional[str]

# Alias for LangGraph compatibility
class LangGraphAgentState(AgentState):
    # ... (existing fields) ...
    pass
```





```
# subgraphs.py (additions)
# ... (existing imports and subgraphs) ...

import logging
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import HumanMessage, BaseMessage
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
# Import init_chat_model instead of ChatOpenAI
from langchain.chat_models import init_chat_model
from typing import Dict, Any, List

# --- Pydantic model for structured code output ---
from pydantic import BaseModel, Field

class CodePlan(BaseModel):
    """A detailed plan for generating code from a research paper."""
    architecture_summary: str = Field(description="High-level summary of the system architecture.")
    implementation_plan: List[str] = Field(description="Step-by-step plan for implementation.")
    required_packages: List[str] = Field(description="List of required Python packages.")

class CodeResponse(BaseModel):
    preamble: str = Field(description="A preamble describing the code solution.")
    import_block: str = Field(description="The import statements for the code.")
    code: str = Field(description="The main body of the code.")

# --- Code Generation Subgraph ---

# Update type hint from ChatOpenAI to Any or a protocol if preferred
async def interpret_paper_node(state: LangGraphAgentState, llm: Any) -> dict:
    """
    Interprets a research paper and generates a high-level code plan.
    This uses the Nested KnowledgeState for context.
    """
    knowledge_state = state.get("knowledge_state")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert software engineer. Given the following research paper content,
    provide a high-level architecture summary, a step-by-step implementation plan,
    and a list of required Python packages.

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodePlan)
    plan = await chain.ainvoke({"paper_content": paper_content})

    return {"code_generation_plan": plan.dict()}

# Update type hint from ChatOpenAI to Any or a protocol if preferred
async def generate_code_node(state: LangGraphAgentState, llm: Any) -> dict:
    """Generates code based on the plan and the research paper content."""
    knowledge_state = state.get("knowledge_state")
    plan = state.get("code_generation_plan")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert Python developer. Given the following research paper content and
    implementation plan, generate the complete, production-ready, and executable code.

    Implementation Plan:
    {plan}

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    code_response = await chain.ainvoke({"plan": plan, "paper_content": paper_content})

    return {"code_response": code_response.dict()}

async def validate_code_node(state: LangGraphAgentState, executor_tool) -> dict:
    """
    Validates the generated code by executing it.
    This node would call an external code execution tool.
    """
    code_response = state.get("code_response")
    full_code = f"{code_response.get('import_block')}\n\n{code_response.get('code')}"

    # Use the code_executor_tool (from your main agent's toolset)
    result = await executor_tool.invoke({"code": full_code})

    if result.get("error"):
        return {"code_validation_status": "failed", "code_execution_result": result}
    else:
        return {"code_validation_status": "success", "code_execution_result": result}

# Update type hint from ChatOpenAI to Any or a protocol if preferred
async def reflect_and_correct_node(state: LangGraphAgentState, llm: Any) -> dict:
    """Reflects on validation errors and generates a corrected version."""
    code_execution_result = state.get("code_execution_result")

    prompt = PromptTemplate.from_template("""
    The following code failed to execute. Analyze the error and provide a corrected version.

    Code:
    {code}

    Error:
    {error}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    corrected_code = await chain.ainvoke({"code": state.get("code_response"), "error": code_execution_result.get("error")})

    return {"code_response": corrected_code.dict()}

# --- Decision Nodes ---

def check_validation_status(state: LangGraphAgentState) -> str:
    """Conditional edge to check if code validation passed."""
    return "finalize" if state.get("code_validation_status") == "success" else "reflect_and_correct"

def check_reflection_limit(state: LangGraphAgentState) -> str:
    """Prevents infinite loops by limiting reflection attempts."""
    reflection_count = state.get("reflection_count", 0) + 1
    state["reflection_count"] = reflection_count
    return "validate" if reflection_count <= 3 else "fail"

# Update type hint from ChatOpenAI to Any or a protocol if preferred
def create_code_generation_subgraph(llm: Any, code_executor_tool: Any) -> StateGraph:
    """
    Creates the complete code generation subgraph.
    """
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("interpret", lambda state: interpret_paper_node(state, llm))
    graph.add_node("generate", lambda state: generate_code_node(state, llm))
    graph.add_node("validate", lambda state: validate_code_node(state, code_executor_tool))
    graph.add_node("reflect_and_correct", lambda state: reflect_and_correct_node(state, llm))

    graph.add_edge(START, "interpret")
    graph.add_edge("interpret", "generate")
    graph.add_edge("generate", "validate")
    graph.add_conditional_edges("validate", check_validation_status, {
        "finalize": END,
        "reflect_and_correct": "reflect_and_correct"
    })
    graph.add_edge("reflect_and_correct", "validate")

    return graph.compile()
```





```
# subgraphs.py

import logging
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
# Import init_chat_model instead of ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_community.tools import ShellTool
from typing import Dict, Any, List

from agent_state import LangGraphAgentState, CodeProjectState

# Import the actual entry point for your knowledge acquisition graph
from workflow import create_knowledge_graph

logger = logging.getLogger(__name__)

# --- Simple Placeholder Nodes ---

async def _simple_passthrough_node(state: LangGraphAgentState) -> LangGraphAgentState:
    """A simple passthrough node for placeholder subgraphs."""
    # Simulate some work
    message_content = f"Executed placeholder node for {state['messages'][-1].content}"
    # Use operator.add behavior for messages
    new_messages = [HumanMessage(content=message_content)]
    state['messages'].extend(new_messages)
    return state

# --- Subgraph Creation Functions ---

# Update type hint from ChatOpenAI to Any or a protocol if preferred
def create_knowledge_subgraph(tools_and_configs: dict) -> StateGraph:
    """Creates the knowledge acquisition subgraph using your existing workflow."""
    # The actual implementation of create_knowledge_graph should be in workflow.py
    return create_knowledge_graph(tools_and_configs).compile()

def create_code_executor_subgraph() -> StateGraph:
    """Creates the code execution subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("code_node", _simple_passthrough_node)
    graph.add_edge("code_node", END)
    return graph.compile()

def create_bash_subgraph() -> StateGraph:
    """Creates the bash operation subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("bash_node", _simple_passthrough_node)
    graph.add_edge("bash_node", END)
    return graph.compile()

# --- Code Development Subgraph ---
# Placeholder function - actual implementation is in fgaRKOwKPVBw
# Update type hint from ChatOpenAI to Any or a protocol if preferred
def create_code_development_subgraph(llm: Any, shell_tool: ShellTool) -> StateGraph:
    """Creates the code development subgraph."""
    # This function is defined in cell fgaRKOwKPVBw, no need to redefine here
    # However, for the main agent workflow to call it, it needs to be accessible.
    # Let's ensure the definition in fgaRKOwKPVBw is the one used.
    pass # Keep this as a placeholder or remove if the definition is in fgaRKOwKPVBw


def create_file_operation_subgraph() -> StateGraph:
    """Creates the file operation subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("file_node", _simple_passthrough_node)
    graph.add_edge("file_node", END)
    return graph.compile()

def create_bash_subgraph() -> StateGraph:
    """Creates the bash operation subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("bash_node", _simple_passthrough_node)
    graph.add_edge("bash_node", END)
    return graph.compile()

def create_financial_trader_subgraph() -> StateGraph:
    """Creates the financial trader subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("trade", _simple_passthrough_node)
    graph.add_edge("trade", END)
    return graph.compile()

def create_planning_subgraph() -> StateGraph:
    """Creates the planning node subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("plan_node", _simple_passthrough_node)
    graph.add_edge("plan_node", END)
    return graph.compile()

def create_strategy_subgraph() -> StateGraph:
    """Creates the strategy advice subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("strategy_node", _simple_passthrough_node)
    graph.add_edge("strategy_node", END)
    return graph.compile()

def create_reasoning_subgraph() -> StateGraph:
    """Creates the reasoning node subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("reasoning_node", _simple_passthrough_node)
    graph.add_edge("reasoning_node", END)
    return graph.compile()

def create_corrective_rag_subgraph() -> StateGraph:
    """Creates the corrective RAG node subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("corrective_rag_node", _simple_passthrough_node)
    graph.add_edge("corrective_rag_node", END)
    return graph.compile()
```





```
# subgraphs.py (additions)
# ... (existing imports and subgraphs) ...

import logging
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import HumanMessage, BaseMessage
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
# Import init_chat_model instead of ChatOpenAI
from langchain.chat_models import init_chat_model
from typing import Dict, Any, List

# --- Pydantic model for structured code output ---
from pydantic import BaseModel, Field

class CodePlan(BaseModel):
    """A detailed plan for generating code from a research paper."""
    architecture_summary: str = Field(description="High-level summary of the system architecture.")
    implementation_plan: List[str] = Field(description="Step-by-step plan for implementation.")
    required_packages: List[str] = Field(description="List of required Python packages.")

class CodeResponse(BaseModel):
    preamble: str = Field(description="A preamble describing the code solution.")
    import_block: str = Field(description="The import statements for the code.")
    code: str = Field(description="The main body of the code.")

# --- Code Generation Subgraph ---

# Update type hint from ChatOpenAI to Any or a protocol if preferred
async def interpret_paper_node(state: LangGraphAgentState, llm: Any) -> dict:
    """
    Interprets a research paper and generates a high-level code plan.
    This uses the Nested KnowledgeState for context.
    """
    knowledge_state = state.get("knowledge_state")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert software engineer. Given the following research paper content,
    provide a high-level architecture summary, a step-by-step implementation plan,
    and a list of required Python packages.

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodePlan)
    plan = await chain.ainvoke({"paper_content": paper_content})

    return {"code_generation_plan": plan.dict()}

# Update type hint from ChatOpenAI to Any or a protocol if preferred
async def generate_code_node(state: LangGraphAgentState, llm: Any) -> dict:
    """Generates code based on the plan and the research paper content."""
    knowledge_state = state.get("knowledge_state")
    plan = state.get("code_generation_plan")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert Python developer. Given the following research paper content and
    implementation plan, generate the complete, production-ready, and executable code.

    Implementation Plan:
    {plan}

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    code_response = await chain.ainvoke({"plan": plan, "paper_content": paper_content})

    return {"code_response": code_response.dict()}

async def validate_code_node(state: LangGraphAgentState, executor_tool) -> dict:
    """
    Validates the generated code by executing it.
    This node would call an external code execution tool.
    """
    code_response = state.get("code_response")
    full_code = f"{code_response.get('import_block')}\n\n{code_response.get('code')}"

    # Use the code_executor_tool (from your main agent's toolset)
    result = await executor_tool.invoke({"code": full_code})

    if result.get("error"):
        return {"code_validation_status": "failed", "code_execution_result": result}
    else:
        return {"code_validation_status": "success", "code_execution_result": result}

# Update type hint from ChatOpenAI to Any or a protocol if preferred
async def reflect_and_correct_node(state: LangGraphAgentState, llm: Any) -> dict:
    """Reflects on validation errors and generates a corrected version."""
    code_execution_result = state.get("code_execution_result")

    prompt = PromptTemplate.from_template("""
    The following code failed to execute. Analyze the error and provide a corrected version.

    Code:
    {code}

    Error:
    {error}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    corrected_code = await chain.ainvoke({"code": state.get("code_response"), "error": code_execution_result.get("error")})

    return {"code_response": corrected_code.dict()}

# --- Decision Nodes ---

def check_validation_status(state: LangGraphAgentState) -> str:
    """Conditional edge to check if code validation passed."""
    return "finalize" if state.get("code_validation_status") == "success" else "reflect_and_correct"

def check_reflection_limit(state: LangGraphAgentState) -> str:
    """Prevents infinite loops by limiting reflection attempts."""
    reflection_count = state.get("reflection_count", 0) + 1
    state["reflection_count"] = reflection_count
    return "validate" if reflection_count <= 3 else "fail"

# Update type hint from ChatOpenAI to Any or a protocol if preferred
def create_code_generation_subgraph(llm: Any, code_executor_tool: Any) -> StateGraph:
    """
    Creates the complete code generation subgraph.
    """
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("interpret", lambda state: interpret_paper_node(state, llm))
    graph.add_node("generate", lambda state: generate_code_node(state, llm))
    graph.add_node("validate", lambda state: validate_code_node(state, code_executor_tool))
    graph.add_node("reflect_and_correct", lambda state: reflect_and_correct_node(state, llm))

    graph.add_edge(START, "interpret")
    graph.add_edge("interpret", "generate")
    graph.add_edge("generate", "validate")
    graph.add_conditional_edges("validate", check_validation_status, {
        "finalize": END,
        "reflect_and_correct": "reflect_and_correct"
    })
    graph.add_edge("reflect_and_correct", "validate")

    return graph.compile()
@tool
def code_generator_tool(query: str) -> str:
    """Generates code from a research paper or a detailed description."""
    return f"CALL_SUBGRAPH:code_generation:{query}"
```




```
# agent_workflow.py
import operator
from typing import List, Dict, Any, Union
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, tool_executor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.agents import AgentAction, AgentFinish, AgentActionMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_community.tools import ShellTool # Use for code execution
from agent_state import LangGraphAgentState
from subgraphs import (
    create_knowledge_subgraph, create_code_executor_subgraph,
    create_file_operation_subgraph, create_bash_subgraph,
    create_financial_trader_subgraph, create_planning_subgraph,
    create_strategy_subgraph, create_reasoning_subgraph,
    create_corrective_rag_subgraph,
    create_code_development_subgraph
)
from config import GlobalConfig
shell_tool = ShellTool()

# --- Define Tools for the Main LLM Router ---

@tool
def knowledge_acquisition_tool(query: str) -> str:
    """Acquire comprehensive knowledge on a query."""
    return f"CALL_SUBGRAPH:knowledge_acquisition:{query}"

@tool
def code_executor_tool(code: str) -> str:
    """Execute code and get the result."""
    return f"CALL_SUBGRAPH:code_executor:{code}"

@tool
def file_operation_tool(operation: str, filename: str, content: str = "") -> str:
    """Perform file operations (read, write, delete)."""
    return f"CALL_SUBGRAPH:file_operation:{operation}:{filename}:{content}"

@tool
def bash_tool(command: str) -> str:
    """Execute a bash command."""
    return f"CALL_SUBGRAPH:bash:{command}"

@tool
def financial_trader_tool(query: str) -> str:
    """Execute a financial trading action based on a query."""
    return f"CALL_SUBGRAPH:financial_trader:{query}"

@tool
def planning_tool(query: str) -> str:
    """Create a step-by-step plan for a complex task."""
    return f"CALL_SUBGRAPH:planning:{query}"

@tool
def strategy_tool(query: str) -> str:
    """Provide strategic advice or analysis."""
    return f"CALL_SUBGRAPH:strategy:{query}"

@tool
def reasoning_tool(query: str) -> str:
    """Perform logical reasoning and inference."""
    return f"CALL_SUBGRAPH:reasoning:{query}"

@tool
def corrective_rag_tool(query: str) -> str:
    """Perform corrective retrieval-augmented generation."""
    return f"CALL_SUBGRAPH:corrective_rag:{query}"

@tool
def respond_tool(message: str) -> str:
    """Respond directly to the user."""
    return f"FINAL_RESPONSE:{message}"

@tool
def code_development_tool(query: str) -> str:
    """Generate, test, and debug code for a specific task."""
    return f"CALL_SUBGRAPH:code_development:{query}"

def create_unified_agent_workflow(llm: ChatOpenAI, all_tools: list, tools_and_configs: dict) -> StateGraph:
    """
    Creates the main agent graph with subgraphs as nodes.
    """
    # Compile the subgraphs
    subgraphs = {
        "knowledge_acquisition": create_knowledge_subgraph(tools_and_configs),
        "code_executor": create_code_executor_subgraph(),
        "file_operation": create_file_operation_subgraph(),
        "bash": create_bash_subgraph(),
        "financial_trader": create_financial_trader_subgraph(),
        "planning": create_planning_subgraph(),
        "strategy": create_strategy_subgraph(),
        "reasoning": create_reasoning_subgraph(),
        "corrective_rag": create_corrective_rag_subgraph(),
        "code_generation": create_code_generation_subgraph(llm, shell_tool),
        "code_development": create_code_development_subgraph(llm, shell_tool),
    }

    # Bind tools to the LLM
    llm_with_tools = llm.bind_tools(all_tools)

    workflow = StateGraph(LangGraphAgentState)

    async def route_agent_node(state: LangGraphAgentState) -> Dict[str, Any]:
        """The main agent reasoning node for routing."""
        # Use only the last message for routing to prevent context flooding
        messages = state['messages'][-1:]
        response = await llm_with_tools.ainvoke(messages)
        return {"messages": [response]}

    async def subgraph_executor_node(state: LangGraphAgentState) -> Dict[str, Any]:
        """Executes the subgraph selected by the router."""
        last_message = state['messages'][-1]

        # Extract subgraph call info from the dispatcher tool's output
        subgraph_call_string = last_message.tool_calls[0]['args']['query']

        if subgraph_call_string.startswith("CALL_SUBGRAPH:"):
            parts = subgraph_call_string.split(":", 2)
            subgraph_name = parts[1]
            # Assuming the query is the last part
            query = parts[2]

            subgraph = subgraphs.get(subgraph_name)
            if subgraph:
                # The subgraph will update the state directly
                return await subgraph.ainvoke(state)
            else:
                state['error'] = f"Unknown subgraph: {subgraph_name}"
                return state
        elif subgraph_call_string.startswith("FINAL_RESPONSE:"):
            response = subgraph_call_string.split(":", 1)[1]
            return {"final_answer": response}

        return state

    workflow.add_node("router", route_agent_node)
    workflow.add_node("executor", subgraph_executor_node)

    def route_to_executor(state: LangGraphAgentState) -> Union[str, END]:
        last_message = state['messages'][-1]
        if last_message.tool_calls:
            return "executor"
        else:
            return END

    workflow.add_conditional_edges(
        "router",
        route_to_executor,
        {"executor": "executor", END: END}
    )

    # Executor always returns to the router to check for follow-up actions
    workflow.add_edge("executor", "router")

    workflow.set_entry_point("router")

    return workflow.compile(checkpointer=MemorySaver())

# This is formatted as code
```





```
# cli.py
import asyncio
import logging
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

from agent_state import LangGraphAgentState, KnowledgeState
from agent_workflow import create_unified_agent_workflow
from config import GlobalConfig
from tools import (
    knowledge_acquisition_tool, code_executor_tool,
    file_operation_tool, bash_tool, financial_trader_tool,
    planning_tool, strategy_tool, reasoning_tool,
    corrective_rag_tool, respond_tool, code_development_tool
)


logging.basicConfig(level=logging.INFO)

async def main():
    """Main function for the agent CLI."""
    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    # Configure tools and subgraphs
    global_config = GlobalConfig()

    tools = [
        knowledge_acquisition_tool, code_executor_tool,
        file_operation_tool, bash_tool, financial_trader_tool,
        planning_tool, strategy_tool, reasoning_tool,
        corrective_rag_tool, respond_tool, code_development_tool
    ]

    # Pass tools and configs to the workflow creator
    tools_and_configs = {"llm": llm, "config": global_config}
    agent_graph = create_unified_agent_workflow(llm, tools, tools_and_configs)

    while True:
        user_input = input("Enter your query (type 'exit' to quit): ")
        if user_input.lower() == 'exit':
            break

        # Create initial state
        initial_state = LangGraphAgentState(
            messages=[HumanMessage(content=user_input)],
            knowledge_state=KnowledgeState.(main_query=user_input),
            coding_project=None,
            code_results=[],
            file_results=[],
            bash_results=[],
            financial_trades=[],
            current_plan=None,
            advice=None,
            reasoning=None,
            rag_correction=None,
            error=None,
            final_answer=None
        )

        try:
            print("--- Invoking agent ---")
            final_state = await agent_graph.ainvoke(initial_state)
            print("--- Agent finished ---")
            print(f"Final response: {final_state.get('final_answer', 'No final answer generated.')}")

        except Exception as e:
            logging.error(f"An error occurred during agent execution: {e}")

if __name__ == "__main__":
    asyncio.run(main())

```

